# SatQuery AI — Preprocessing Pipeline (Kaggle)

Auto-generated from the `satquery-preprocess-test` repo — every `.py` file materialized via `%%writefile` so the package imports (`preprocess.common.*`, `configs.paths`) work unmodified, then run from `/kaggle/working`.

Mount your dataset(s) under `/kaggle/input/...` and fill in `--input-dir` below.

## Setup — install deps, create package dirs

In [ ]:
%%writefile requirements.txt
# SatQuery AI — Pinned Dependencies
# Update versions deliberately; run `pip install -r requirements.txt` yourself.

# --- Step 1: Validator + Tests ---
jsonschema>=4.21,<5
pytest>=8.0,<9

# --- Step 2+: Preprocessing (common/ modules + tier scripts) ---
rasterio>=1.3,<2
numpy>=1.26,<2
Pillow>=10.0,<11
tifffile>=2024.1,<2025  # tier1_oscd, tier1_sn6_opt, tier2_sn6_sar
pandas>=2.0,<3          # tier0_bigen, tier2_sardet/sn6_sar sanity checks
pyarrow>=14.0            # parquet read/write backend for pandas — no upper
                         # pin: <16 has no Python 3.13 wheel and fails to
                         # build from source without a C++ toolchain
PyYAML>=6.0,<7           # training/train.py config loading
scipy>=1.11,<2           # tier1_oscd connected-component bboxes (has a fallback if absent)

# --- Training (GPU session only — not needed for preprocessing) ---
# torch, unsloth, safetensors: install per Unsloth's Kaggle install docs,
# pinned to whatever CUDA/driver combo the Kaggle GPU image ships that week.
# Do not pin here — Unsloth's own installer resolves the right build.


In [ ]:
import os
for d in ['configs', 'preprocess', 'preprocess/common', 'tests', 'training']:
    os.makedirs(d, exist_ok=True)


In [ ]:
!pip install -q -r requirements.txt

## `configs/`

In [ ]:
%%writefile configs/paths.py
import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("SATQUERY_BASE_DIR", "./data/working"))
SCRATCH_DIR = Path(os.environ.get("SATQUERY_SCRATCH_DIR", "./data/tmp"))
INPUT_DIR = Path(os.environ.get("SATQUERY_INPUT_DIR", "./data/input"))

BASE_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)


## `preprocess/common/`

In [ ]:
%%writefile preprocess/common/__init__.py
"""SatQuery preprocessing common utilities.

Implements the R1–R7 rules from the frozen spec (v4).
"""


In [ ]:
%%writefile preprocess/common/bbox.py
"""R6 — Bounding box format conversion.

Qwen convention: normalized [0,1000], ordered
(x_topleft, y_topleft), (x_bottomright, y_bottomright).  Not y-first.

Public API:
    pixel_to_normalized(bbox, img_w, img_h) -> [[x1,y1,x2,y2]]
    normalized_to_pixel(bbox, img_w, img_h) -> (x1, y1, x2, y2)
    normalized_to_latlon(bbox, transform, crs) -> [[lon1,lat1],[lon2,lat2]]
    validate_bbox_ordering(bbox) -> bool
"""

from __future__ import annotations

from typing import Any

import numpy as np


def unit01_to_qwen(bbox_01: list[float]) -> list[list[float]]:
    """Convert 0-1 normalized bbox to Qwen [0,1000] format.

    Parameters
    ----------
    bbox_01 : [x1, y1, x2, y2] in 0-1 normalized coordinates.

    Returns
    -------
    [[x1, y1, x2, y2]] in [0,1000] Qwen order.
    """
    x1, y1, x2, y2 = bbox_01
    return [[x1 * 1000, y1 * 1000, x2 * 1000, y2 * 1000]]


def pixel_to_normalized(
    bbox: tuple[int, int, int, int],
    img_width: int,
    img_height: int,
) -> list[list[float]]:
    """Convert pixel-space bbox (x1,y1,x2,y2) to [0,1000] normalized.

    Parameters
    ----------
    bbox : (x_topleft, y_topleft, x_bottomright, y_bottomright) in pixels.
    img_width, img_height : dimensions of the source image in pixels.

    Returns
    -------
    [[x1_norm, y1_norm, x2_norm, y2_norm]] in [0,1000] Qwen order.
    """
    x1, y1, x2, y2 = bbox
    return [[
        round(x1 / img_width * 1000, 2),
        round(y1 / img_height * 1000, 2),
        round(x2 / img_width * 1000, 2),
        round(y2 / img_height * 1000, 2),
    ]]


def normalized_to_pixel(
    bbox: list[list[float]],
    img_width: int,
    img_height: int,
) -> tuple[int, int, int, int]:
    """Convert [0,1000] normalized bbox back to pixel coordinates.

    Returns (x_topleft, y_topleft, x_bottomright, y_bottomright) as ints.
    """
    x1n, y1n, x2n, y2n = bbox[0]
    return (
        int(round(x1n / 1000 * img_width)),
        int(round(y1n / 1000 * img_height)),
        int(round(x2n / 1000 * img_width)),
        int(round(y2n / 1000 * img_height)),
    )


def normalized_to_latlon(
    bbox: list[list[float]],
    transform: Any,
    crs: Any,
) -> list[list[float]]:
    """Convert [0,1000] normalized bbox to (lon, lat) via rasterio transform.

    Parameters
    ----------
    bbox : [[x1n, y1n, x2n, y2n]] in [0,1000].
    transform : rasterio.Affine (pixel → CRS coordinates).
    crs : rasterio CRS object.

    Returns
    -------
    [[lon1, lat1], [lon2, lat2]] (lon/lat if CRS is geographic,
    projected coords otherwise).
    """
    try:
        from rasterio.transform import xy as rio_xy
    except ImportError:
        raise ImportError("rasterio is required for normalized_to_latlon()")

    x1n, y1n, x2n, y2n = bbox[0]

    # Convert [0,1000] → pixel coords (float, not rounded — we want precise projection)
    # The [0,1000] range maps to [0, img_width) and [0, img_height) conceptually.
    # Since we don't have img dimensions here, we treat 1000 as the full extent.
    px1 = x1n / 1000
    py1 = y1n / 1000
    px2 = x2n / 1000
    py2 = y2n / 1000

    # rasterio xy gives (x, y) in CRS units for (col, row)
    lon1, lat1 = rio_xy(transform, px1, py1)
    lon2, lat2 = rio_xy(transform, px2, py2)

    return [[lon1, lat1], [lon2, lat2]]


def validate_bbox_ordering(bbox: list[list[float]]) -> bool:
    """Verify topleft/bottomright ordering (R6 convention).

    Returns True if x1 <= x2 and y1 <= y2 (valid ordering).
    Returns False if the box is degenerate/reversed.
    """
    x1, y1, x2, y2 = bbox[0]
    return x1 <= x2 and y1 <= y2


In [ ]:
%%writefile preprocess/common/concat.py
"""R7: 1×2 spatial concatenation for bi-temporal/cross-modal image pairs.

Generates a side-by-side concatenated PNG for visual-evidence display
(inference-time, Gradio gallery). Not called by tier scripts during
training preprocessing — only used at inference/GUI time.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image


def concat_horizontal(
    img_left: np.ndarray | Image.Image,
    img_right: np.ndarray | Image.Image,
    *,
    target_height: int | None = None,
    separator_width: int = 2,
    separator_color: int = 128,
) -> Image.Image:
    """Concatenate two images side-by-side (1×2 horizontal).

    Args:
        img_left: Left image (H, W, 3) uint8 ndarray or PIL Image.
        img_right: Right image (H, W, 3) uint8 ndarray or PIL Image.
        target_height: If set, resize both images to this height before
            concatenating. If None, uses left image's height and resizes
            right to match.
        separator_width: Width of the vertical separator line in pixels.
        separator_color: Grayscale value for the separator (0-255).

    Returns:
        PIL Image of shape (H, W_left + separator + W_right, 3) uint8.
    """
    if isinstance(img_left, Image.Image):
        img_left = np.array(img_left)
    if isinstance(img_right, Image.Image):
        img_right = np.array(img_right)

    if img_left.ndim != 3 or img_right.ndim != 3:
        raise ValueError(f"Expected 3-channel images, got ndim={img_left.ndim} and {img_right.ndim}")
    if img_left.shape[2] != 3 or img_right.shape[2] != 3:
        raise ValueError(f"Expected 3 channels, got {img_left.shape[2]} and {img_right.shape[2]}")

    left_pil = Image.fromarray(img_left)
    right_pil = Image.fromarray(img_right)

    if target_height is not None:
        # Resize both to target height, preserving aspect ratio
        l_ratio = target_height / left_pil.height
        left_pil = left_pil.resize(
            (max(1, int(left_pil.width * l_ratio)), target_height), Image.BILINEAR
        )
        r_ratio = target_height / right_pil.height
        right_pil = right_pil.resize(
            (max(1, int(right_pil.width * r_ratio)), target_height), Image.BILINEAR
        )
    else:
        # Resize right to match left's height
        if right_pil.height != left_pil.height:
            ratio = left_pil.height / right_pil.height
            right_pil = right_pil.resize(
                (max(1, int(right_pil.width * ratio)), left_pil.height), Image.BILINEAR
            )

    canvas_width = left_pil.width + separator_width + right_pil.width
    canvas_height = left_pil.height

    canvas = Image.new("RGB", (canvas_width, canvas_height), (separator_color,) * 3)
    canvas.paste(left_pil, (0, 0))
    canvas.paste(right_pil, (left_pil.width + separator_width, 0))

    return canvas


def save_concat(
    left_path: Path,
    right_path: Path,
    output_path: Path,
    **kwargs,
) -> None:
    """Load two images and save their 1×2 concatenation.

    Args:
        left_path: Path to left/before image.
        right_path: Path to right/after image.
        output_path: Path to save concatenated PNG.
        **kwargs: Passed to concat_horizontal.
    """
    left = Image.open(left_path)
    right = Image.open(right_path)
    result = concat_horizontal(left, right, **kwargs)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.save(str(output_path))


In [ ]:
%%writefile preprocess/common/gsd.py
"""R4/R5 — GSD bucket assignment and dual-resolution curriculum.

Section 5 rules (authoritative):
    Datasets with known, uniform native GSD → literal tags [GSD:Xm]
    Datasets with variable/unknown per-tile GSD → categorical buckets

    Known-uniform datasets:
        RSVQA-HR:           0.15 m   → [GSD:0.15m]
        SpaceNet6 optical:  0.5 m    → [GSD:0.5m]
        LEVIR-CD:           0.5 m    → [GSD:0.5m]
        SpaceNet6 SAR:      0.5 m    → [GSD:0.5m]
        SARDet-100K:        2–10 m   → [GSD:5m] (post-R5 midpoint)
        BigEarthNet:        10 m     → [GSD:10m]
        OSCD:               10 m Sentinel-2, same sensor as BigEarthNet → [GSD:10m]
        Sen-2 LULC:         10 m Sentinel-2 → [GSD:10m]

    Variable/unknown GSD datasets:
        VRSBench            → VHR-native  (native branch)
        VRSBench proxy      → CARTOSAT-proxy
        CDVQA                → SECOND imagery spans ~0.5-2m per scene, no single
                                 native value → VHR-native (no R4 doubling — see
                                 DUAL_RESOLUTION_DATASETS below)

R4 dual-resolution branching (for benchmark datasets only):
    Each training image → two independent samples:
        Native branch  → gsd_bucket per assignment rule
        Proxy branch   → CARTOSAT-proxy (optical ~2m) or RISAT-band (SAR ~2-10m)
    Curriculum mixing across Stage 2:
        80:20 → 50:50 → 30:70  (native:proxy)

R5 SAR GSD calibration:
    Optical VHR → Cartosat-2S (~2m MS)
    SAR → RISAT proxy band (~2–10m)
    SpaceNet6 SAR (0.5m) & SARDet-100K finer → downsampled to RISAT band
    BigEarthNet S1 (10m) → coarse anchor, left as-is

Public API:
    assign_gsd_bucket(dataset, native_gsd=None) -> str
    create_proxy_sample(sample, proxy_gsd=None) -> dict
    curriculum_mix_ratio(stage) -> (float, float)
    DUAL_RESOLUTION_DATASETS — set of datasets that get dual-res branching
"""

from __future__ import annotations

import copy
import sys
from pathlib import Path
from typing import Any


# ---------------------------------------------------------------------------
# Known uniform-GSD datasets  (literal tag mapping)
# ---------------------------------------------------------------------------

_KNOWN_GSD: dict[str, str] = {
    "rsvqa_hr":   "[GSD:0.15m]",
    "sn6_opt":    "[GSD:0.5m]",
    "levir_cd":   "[GSD:0.5m]",
    "sn6_sar":    "[GSD:0.5m]",
    "sardet":     "[GSD:5m]",      # midpoint of 2–10m RISAT band
    "bigen":      "[GSD:10m]",     # BigEarthNet S1 anchor
    "oscd":       "[GSD:10m]",     # Sentinel-2, same native GSD as BigEarthNet
    "sen2lulc":   "[GSD:10m]",     # Sentinel-2
}

# Datasets eligible for R4 dual-resolution branching.
# Resolved spec ambiguity (Section 2 R4 prose names CDVQA; the Tier 1
# table's per-row Treatment column is more specific and is authoritative
# here): every Tier 1 row whose Treatment column literally says "R4"
# gets it — vrsbench, rsvqa_hr, levir_cd, sn6_opt. CDVQA's row says
# "native res kept" (no R4) and OSCD's row doesn't mention R4 either;
# both have fixed/known native resolutions already in the VHR range,
# so doubling them buys no resolution diversity.
DUAL_RESOLUTION_DATASETS: set[str] = {
    "vrsbench",
    "rsvqa_hr",
    "levir_cd",
    "sn6_opt",
}

# Proxy bucket names
CARTOSAT_PROXY = "CARTOSAT-proxy"
RISAT_PROXY = "RISAT-proxy"

# Default proxy GSD (Cartosat-2S ~2m)
DEFAULT_OPTICAL_PROXY_GSD = 2.0
# Default SAR proxy GSD (RISAT band midpoint ~5m)
DEFAULT_SAR_PROXY_GSD = 5.0


# ---------------------------------------------------------------------------
# GSD bucket assignment
# ---------------------------------------------------------------------------

def assign_gsd_bucket(
    dataset: str,
    native_gsd: float | None = None,
) -> str:
    """Assign the gsd_bucket string for a sample.

    Parameters
    ----------
    dataset : dataset identifier (matches SCHEMA enum values).
    native_gsd : per-tile GSD in metres, if known.  For datasets with
                 variable GSD (VRSBench, CDVQA), this is typically None
                 and the categorical bucket is returned.

    Returns
    -------
    GSD bucket string: either a literal tag like "[GSD:0.5m]" or a
    categorical bucket like "VHR-native" / "CARTOSAT-proxy".
    """
    if dataset in _KNOWN_GSD:
        return _KNOWN_GSD[dataset]

    # Variable/unknown GSD datasets
    if native_gsd is not None:
        return f"[GSD:{_fmt_gsd(native_gsd)}]"

    # No native GSD provided → categorical bucket
    if dataset in ("vrsbench", "cdvqa"):
        return "VHR-native"

    # Fallback: unknown dataset, unknown GSD
    return "VHR-native"


def _fmt_gsd(gsd: float) -> str:
    """Format GSD value for bucket tag."""
    if gsd == int(gsd):
        return f"{int(gsd)}m"
    return f"{gsd:.2f}m"


# ---------------------------------------------------------------------------
# R4 — Dual-resolution proxy sample creation
# ---------------------------------------------------------------------------

def create_proxy_sample(
    sample: dict[str, Any],
    proxy_gsd: float | None = None,
) -> dict[str, Any]:
    """Create a proxy-branch copy of a sample for dual-resolution branching.

    The proxy sample has:
        - gsd_bucket set to CARTOSAT-proxy or RISAT-proxy
        - A synthetic id suffix "_proxy"
        - Original image paths preserved by default

    For actual pixel-level GSD transformation (Issue 5), callers should use
    ``generate_proxy_image()`` from this module to bicubic-downsample the source
    image and then update ``proxy["image_path"]`` to point at the new file.
    Without that step the GSD token carries no visual information.

    Parameters
    ----------
    sample : original training sample dict.
    proxy_gsd : explicit proxy GSD; if None, inferred from modality.

    Returns
    -------
    New dict (deep copy) with proxy metadata.
    """
    proxy = copy.deepcopy(sample)

    # Determine proxy bucket from modality
    modality = proxy.get("modality", "optical")
    if modality == "sar":
        proxy_bucket = RISAT_PROXY
        default_gsd = DEFAULT_SAR_PROXY_GSD
    else:
        proxy_bucket = CARTOSAT_PROXY
        default_gsd = DEFAULT_OPTICAL_PROXY_GSD

    gsd = proxy_gsd if proxy_gsd is not None else default_gsd

    proxy["gsd_bucket"] = f"{proxy_bucket}[GSD:{_fmt_gsd(gsd)}]"
    proxy["id"] = f"{proxy['id']}_proxy"

    return proxy


# ---------------------------------------------------------------------------
# R4 — Proxy image generation (Issue 5 fix: generalised from tier1_vrsbench)
# ---------------------------------------------------------------------------

def generate_proxy_image(
    src_path: "Path | str",
    proxy_dir: "Path | str",
    image_name: str,
    scale_factor: int = 4,
) -> str | None:
    """Generate a bicubic-downsampled proxy image for R4 dual-resolution branching.

    Issue 5 fix: previously only VRSBench called ``generate_proxy_image()``;
    rsvqa_hr, levir_cd, and sn6_opt reused identical pixels with a different
    GSD tag, making the tag meaningless.  This generalised version lives in
    ``common/gsd.py`` so every DUAL_RESOLUTION_DATASETS tier script can call it.

    Issue 7 fix: scale is dynamic (``w // scale_factor``, ``h // scale_factor``)
    rather than a hardcoded (128, 128) target, so any source image size receives
    the intended GSD reduction.

    Issue 9 fix: writes with ``compress_level=6`` to avoid Kaggle storage bloat.

    Parameters
    ----------
    src_path : source image file (any PIL-readable format).
    proxy_dir : directory into which the proxy PNG is written.
    image_name : filename for the proxy (usually the source basename).
    scale_factor : integer divisor applied to both width and height (default 4).
                   Resulting size is clamped to a minimum of 32px per side.

    Returns
    -------
    Absolute path string to the written proxy PNG, or None on failure.
    """
    from PIL import Image  # lazy import — PIL not needed at module load time

    src_path = Path(src_path)
    proxy_dir = Path(proxy_dir)
    proxy_path = proxy_dir / image_name

    if proxy_path.exists():
        return str(proxy_path)

    try:
        img = Image.open(str(src_path))
        if img.mode != "RGB":
            img = img.convert("RGB")
        w, h = img.size
        proxy_w = max(32, w // scale_factor)
        proxy_h = max(32, h // scale_factor)
        proxy = img.resize((proxy_w, proxy_h), Image.BICUBIC)
        proxy_path.parent.mkdir(parents=True, exist_ok=True)
        proxy.save(str(proxy_path), format="PNG", compress_level=6)
        return str(proxy_path)
    except Exception as e:
        print(f"  [WARN] generate_proxy_image failed for {image_name}: {e}", file=sys.stderr)
        return None


# ---------------------------------------------------------------------------
# R4 — Curriculum mixing schedule
# ---------------------------------------------------------------------------

# Stage 2 curriculum: (native_ratio, proxy_ratio)
_CURRICULUM = {
    1: (1.0, 0.0),     # Stage 1: all native (no dual-res yet)
    2: (0.8, 0.2),     # Stage 2 start: 80:20 native:proxy
    3: (0.5, 0.5),     # Stage 2 mid: 50:50
    4: (0.3, 0.7),     # Stage 2 end: 30:70
}


def curriculum_mix_ratio(stage: int) -> tuple[float, float]:
    """Return (native_ratio, proxy_ratio) for the given training stage.

    Parameters
    ----------
    stage : training stage number (1–4 mapped to curriculum steps).

    Returns
    -------
    (native_ratio, proxy_ratio) as floats summing to 1.0.
    """
    if stage in _CURRICULUM:
        return _CURRICULUM[stage]
    if stage > 4:
        return _CURRICULUM[4]  # Stay at final ratio
    return _CURRICULUM[1]


In [ ]:
%%writefile preprocess/common/io.py
"""Shared I/O for preprocessing tiers.

Handles:
    - Manifest tracking (id, dataset, split, processed, output_shard)
    - PNG writing (8-bit, 3-channel, lossless)
    - JSONL line writing
    - Tar sharding (≤2000 files per shard)

Public API:
    Manifest  — tracks processing state per sample id
    write_png(arr, path) — write 8-bit 3-channel PNG
    append_jsonl(sample, path) — append one JSONL line
    ShardWriter — manages tar shards with file-count caps
"""

from __future__ import annotations

import json
import tarfile
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------

class Manifest:
    """Track which sample ids have been processed.

    Backed by a simple JSONL file.  On restart, load existing manifest
    and skip any id already marked processed=True.

    File format (one line per sample):
        {"id": "...", "dataset": "...", "split": "...", "processed": true, "output_shard": "shard_000"}
    """

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self._entries: dict[str, dict[str, Any]] = {}
        self._load()

    def _load(self) -> None:
        if not self.path.exists():
            return
        with open(self.path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                self._entries[entry["id"]] = entry

    def is_processed(self, sample_id: str) -> bool:
        return self._entries.get(sample_id, {}).get("processed", False)

    def mark_processed(
        self,
        sample_id: str,
        dataset: str,
        split: str,
        output_shard: str,
    ) -> None:
        entry = {
            "id": sample_id,
            "dataset": dataset,
            "split": split,
            "processed": True,
            "output_shard": output_shard,
        }
        self._entries[sample_id] = entry
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")

    def count_processed(self) -> int:
        return sum(1 for e in self._entries.values() if e.get("processed"))

    def count_total(self) -> int:
        return len(self._entries)


# ---------------------------------------------------------------------------
# PNG writing
# ---------------------------------------------------------------------------

def write_png(arr: np.ndarray, path: str | Path) -> None:
    """Write a numpy array as an 8-bit, 3-channel, lossless PNG.

    Parameters
    ----------
    arr : (H, W, 3) uint8 array, or (H, W) / (H, W, 1) that will be
          converted to 3-channel by replication.
    path : output file path.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    elif arr.ndim == 3 and arr.shape[2] == 1:
        arr = np.concatenate([arr, arr, arr], axis=-1)

    assert arr.ndim == 3 and arr.shape[2] == 3, \
        f"Expected (H, W, 3), got {arr.shape}"

    img = Image.fromarray(arr.astype(np.uint8), mode="RGB")
    img.save(str(path), format="PNG", compress_level=6)  # lossless, balanced compression


# ---------------------------------------------------------------------------
# JSONL writing
# ---------------------------------------------------------------------------

def append_jsonl(sample: dict[str, Any], path: str | Path) -> None:
    """Append one sample dict as a JSONL line."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")


# ---------------------------------------------------------------------------
# Tar shard writer
# ---------------------------------------------------------------------------

class ShardWriter:
    """Manage output sharding into tar archives with file-count caps.

    Each shard is a tar.gz containing up to max_files PNGs.
    When a shard reaches capacity, a new one is automatically started.

    Usage:
        writer = ShardWriter(output_dir, prefix="tier1_oscd", max_files=2000)
        shard_name = writer.add(png_path)
        writer.close()
    """

    def __init__(
        self,
        output_dir: str | Path,
        prefix: str = "shard",
        max_files: int = 2000,
    ):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.prefix = prefix
        self.max_files = max_files
        self._shard_count = 0
        self._file_count = 0
        self._current_tar: tarfile.TarFile | None = None
        self._open_new_shard()

    def _open_new_shard(self) -> None:
        if self._current_tar is not None:
            self._current_tar.close()
        shard_name = f"{self.prefix}_shard_{self._shard_count:04d}.tar.gz"
        shard_path = self.output_dir / shard_name
        self._current_tar = tarfile.open(str(shard_path), "w:gz")
        self._file_count = 0

    @property
    def current_shard_name(self) -> str:
        return f"{self.prefix}_shard_{self._shard_count:04d}.tar.gz"

    def add(self, file_path: str | Path) -> str:
        """Add a file to the current shard.  Returns the shard name.

        If the shard is full, a new one is started automatically.
        """
        file_path = Path(file_path)
        if self._file_count >= self.max_files:
            self._shard_count += 1
            self._open_new_shard()

        self._current_tar.add(str(file_path), arcname=file_path.name)
        self._file_count += 1
        return self.current_shard_name

    def close(self) -> None:
        if self._current_tar is not None:
            self._current_tar.close()
            self._current_tar = None


In [ ]:
%%writefile preprocess/common/sample.py
"""Shared stratified subsampling for tier scripts with a Selection requirement.

Several Tier 1/2/3 datasets specify a stratified selection fraction (e.g.
"10% stratified by change magnitude", "5% stratified by building-density
quartile", "stratified uniform across 6 categories"). This module gives
every tier script the same tested implementation instead of each rolling
its own `random.sample`, which silently degrades to plain uniform sampling.

Public API:
    stratified_sample(items, key_fn, fraction, seed) -> list
    quantile_bucket(value, edges) -> int
"""

from __future__ import annotations

import random
from collections import defaultdict
from typing import Callable, TypeVar

T = TypeVar("T")


def stratified_sample(
    items: list[T],
    key_fn: Callable[[T], object],
    fraction: float,
    seed: int = 42,
) -> list[T]:
    """Sample `fraction` of items, proportionally within each key_fn(item) group.

    Every group contributes at least 1 item (if it has any), so small
    strata aren't wiped out by rounding down.
    """
    if fraction >= 1.0:
        return list(items)
    if not items:
        return []

    groups: dict[object, list[T]] = defaultdict(list)
    for it in items:
        groups[key_fn(it)].append(it)

    rng = random.Random(seed)
    out: list[T] = []
    for group in groups.values():
        rng.shuffle(group)
        n = max(1, round(len(group) * fraction))
        out.extend(group[: min(n, len(group))])

    rng.shuffle(out)
    return out


def quantile_bucket(value: float, edges: list[float]) -> int:
    """Bucket a scalar into len(edges)+1 bins given ascending edges.

    quantile_bucket(x, [0.25, 0.5, 0.75]) returns 0..3 (quartiles).
    """
    for i, e in enumerate(edges):
        if value <= e:
            return i
    return len(edges)


def quantile_edges(values: list[float], n_buckets: int) -> list[float]:
    """Compute the (n_buckets - 1) quantile edges that split `values` evenly."""
    if not values or n_buckets <= 1:
        return []
    import numpy as np

    qs = [100.0 * i / n_buckets for i in range(1, n_buckets)]
    return [float(v) for v in np.percentile(values, qs)]


In [ ]:
%%writefile preprocess/common/sar.py
"""R3 — SAR pseudo-RGB generation and R5 — SAR GSD downsampling.

R3 pipeline:
    1. Linear → dB:  dB = 10·log10(x + ε)
    2. Clip:  VV ∈ [-25, 0] dB,  VH ∈ [-30, -5] dB
    3. Map (dual-pol):  R = VV(dB),  G = VH(dB),  B = (VV − VH) dB post-clip
    4. Map (quad-pol SpaceNet6):  R = HH,  G = VV,  B = VH

R5 SAR GSD:
    SpaceNet6 SAR (0.5m) and SARDet-100K finer patches are downsampled
    into the RISAT proxy band (~2–10 m).  BigEarthNet S1 (10 m) is the
    coarse anchor and is left as-is.

Public API:
    linear_to_db(arr, epsilon)
    clip_vv_db(arr) / clip_vh_db(arr)
    sar_pseudo_rgb(vv, vh, hh)
    downsample_sar(arr, target_gsd, native_gsd)
"""

from __future__ import annotations

import numpy as np


# ---------------------------------------------------------------------------
# R3 — dB conversion and clipping
# ---------------------------------------------------------------------------

def linear_to_db(arr: np.ndarray, epsilon: float = 1e-10) -> np.ndarray:
    """Convert linear-scale SAR amplitude/power to decibels.

    Parameters
    ----------
    arr : numpy array (any shape), linear scale.
    epsilon : small constant to avoid log10(0).

    Returns
    -------
    numpy array in dB (same shape).

    Notes
    -----
    np.clip(arr, epsilon, None) clamps the lower bound to epsilon.
    This handles zero AND negative values (from floating-point noise
    in upstream denoising or VV-VH math) — negatives become ~-100 dB
    rather than producing NaN.  This is intentional: SAR linear power
    should never be negative, so any negative values are artifacts.
    """
    return 10.0 * np.log10(np.clip(arr, epsilon, None))


def clip_vv_db(arr: np.ndarray) -> np.ndarray:
    """Clip VV polarisation to [-25, 0] dB (R3 spec)."""
    return np.clip(arr, -25.0, 0.0)


def clip_vh_db(arr: np.ndarray) -> np.ndarray:
    """Clip VH polarisation to [-30, -5] dB (R3 spec)."""
    return np.clip(arr, -30.0, -5.0)


# ---------------------------------------------------------------------------
# R3 — Pseudo-RGB construction
# ---------------------------------------------------------------------------

def sar_pseudo_rgb(
    vv: np.ndarray,
    vh: np.ndarray,
    hh: np.ndarray | None = None,
) -> np.ndarray:
    """Build a 3-channel SAR pseudo-RGB image.

    Dual-pol (Sentinel-1, RISAT):
        R = VV(dB)  clipped to [-25, 0]
        G = VH(dB)  clipped to [-30, -5]
        B = (VV − VH) in dB, post-clipping

    Quad-pol (SpaceNet6 SAR):
        R = HH(dB)
        G = VV(dB)
        B = VH(dB)

    Parameters
    ----------
    vv, vh : 2-D numpy arrays in linear scale.
    hh : 2-D numpy array in linear scale (quad-pol only).  If None,
         dual-pol mapping is used.

    Returns
    -------
    (H, W, 3) uint8 array in [0, 255], suitable for PNG output.
    """
    vv_db = linear_to_db(vv)
    vh_db = linear_to_db(vh)

    if hh is not None:
        # Quad-pol: R=HH, G=VV, B=VH  (R3 rule 4)
        hh_db = linear_to_db(hh)
        r = _db_to_uint8(hh_db, vmin=-25, vmax=0)
        g = _db_to_uint8(vv_db, vmin=-25, vmax=0)
        b = _db_to_uint8(vh_db, vmin=-30, vmax=-5)
    else:
        # Dual-pol: R=VV, G=VH, B=VV−VH  (R3 rules 2–3)
        # CRITICAL: clip FIRST, then subtract.  R3 says "post-clipping"
        # meaning: clip VV to [-25,0] dB and VH to [-30,-5] dB FIRST,
        # then compute B = VV_clipped - VH_clipped.  This is NOT the same
        # as computing raw VV-VH and clipping that result.
        vv_clipped = clip_vv_db(vv_db)
        vh_clipped = clip_vh_db(vh_db)
        diff_db = vv_clipped - vh_clipped

        r = _db_to_uint8(vv_clipped, vmin=-25, vmax=0)
        g = _db_to_uint8(vh_clipped, vmin=-30, vmax=-5)
        b = _db_to_uint8(diff_db, vmin=0, vmax=25)

    return np.stack([r, g, b], axis=-1)


def sar_intensity_pseudo_gray(intensity: np.ndarray) -> np.ndarray:
    """Render a single-channel SAR intensity image as an honest grayscale-in-dB PNG.

    Some sources (SARDet-100K's shipped PNGs, single-band fallback tiles)
    give us amplitude/intensity only — no separate VV/VH. R3's dual-pol
    mapping needs two channels; fabricating a second one (e.g. VH = VV*0.8)
    invents a physically meaningless, constant ratio real backscatter
    never has. Converting the one real channel to dB and replicating it
    across R/G/B is the honest "pseudo-RGB" for single-pol data — visually
    a grayscale SAR render, same R3 dB pipeline, no invented signal.
    """
    db = linear_to_db(intensity)
    db_clipped = clip_vv_db(db)  # reuse VV's [-25, 0] range as the general SAR floor
    gray = _db_to_uint8(db_clipped, vmin=-25, vmax=0)
    return np.stack([gray, gray, gray], axis=-1)


def _db_to_uint8(arr: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    """Linearly map a dB-range array to uint8 [0, 255]."""
    normalized = (arr - vmin) / (vmax - vmin)
    normalized = np.clip(normalized, 0.0, 1.0)
    return (normalized * 255).astype(np.uint8)


# ---------------------------------------------------------------------------
# R5 — SAR GSD downsampling
# ---------------------------------------------------------------------------

def downsample_sar(
    arr: np.ndarray,
    target_gsd: float,
    native_gsd: float,
) -> np.ndarray:
    """Downsample a SAR image from native GSD to target GSD.

    Uses area interpolation (averaging) to preserve radiometric integrity.
    If native_gsd <= target_gsd (already coarser or equal), returns as-is.

    Parameters
    ----------
    arr : (H, W) or (H, W, C) numpy array.
    target_gsd : desired ground sampling distance in metres.
    native_gsd : current GSD in metres.

    Returns
    -------
    Downsampled array (or original if no downsampling needed).
    """
    if native_gsd >= target_gsd:
        return arr  # Already at or coarser than target

    scale = native_gsd / target_gsd  # < 1.0
    new_h = max(1, int(round(arr.shape[0] * scale)))
    new_w = max(1, int(round(arr.shape[1] * scale)))

    if arr.ndim == 2:
        return _resize_2d(arr, new_h, new_w)
    else:
        channels = []
        for c in range(arr.shape[2]):
            channels.append(_resize_2d(arr[:, :, c], new_h, new_w))
        return np.stack(channels, axis=-1)


def _resize_2d(arr: np.ndarray, new_h: int, new_w: int) -> np.ndarray:
    """Resize a 2D array using area averaging (no scipy dependency)."""
    from PIL import Image

    img = Image.fromarray(arr.astype(np.float32) if arr.dtype != np.float32 else arr)
    resized = img.resize((new_w, new_h), resample=Image.Resampling.BOX)
    return np.array(resized, dtype=arr.dtype)


In [ ]:
%%writefile preprocess/common/stats.py
"""R2 — Two-pass global percentile computation for radiometric normalization.

Percentile clipping (2nd/98th) is computed ONCE globally per sensor-and-band,
not per-image.  This module provides an accumulator that processes images in
two passes: first to gather statistics, then to apply clipping.

Public API:
    GlobalPercentileStats  — accumulator class
    clip_percentile(arr, low, high) -> np.ndarray
"""

from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np


class GlobalPercentileStats:
    """Accumulate per-band statistics across many images for global percentiles.

    Usage (two-pass):
        stats = GlobalPercentileStats()

        # Pass 1: accumulate from each image
        for img_path in image_paths:
            arr = load_image(img_path)  # (H, W, C) or (H, W)
            stats.accumulate(arr, bands=["B4", "B3", "B2"])

        # Compute global percentiles
        percentiles = stats.compute_percentiles(low=2, high=98)

        # Pass 2: apply clipping
        for img_path in image_paths:
            arr = load_image(img_path)
            clipped = stats.apply_clip(arr, bands=["B4", "B3", "B2"])

    The accumulator stores per-band min, max, and a reservoir sample for
    percentile estimation.  For large datasets, the reservoir is capped at
    ``max_samples`` per band to bound memory usage.
    """

    def __init__(self, max_samples: int = 50_000):
        self.max_samples = max_samples
        self._samples: dict[str, list[np.ndarray]] = defaultdict(list)
        self._counts: dict[str, int] = defaultdict(int)

    def accumulate(
        self,
        arr: np.ndarray,
        bands: list[str] | None = None,
    ) -> None:
        """Accumulate statistics from a single image array.

        Parameters
        ----------
        arr : (H, W, C) or (H, W) numpy array.
        bands : list of band names, one per channel.  If arr is 2D,
                 pass a single-element list like ["VV"].
        """
        if arr.ndim == 2:
            arr = arr[:, :, np.newaxis]
            if bands is None:
                bands = ["band_0"]
        if bands is None:
            bands = [f"band_{i}" for i in range(arr.shape[2])]

        for c, band_name in enumerate(bands):
            channel = arr[:, :, c].astype(np.float64).ravel()

            # Reservoir sampling: cap memory at max_samples
            n_new = len(channel)
            current_count = self._counts[band_name]

            if current_count >= self.max_samples:
                # Already full — skip
                continue

            available = self.max_samples - current_count
            if n_new <= available:
                self._samples[band_name].append(channel)
            else:
                # Take a random subsample to fill the reservoir
                indices = np.random.choice(n_new, size=available, replace=False)
                self._samples[band_name].append(channel[indices])

            self._counts[band_name] = current_count + n_new

    def compute_percentiles(
        self,
        low: float = 2.0,
        high: float = 98.0,
    ) -> dict[str, tuple[float, float]]:
        """Compute global (low, high) percentiles for each accumulated band.

        Returns dict mapping band_name → (low_val, high_val).
        """
        result: dict[str, tuple[float, float]] = {}
        for band_name, sample_list in self._samples.items():
            if not sample_list:
                raise ValueError(f"No data accumulated for band '{band_name}'")
            combined = np.concatenate(sample_list)
            low_val = float(np.percentile(combined, low))
            high_val = float(np.percentile(combined, high))
            result[band_name] = (low_val, high_val)
        return result

    def apply_clip(
        self,
        arr: np.ndarray,
        percentiles: dict[str, tuple[float, float]],
        bands: list[str] | None = None,
    ) -> np.ndarray:
        """Apply percentile clipping to an image array.

        Parameters
        ----------
        arr : (H, W, C) or (H, W) numpy array.
        percentiles : dict from compute_percentiles().  This MUST be the
                       output of compute_percentiles() — it is NOT
                       recomputed here.  R2 requires a single, stable,
                       global stat per sensor-band; recomputing would
                       reintroduce per-image inconsistency.
        bands : band names matching the channels in arr.

        Returns
        -------
        Clipped array (same shape, dtype preserved where possible).
        """
        out = arr.copy()
        if out.ndim == 2:
            out = out[:, :, np.newaxis]
            if bands is None:
                bands = ["band_0"]

        if bands is None:
            bands = [f"band_{i}" for i in range(out.shape[2])]

        for c, band_name in enumerate(bands):
            if band_name not in percentiles:
                continue
            low_val, high_val = percentiles[band_name]
            out[:, :, c] = np.clip(out[:, :, c], low_val, high_val)

        # Squeeze back if input was 2D
        if arr.ndim == 2:
            out = out[:, :, 0]
        return out

    def save(self, path: str | Path) -> None:
        """Persist accumulator state to disk (numpy compressed)."""
        np.savez_compressed(
            str(path),
            samples={k: np.concatenate(v) for k, v in self._samples.items()},
            counts=dict(self._counts),
            max_samples=self.max_samples,
        )

    @classmethod
    def load(cls, path: str | Path) -> GlobalPercentileStats:
        """Restore accumulator state from disk."""
        data = np.load(str(path), allow_pickle=True)
        obj = cls(max_samples=int(data["max_samples"]))
        obj._counts = defaultdict(int, data["counts"].item())
        for band_name, samples in data["samples"].item().items():
            obj._samples[band_name] = [samples]
        return obj


def clip_percentile(
    arr: np.ndarray,
    low: float,
    high: float,
) -> tuple[np.ndarray, float, float]:
    """Quick single-image percentile clip (for small datasets / testing).

    Returns (clipped_array, low_val, high_val).
    """
    low_val = float(np.percentile(arr, low))
    high_val = float(np.percentile(arr, high))
    return np.clip(arr, low_val, high_val), low_val, high_val


## `preprocess/`

In [ ]:
%%writefile preprocess/merge_and_package.py
"""Merge and package processed JSONLs into final dataset.

Validates every sample against frozen schema, merges tier outputs into a
single dataset.jsonl, and optionally tars image shards for Kaggle upload.

Usage:
    python -m preprocess.merge_and_package --tier-dirs data/oscd_output data/vrsbench_output --output-dir data/final
"""

from __future__ import annotations

import json
import time
from collections import Counter
from pathlib import Path

from preprocess.common.io import ShardWriter
from preprocess.validator import validate_sample


def load_jsonl(path: Path) -> list[dict]:
    """Load all samples from a JSONL file."""
    samples = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


def validate_and_filter(
    samples: list[dict],
    tier_name: str,
) -> tuple[list[dict], list[dict]]:
    """Validate samples, returning (valid, errors)."""
    valid = []
    errors = []
    for s in samples:
        ok, errs = validate_sample(s)
        if ok:
            valid.append(s)
        else:
            errors.append({"id": s.get("id", "?"), "tier": tier_name, "errors": errs})
    return valid, errors


def _select_source_files(jsonl_files: list[Path]) -> list[Path]:
    """Pick which JSONL files in a tier dir actually feed the merge.

    split_internal_val.py (when it has run) replaces `<stem>.jsonl` with
    `<stem>_train.jsonl` + `<stem>_val_internal.jsonl` — those carry the
    real split tags and must be preferred. Falling back to the original
    `<stem>.jsonl` silently drops every val_internal row (it only exists
    with everything tagged "train"), which is what happened here before:
    merge always skipped `_train`/`_val_internal` files and re-read
    the un-split original.
    """
    by_stem = {p.stem: p for p in jsonl_files}
    train_suffix = "_train"
    val_suffix = "_val_internal"

    split_bases = {
        stem[: -len(train_suffix)]
        for stem in by_stem
        if stem.endswith(train_suffix)
    }

    selected: list[Path] = []
    for stem, path in by_stem.items():
        if stem.endswith(train_suffix) or stem.endswith(val_suffix):
            selected.append(path)
            continue
        if stem in split_bases:
            continue  # superseded by <stem>_train.jsonl / _val_internal.jsonl
        selected.append(path)  # no split output for this file — use as-is

    return selected


def merge_tiers(
    tier_dirs: list[Path],
) -> tuple[list[dict], list[dict]]:
    """Merge all tier outputs, validating each sample.

    Returns (all_valid, all_errors).
    """
    all_valid = []
    all_errors = []

    for tier_dir in tier_dirs:
        tier_name = tier_dir.name
        jsonl_files = list(tier_dir.glob("*.jsonl"))
        if not jsonl_files:
            print(f"  [WARN] No JSONL files in {tier_dir}")
            continue

        for jsonl_path in _select_source_files(jsonl_files):
            samples = load_jsonl(jsonl_path)
            valid, errors = validate_and_filter(samples, tier_name)
            all_valid.extend(valid)
            all_errors.extend(errors)
            print(f"  {tier_name}/{jsonl_path.name}: {len(valid)} valid, {len(errors)} errors")

    return all_valid, all_errors


def check_id_uniqueness(samples: list[dict]) -> list[str]:
    """Return list of duplicate IDs, formatted as '<id> (×N)'."""
    seen: dict[str, int] = {}
    for s in samples:
        sid = s.get("id", "")
        seen[sid] = seen.get(sid, 0) + 1
    return [f"{sid} (×{count})" for sid, count in seen.items() if count > 1]


def dedupe_by_id(samples: list[dict]) -> tuple[list[dict], int]:
    """Keep the first occurrence of each id. Returns (deduped, n_dropped).

    "id" is documented as globally unique (Section 4) — the schema can't
    enforce that across files, so this is where cross-tier collisions
    (mostly a same-id `id` scheme reused between two tier scripts) get
    caught before they reach the training set as silent duplicates.
    """
    seen: set[str] = set()
    out: list[dict] = []
    dropped = 0
    for s in samples:
        sid = s.get("id", "")
        if sid in seen:
            dropped += 1
            continue
        seen.add(sid)
        out.append(s)
    return out, dropped


def write_dataset(
    samples: list[dict],
    output_path: Path,
) -> None:
    """Write merged samples to a single JSONL."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")


def tar_directory(
    src_dir: Path,
    tar_dir: Path,
    *,
    prefix: str,
    include_jsonl: bool = False,
    max_files_per_shard: int = 2000,
) -> int:
    """Shard a directory of images into <=max_files_per_shard tar.gz archives.

    Kaggle Dataset uploads cap raw file counts (Section 8), and a single
    giant tar risks losing an entire tier's images to one failed
    upload/session. ShardWriter (already tested) produces
    `<prefix>_shard_0000.tar.gz`, `_0001.tar.gz`, ... in tar_dir.

    Returns total file count across all shards.
    """
    writer = ShardWriter(tar_dir, prefix=prefix, max_files=max_files_per_shard)
    count = 0
    for path in sorted(src_dir.rglob("*")):
        if path.is_file():
            if not include_jsonl and path.suffix == ".jsonl":
                continue
            writer.add(path)
            count += 1
    writer.close()
    return count


def compute_stats(samples: list[dict]) -> dict:
    """Compute dataset statistics."""
    stats: dict = {}
    stats["total_samples"] = len(samples)

    # By dataset
    by_dataset = Counter(s.get("dataset", "?") for s in samples)
    stats["by_dataset"] = dict(by_dataset)

    # By task
    by_task = Counter(s.get("task", "?") for s in samples)
    stats["by_task"] = dict(by_task)

    # By pair_type
    by_pair = Counter(s.get("pair_type", "?") for s in samples)
    stats["by_pair_type"] = dict(by_pair)

    # By modality
    by_mod = Counter(s.get("modality", "?") for s in samples)
    stats["by_modality"] = dict(by_mod)

    # By split
    by_split = Counter(s.get("split", "?") for s in samples)
    stats["by_split"] = dict(by_split)

    # Bbox coverage
    has_bbox = sum(1 for s in samples if s.get("bbox") is not None)
    stats["bbox_coverage"] = f"{has_bbox}/{len(samples)} ({100*has_bbox/max(1,len(samples)):.1f}%)"

    # Image path stats
    single = sum(1 for s in samples if len(s.get("image_path", [])) == 1)
    paired = sum(1 for s in samples if len(s.get("image_path", [])) == 2)
    stats["image_paths"] = {"single": single, "paired": paired}

    return stats


def merge_and_package(
    tier_dirs: list[Path],
    output_dir: Path,
    *,
    tar_shards: bool = False,
) -> dict:
    """Full merge pipeline: validate → deduplicate → merge → package.

    Returns stats dict.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    start = time.time()

    print("Merging tiers...")
    all_valid, all_errors = merge_tiers(tier_dirs)

    if all_errors:
        print(f"\n  {len(all_errors)} validation errors:")
        for e in all_errors[:10]:
            print(f"    {e['id']}: {e['errors']}")
        if len(all_errors) > 10:
            print(f"    ... and {len(all_errors) - 10} more")

    # Check + drop ID collisions — a duplicate id must not reach the
    # training set silently (it did before: this was report-only).
    dupes = check_id_uniqueness(all_valid)
    if dupes:
        print(f"\n  {len(dupes)} duplicate IDs (keeping first occurrence):")
        for d in dupes[:10]:
            print(f"    {d}")
    all_valid, n_dropped = dedupe_by_id(all_valid)

    # Write merged dataset
    dataset_path = output_dir / "dataset.jsonl"
    write_dataset(all_valid, dataset_path)
    print(f"\n  Wrote {len(all_valid)} samples to {dataset_path}")

    # Compute stats
    stats = compute_stats(all_valid)
    stats["errors"] = len(all_errors)
    stats["duplicate_ids"] = len(dupes)

    # Save stats
    stats_path = output_dir / "stats.json"
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    # Shard-tar images if requested (Kaggle Dataset file-count cap: Section 8)
    if tar_shards:
        for tier_dir in tier_dirs:
            images_dir = tier_dir / "images"
            if images_dir.exists():
                count = tar_directory(
                    images_dir, output_dir, prefix=f"{tier_dir.name}_images",
                )
                print(f"  Tared {count} files → {output_dir}/{tier_dir.name}_images_shard_*.tar.gz")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed:.1f}s")

    return stats


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Merge tier outputs into final dataset",
    )
    parser.add_argument(
        "--tier-dirs", type=Path, nargs="+", required=True,
        help="Tier output directories to merge",
    )
    parser.add_argument(
        "--output-dir", type=Path, required=True,
        help="Output directory for merged dataset",
    )
    parser.add_argument(
        "--tar-shards", action="store_true",
        help="Tar image directories for Kaggle upload",
    )
    args = parser.parse_args()

    stats = merge_and_package(
        args.tier_dirs,
        args.output_dir,
        tar_shards=args.tar_shards,
    )

    print(f"\nStats:")
    print(json.dumps(stats, indent=2))


In [ ]:
%%writefile preprocess/sanity_check.py
"""Post-download sanity checks for all datasets.

Verifies file counts, annotation structure, and split proportions
before preprocessing begins. Fail fast if mirror is truncated/reshuffled.

Usage:
    python -m preprocess.sanity_check --dataset oscd --input-dir data/oscd
    python -m preprocess.sanity_check --dataset all --input-dir data/
"""

from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any


class SanityCheckError(Exception):
    """Raised when a sanity check fails."""
    pass


def check_oscd(input_dir: Path) -> dict[str, Any]:
    """Check OSCD dataset structure."""
    results = {"dataset": "oscd", "checks": []}

    # Check images directory
    images_dir = input_dir / "images"
    if not images_dir.exists():
        raise SanityCheckError(f"Missing images directory: {images_dir}")

    locations = [d for d in images_dir.iterdir() if d.is_dir()]
    results["checks"].append({"name": "locations", "count": len(locations), "expected": 24})

    if len(locations) < 24:
        print(f"  [WARN] Expected 24 locations, found {len(locations)}")

    # Check each location has imgs_1 and imgs_2
    missing_pairs = []
    for loc in locations:
        if not (loc / "imgs_1").exists() or not (loc / "imgs_2").exists():
            missing_pairs.append(loc.name)

    results["checks"].append({
        "name": "complete_pairs",
        "missing": len(missing_pairs),
        "missing_locations": missing_pairs[:5],
    })

    n_complete = len(locations) - len(missing_pairs)
    if locations and n_complete == 0:
        raise SanityCheckError(
            f"No complete OSCD pairs found in {len(locations)} locations "
            f"(all missing imgs_1/imgs_2) — mirror looks truncated or wrong path"
        )

    # Check labels
    labels_dir = input_dir / "labels"
    if labels_dir.exists():
        label_locs = [d for d in labels_dir.iterdir() if d.is_dir()]
        results["checks"].append({"name": "label_dirs", "count": len(label_locs)})

    results["status"] = "pass"
    return results


def check_vrsbench(input_dir: Path) -> dict[str, Any]:
    """Check VRSBench dataset structure."""
    results = {"dataset": "vrsbench", "checks": []}

    # Check main JSON
    json_path = input_dir / "VRSBench_train.json"
    if not json_path.exists():
        raise SanityCheckError(f"Missing VRSBench_train.json")

    with open(json_path) as f:
        data = json.load(f)

    if len(data) == 0:
        raise SanityCheckError(f"VRSBench_train.json is empty — mirror looks truncated")

    results["checks"].append({"name": "total_entries", "count": len(data)})

    # Check task distribution
    tasks = {}
    for entry in data:
        conv = entry.get("conversations", [{}])[0].get("value", "")
        if "[caption]" in conv:
            tasks["caption"] = tasks.get("caption", 0) + 1
        elif "[refer]" in conv:
            tasks["refer"] = tasks.get("refer", 0) + 1
        elif "[vqa]" in conv:
            tasks["vqa"] = tasks.get("vqa", 0) + 1

    results["checks"].append({"name": "task_distribution", "tasks": tasks})

    # Check image zips exist
    train_zip = input_dir / "Images_train.zip"
    val_zip = input_dir / "Images_val.zip"
    if not train_zip.exists() and not val_zip.exists():
        raise SanityCheckError(
            f"Neither Images_train.zip nor Images_val.zip found in {input_dir}"
        )
    results["checks"].append({
        "name": "image_zips",
        "train_exists": train_zip.exists(),
        "val_exists": val_zip.exists(),
        "train_size_gb": round(train_zip.stat().st_size / 1e9, 2) if train_zip.exists() else 0,
    })

    results["status"] = "pass"
    return results


def check_bigearthnet(input_dir: Path) -> dict[str, Any]:
    """Check BigEarthNet dataset structure (this script's Kaggle-side
    metadata.parquet + s2_npy/s1_npy layout — see tier0_bigen.py)."""
    results = {"dataset": "bigearthnet", "checks": []}

    metadata_path = input_dir / "metadata.parquet"
    h5_files = list(input_dir.glob("*.h5")) + list(input_dir.glob("*.hdf5"))
    s2_dir = input_dir / "s2_npy"

    if not metadata_path.exists() and not h5_files:
        raise SanityCheckError(
            f"Neither metadata.parquet nor .h5/.hdf5 files found in {input_dir}"
        )

    if metadata_path.exists():
        import pandas as pd
        metadata = pd.read_parquet(metadata_path)
        if len(metadata) == 0:
            raise SanityCheckError(f"metadata.parquet in {input_dir} has 0 rows")
        results["checks"].append({"name": "total_patches", "count": len(metadata)})

        if s2_dir.exists():
            n_npy = len(list(s2_dir.glob("*.npy")))
            if n_npy == 0:
                raise SanityCheckError(f"s2_npy/ in {input_dir} has no .npy patches")
            results["checks"].append({"name": "s2_npy_count", "count": n_npy})

        if "country" in metadata.columns:
            countries = metadata["country"].value_counts().to_dict()
            results["checks"].append({"name": "countries", "count": len(countries)})

        if "labels" in metadata.columns:
            results["checks"].append({"name": "has_labels_column", "value": True})
    else:
        results["checks"].append({"name": "h5_files", "count": len(h5_files)})

    results["status"] = "pass"
    return results


def check_rsvqa_hr(input_dir: Path) -> dict[str, Any]:
    """Check RSVQA-HR dataset structure."""
    results = {"dataset": "rsvqa_hr", "checks": []}

    found_split = False
    for split in ["train", "val", "test"]:
        split_path = input_dir / f"{split}.json"
        if split_path.exists():
            found_split = True
            with open(split_path) as f:
                data = json.load(f)
            count = len(data) if isinstance(data, list) else len(data.get("questions", []))
            results["checks"].append({"name": f"{split}_count", "count": count})

    if not found_split:
        raise SanityCheckError(f"No train.json/val.json/test.json found in {input_dir}")

    images_dir = input_dir / "images"
    n_images = 0
    if images_dir.exists():
        n_images = len(list(images_dir.glob("*.png"))) + len(list(images_dir.glob("*.jpg")))
        results["checks"].append({"name": "image_count", "count": n_images})
    if not images_dir.exists() or n_images == 0:
        raise SanityCheckError(f"No images found under {images_dir}")

    results["status"] = "pass"
    return results


def check_cdvqa(input_dir: Path) -> dict[str, Any]:
    """Check CDVQA dataset structure."""
    results = {"dataset": "cdvqa", "checks": []}

    found_split = False
    for split in ["train", "val"]:
        split_path = input_dir / f"{split}.json"
        if split_path.exists():
            found_split = True
            with open(split_path) as f:
                data = json.load(f)
            count = len(data) if isinstance(data, list) else len(data.get("questions", []))
            results["checks"].append({"name": f"{split}_count", "count": count})

    if not found_split:
        raise SanityCheckError(f"Neither train.json nor val.json found in {input_dir}")

    # Reject test splits
    test_path = input_dir / "test.json"
    if test_path.exists():
        print("  [WARN] test.json found — will be EXCLUDED from training")

    results["status"] = "pass"
    return results


def check_levir_cd(input_dir: Path) -> dict[str, Any]:
    """Check LEVIR-CD dataset structure."""
    results = {"dataset": "levir_cd", "checks": []}

    found_any = False
    for split in ["train", "val", "test"]:
        split_dir = input_dir / split
        if split_dir.exists():
            a_dir = split_dir / "A"
            b_dir = split_dir / "B"
            label_dir = split_dir / "label"

            n_before = len(list(a_dir.glob("*.png"))) if a_dir.exists() else 0
            n_after = len(list(b_dir.glob("*.png"))) if b_dir.exists() else 0
            n_labels = len(list(label_dir.glob("*.png"))) if label_dir.exists() else 0
            if n_before > 0 or n_after > 0:
                found_any = True

            results["checks"].append({
                "name": f"{split}_counts",
                "before": n_before,
                "after": n_after,
                "labels": n_labels,
            })

    if not found_any:
        raise SanityCheckError(f"No LEVIR-CD before/after images found under {input_dir}")

    train_dir = input_dir / "train"
    if not train_dir.exists() or not list((train_dir / "A").glob("*.png")):
        raise SanityCheckError(
            f"train/ split is empty or missing under {input_dir} — "
            f"tier1_levir_cd.py only trains from train/"
        )

    results["status"] = "pass"
    return results


def check_spacenet6(input_dir: Path) -> dict[str, Any]:
    """Check SpaceNet 6 dataset structure."""
    results = {"dataset": "spacenet6", "checks": []}

    train_dir = input_dir / "train"
    if not train_dir.exists():
        raise SanityCheckError(f"train/ directory not found in {input_dir}")

    images_dir = train_dir / "images"
    labels_dir = train_dir / "labels"

    n_images = len(list(images_dir.glob("*.tif"))) if images_dir.exists() else 0
    n_labels = len(list(labels_dir.glob("*.geojson"))) if labels_dir.exists() else 0

    if n_images == 0:
        raise SanityCheckError(f"No .tif tiles found under {images_dir}")

    results["checks"].append({
        "name": "train_counts",
        "images": n_images,
        "labels": n_labels,
    })

    sar_dir = train_dir / "sar"
    if sar_dir.exists():
        n_sar = len(list(sar_dir.glob("*.tif")))
        results["checks"].append({"name": "sar_count", "count": n_sar})

    results["status"] = "pass"
    return results


def check_sardet(input_dir: Path) -> dict[str, Any]:
    """Check SARDet-100K dataset structure."""
    results = {"dataset": "sardet", "checks": []}

    images_dir = input_dir / "images"
    labels_dir = input_dir / "labels"

    n_images = 0
    if images_dir.exists():
        n_images = len(list(images_dir.glob("*.png"))) + len(list(images_dir.glob("*.jpg")))
        results["checks"].append({"name": "image_count", "count": n_images})
    if not images_dir.exists() or n_images == 0:
        raise SanityCheckError(f"No images found under {images_dir}")

    if labels_dir.exists():
        n_labels = len(list(labels_dir.glob("*.txt")))
        results["checks"].append({"name": "label_count", "count": n_labels})

    results["status"] = "pass"
    return results


def check_sen2lulc(input_dir: Path) -> dict[str, Any]:
    """Check Sen-2 LULC dataset structure."""
    results = {"dataset": "sen2lulc", "checks": []}

    found_metadata = False
    for name in ["metadata.csv", "annotations.json"]:
        meta_path = input_dir / name
        if meta_path.exists():
            found_metadata = True
            results["checks"].append({"name": "metadata_file", "file": name, "exists": True})

    if not found_metadata:
        raise SanityCheckError(f"Neither metadata.csv nor annotations.json found in {input_dir}")

    images_dir = input_dir / "images"
    n_images = 0
    if images_dir.exists():
        n_images = len(list(images_dir.glob("*.png")))
        results["checks"].append({"name": "image_count", "count": n_images})
    if not images_dir.exists() or n_images == 0:
        raise SanityCheckError(f"No images found under {images_dir}")

    results["status"] = "pass"
    return results


# Registry of sanity check functions
CHECKS = {
    "oscd": check_oscd,
    "vrsbench": check_vrsbench,
    "bigen": check_bigearthnet,
    "bigearthnet": check_bigearthnet,
    "rsvqa_hr": check_rsvqa_hr,
    "cdvqa": check_cdvqa,
    "levir_cd": check_levir_cd,
    "sn6_opt": check_spacenet6,
    "sn6_sar": check_spacenet6,
    "spacenet6": check_spacenet6,
    "sardet": check_sardet,
    "sen2lulc": check_sen2lulc,
}


def run_sanity_check(dataset: str, input_dir: Path) -> dict[str, Any]:
    """Run sanity check for a specific dataset."""
    check_fn = CHECKS.get(dataset)
    if check_fn is None:
        raise ValueError(f"Unknown dataset: {dataset}. Available: {list(CHECKS.keys())}")

    print(f"Checking {dataset} at {input_dir}...")
    results = check_fn(input_dir)
    print(f"  Status: {results['status']}")
    for check in results["checks"]:
        print(f"  {check}")

    return results


def run_all_checks(data_dir: Path) -> dict[str, dict]:
    """Run sanity checks for all known datasets in a directory."""
    all_results = {}

    for dataset in CHECKS:
        dataset_dir = data_dir / dataset
        if dataset_dir.exists():
            try:
                results = run_sanity_check(dataset, dataset_dir)
                all_results[dataset] = results
            except SanityCheckError as e:
                print(f"  [FAIL] {dataset}: {e}")
                all_results[dataset] = {"status": "fail", "error": str(e)}
        else:
            print(f"  [SKIP] {dataset}: directory not found at {dataset_dir}")

    return all_results


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Post-download sanity checks")
    parser.add_argument("--dataset", type=str, required=True,
                        help="Dataset name or 'all'")
    parser.add_argument("--input-dir", type=Path, required=True,
                        help="Dataset or data directory")
    args = parser.parse_args()

    if args.dataset == "all":
        results = run_all_checks(args.input_dir)
    else:
        results = run_sanity_check(args.dataset, args.input_dir)

    # Print summary
    print("\n" + "=" * 60)
    print("SANITY CHECK SUMMARY")
    print("=" * 60)

    if isinstance(results, dict) and "status" in results:
        # Single dataset
        status = results["status"]
        print(f"\n{args.dataset}: {status.upper()}")
    else:
        # All datasets
        for dataset, result in results.items():
            status = result.get("status", "unknown")
            print(f"  {dataset}: {status.upper()}")


In [ ]:
%%writefile preprocess/split_internal_val.py
"""Split internal validation carve-out.

Carves 2-3% stratified val_internal splits from processed training JSONLs.
Stratification is by (dataset, task) to ensure proportional representation.

Usage:
    python -m preprocess.split_internal_val --jsonl-dir data/oscd_output --val-fraction 0.03
"""

from __future__ import annotations

import json
import sys
from collections import defaultdict
from pathlib import Path

from preprocess.common.io import append_jsonl


def load_samples(jsonl_path: Path) -> list[dict]:
    """Load all samples from a JSONL file."""
    samples = []
    with open(jsonl_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


def stratified_split(
    samples: list[dict],
    val_fraction: float = 0.03,
    seed: int = 42,
) -> tuple[list[dict], list[dict]]:
    """Split samples into train and val_internal, stratified by (dataset, task).

    Returns (train_samples, val_samples).
    """
    import random

    # Group by (dataset, task) for stratification
    groups: dict[tuple[str, str], list[dict]] = defaultdict(list)
    for s in samples:
        key = (s.get("dataset", ""), s.get("task", ""))
        groups[key].append(s)

    rng = random.Random(seed)
    train_samples = []
    val_samples = []

    for key, group in groups.items():
        rng.shuffle(group)
        n_val = max(1, int(len(group) * val_fraction))
        # Cap val at len-1 to ensure at least 1 train sample
        n_val = min(n_val, len(group) - 1) if len(group) > 1 else 0
        val_samples.extend(group[:n_val])
        train_samples.extend(group[n_val:])

    return train_samples, val_samples


def split_and_write(
    input_jsonl: Path,
    output_dir: Path,
    *,
    val_fraction: float = 0.03,
    seed: int = 42,
) -> dict[str, int]:
    """Split a single JSONL into train and val_internal, writing both.

    Returns stats: {total, train, val}.
    """
    samples = load_samples(input_jsonl)
    if not samples:
        return {"total": 0, "train": 0, "val": 0}

    train, val = stratified_split(samples, val_fraction=val_fraction, seed=seed)

    # Tag val samples
    for s in val:
        s["split"] = "val_internal"

    # Write outputs
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = input_jsonl.stem

    train_path = output_dir / f"{stem}_train.jsonl"
    val_path = output_dir / f"{stem}_val_internal.jsonl"

    for s in train:
        append_jsonl(s, train_path)
    for s in val:
        append_jsonl(s, val_path)

    return {"total": len(samples), "train": len(train), "val": len(val)}


def split_all_tiers(
    jsonl_dir: Path,
    *,
    val_fraction: float = 0.03,
    seed: int = 42,
) -> dict[str, dict[str, int]]:
    """Split all *_train.jsonl or *.jsonl files in a directory.

    Processes every JSONL in the directory, creating _train.jsonl and
    _val_internal.jsonl pairs.

    Returns stats per input file.
    """
    results = {}
    for jsonl_path in sorted(jsonl_dir.glob("*.jsonl")):
        if "_train" in jsonl_path.stem or "_val" in jsonl_path.stem:
            continue  # Skip already-split files
        stats = split_and_write(
            jsonl_path, jsonl_dir,
            val_fraction=val_fraction, seed=seed,
        )
        results[jsonl_path.name] = stats
    return results


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Carve stratified val_internal splits from processed JSONLs",
    )
    parser.add_argument(
        "--jsonl-dir", type=Path, required=True,
        help="Directory containing processed JSONL files",
    )
    parser.add_argument(
        "--val-fraction", type=float, default=0.03,
        help="Fraction for val split (default: 0.03 = 3%%)",
    )
    parser.add_argument(
        "--seed", type=int, default=42,
        help="Random seed for reproducibility",
    )
    args = parser.parse_args()

    results = split_all_tiers(
        args.jsonl_dir,
        val_fraction=args.val_fraction,
        seed=args.seed,
    )

    for name, stats in results.items():
        print(f"{name}: {stats['total']} total → {stats['train']} train + {stats['val']} val")


In [ ]:
%%writefile preprocess/tier0_bigen.py
"""Tier 0 — BigEarthNet (reBEN v2.0 / BigEarthNet-MM) preprocessing.

BigEarthNet provides co-registered Sentinel-2 multispectral and Sentinel-1
SAR image patches with 19-class CORINE land cover labels. Used as the
primary RS domain adaptation backbone (~35% of corpus) AND as the primary
source for Mandate 4 (cross-modal optical+SAR) samples, since S1/S2 patches
are already co-registered — no reprojection step needed.

Kaggle input layout (this script's contract — a Kaggle Dataset built by the
acquisition step, not BigEarthNet's own on-disk format):
    <input_dir>/
        metadata.parquet         # columns: patch_id, labels (list[int] class
                                  # indices 0-18 or a 19-length multihot list),
                                  # country (optional), cloud_pct (optional)
        s2_npy/<patch_id>.npy    # (12, H, W) uint16 — REQUIRED
        s1_npy/<patch_id>.npy    # (2, H, W) float32, [VV, VH] linear power —
                                  # OPTIONAL; when present, a SAR VQA sample and
                                  # a cross-modal fusion sample are also emitted

Why this layout and not raw BigEarthNet-MM HDF5: the spec's acquisition table
(Section 9) says "HF lc-col/bigearthnet (HDF5) -> streaming -> filter -> PNG",
but streaming schema field names for that specific HF dataset aren't
verifiable without network access from here. metadata.parquet + per-patch
.npy is the shape a Kaggle-side acquisition notebook can produce from that
stream in one pass, and it's what this script (and its tests) are built
against.
# ponytail: if the real HF dataset ships different field names, adjust
# load_bigen_arrays()'s column lookups — everything downstream is unaffected.

Spec treatment:
    - R1: Sentinel-2 B4/B3/B2 -> RGB (NIR dropped)
    - R2: percentile normalization (per-patch here; global pass is stats.py,
      run separately over the written PNGs before training)
    - R3: SAR pseudo-RGB (VV, VH, VV-VH) via common.sar.sar_pseudo_rgb
    - R5: native 10m (R5 anchor)
    - Products per pair (when S1 available): optical VQA, SAR VQA, 1x2
      concat cross-modal fusion sample
    - dataset enum: "bigen"
    - gsd_bucket: [GSD:10m]

Usage:
    python -m preprocess.tier0_bigen --input-dir <path> --output-dir <path> --max-samples 60000
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

from preprocess.common.concat import concat_horizontal
from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.common.sar import sar_pseudo_rgb
from preprocess.validator import validate_sample

# Sentinel-2 band indices (0-based) for RGB extraction.
# BigEarthNet-MM band order: B01,B02,B03,B04,B05,B06,B07,B08,B8A,B09,B10,B11(,B12)
# We need B04(idx=3), B03(idx=2), B02(idx=1).
_S2_RGB_INDICES = [3, 2, 1]  # B04, B03, B02

# 19-class CORINE labels
_CORINE_CLASSES = [
    "Continuous urban fabric",
    "Discontinuous urban fabric",
    "Industrial or commercial units",
    "Road and rail networks and associated land",
    "Port areas",
    "Airports",
    "Mineral extraction sites",
    "Dump sites",
    "Construction sites",
    "Green urban areas",
    "Sports and leisure facilities",
    "Non-irrigated arable land",
    "Permanently irrigated land",
    "Rice fields",
    "Vineyards",
    "Fruit trees and berry plantations",
    "Pastures",
    "Annual crops associated with permanent crops",
    "Complex cultivation patterns",
]


def s2_to_rgb_uint8(image: np.ndarray) -> np.ndarray:
    """Extract B04/B03/B02 from a (12+, H, W) Sentinel-2 array as uint8 RGB.

    Per-patch p98 stretch. R2's global percentile pass is a separate step
    (stats.py) applied to the written PNGs, not duplicated here.
    """
    rgb = image[_S2_RGB_INDICES]  # (3, H, W)
    rgb = np.transpose(rgb, (1, 2, 0)).astype(np.float32)  # (H, W, 3)

    p98 = np.percentile(rgb, 98)
    if p98 > 0:
        rgb = np.clip(rgb, 0, p98) / p98 * 255.0
    return rgb.astype(np.uint8)


def _labels_to_multihot(labels: Any, n_classes: int = 19) -> np.ndarray:
    """Normalize a metadata 'labels' cell to a 19-length multihot vector.

    Accepts either a list of active class indices (e.g. [0, 11]) or an
    already-multihot list/array of length n_classes.
    """
    vec = np.zeros(n_classes, dtype=np.int32)
    if labels is None:
        return vec
    if isinstance(labels, str) or not hasattr(labels, "__iter__"):
        labels = [labels]
    labels = list(labels)

    if len(labels) == n_classes and all(v in (0, 1) for v in labels if v is not None):
        return np.array([int(v) for v in labels], dtype=np.int32)

    for v in labels:
        try:
            idx = int(v)
        except (TypeError, ValueError):
            continue
        if 0 <= idx < n_classes:
            vec[idx] = 1
    return vec


def parse_bigen_sample(
    sample: dict[str, Any],
    idx: int,
    *,
    max_classes: int = 19,
) -> dict[str, Any] | None:
    """Parse a single BigEarthNet-MM S2 patch + label into our JSONL format.

    `sample` must have "image" ((12+, H, W) array) and "label" (multihot
    or index-list). image_path/gsd_bucket are left for the caller to fill
    in once the PNG has actually been written.

    Returns sample dict or None if invalid.
    """
    image = sample.get("image")
    if image is None:
        return None

    image = np.asarray(image)
    if image.ndim != 3 or image.shape[0] < 4:
        return None

    label = sample.get("label")
    if label is None:
        return None

    label = np.asarray(label)
    active_classes = [
        _CORINE_CLASSES[i]
        for i in range(min(len(label), max_classes))
        if label[i] > 0
    ]
    if not active_classes:
        return None

    response = ", ".join(active_classes)
    sample_id = f"bigen_{idx:08d}"

    return {
        "id": sample_id,
        "dataset": "bigen",
        "task": "vqa",
        "image_path": [],  # Filled by caller after PNG write
        "pair_type": "single",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": "What land cover types are present in this image?",
        "response": response,
        "bbox": None,
        "modality": "optical",
    }


def load_bigen_arrays(
    input_dir: Path,
    row: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray | None]:
    """Load one BigEarthNet patch's S2 array (required) and S1 array (optional).

    Returns (s2_array, s1_array_or_None).
    Raises FileNotFoundError if the required S2 patch is missing.
    """
    patch_id = str(row.get("patch_id") or row.get("id") or "").strip()
    if not patch_id:
        raise ValueError("metadata row has no patch_id/id")

    s2_path = input_dir / "s2_npy" / f"{patch_id}.npy"
    if not s2_path.exists():
        raise FileNotFoundError(f"Missing S2 patch: {s2_path}")
    s2 = np.load(s2_path)

    s1 = None
    s1_path = input_dir / "s1_npy" / f"{patch_id}.npy"
    if s1_path.exists():
        s1 = np.load(s1_path)

    return s2, s1


def _build_sar_and_fusion_samples(
    sample_id: str,
    opt_png: Path,
    opt_rgb: np.ndarray,
    opt_response: str,
    s1: np.ndarray,
    img_dir: Path,
    native_bucket: str,
) -> list[dict[str, Any]]:
    """Build the SAR-VQA and cross-modal fusion samples for a patch with S1 data.

    Satisfies Mandate 4 (cross-modal optical+SAR): BigEarthNet's S1/S2
    patches are already co-registered, so this is the cheapest correct
    source for `pair_type: cross-modal` training rows — nothing else in
    the pipeline currently emits any.
    """
    vv, vh = s1[0].astype(np.float32), s1[1].astype(np.float32)
    sar_rgb = sar_pseudo_rgb(vv, vh)

    sar_png = img_dir / f"{sample_id}_sar.png"
    write_png(sar_rgb, sar_png)

    sar_sample = {
        "id": f"{sample_id}_sar",
        "dataset": "bigen",
        "task": "vqa",
        "image_path": [str(sar_png)],
        "pair_type": "single",
        "gsd_bucket": native_bucket,
        "split": "train",
        "instruction": "Describe the SAR backscatter characteristics of this scene.",
        "response": f"SAR backscatter over a scene classified optically as: {opt_response}.",
        "bbox": None,
        "modality": "sar",
    }

    concat_img = concat_horizontal(opt_rgb, sar_rgb)
    concat_png = img_dir / f"{sample_id}_concat.png"
    concat_img.save(str(concat_png), format="PNG", compress_level=0)

    fusion_sample = {
        "id": f"{sample_id}_fusion",
        "dataset": "bigen",
        "task": "fusion_vqa",
        "image_path": [str(opt_png), str(sar_png)],
        "pair_type": "cross-modal",
        "gsd_bucket": native_bucket,
        "split": "train",
        "instruction": (
            "Using both the optical and SAR images, describe this scene's "
            "land cover and surface conditions."
        ),
        "response": (
            f"Optical imagery shows: {opt_response}. "
            "SAR backscatter is consistent with these cover types."
        ),
        "bbox": None,
        "modality": "optical+sar",
    }

    return [sar_sample, fusion_sample]


def run_tier0_bigen(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int = 60000,
    seed: int = 42,
) -> dict[str, int]:
    """Run BigEarthNet preprocessing with manifest-based resumability.

    Returns stats dict: {processed, skipped, failed}. "processed" counts
    patches (each patch may emit 1-3 JSONL rows: optical, +SAR, +fusion).
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "bigen.jsonl"
    manifest = Manifest(manifest_path)

    native_bucket = assign_gsd_bucket("bigen")

    metadata_path = input_dir / "metadata.parquet"
    if not metadata_path.exists():
        print(f"  [WARN] metadata.parquet not found in {input_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    import pandas as pd
    metadata = pd.read_parquet(metadata_path)
    total = min(len(metadata), max_samples)
    print(f"Loaded metadata: {len(metadata)} patches, processing {total}")

    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for idx in range(total):
        row = metadata.iloc[idx].to_dict()
        sample_id = f"bigen_{idx:08d}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        try:
            s2, s1 = load_bigen_arrays(input_dir, row)
        except (FileNotFoundError, ValueError) as e:
            print(f"  [FAIL] {sample_id}: {e}", file=sys.stderr)
            stats["failed"] += 1
            continue

        label_vec = _labels_to_multihot(row.get("labels"))
        opt_sample = parse_bigen_sample({"image": s2, "label": label_vec}, idx)
        if opt_sample is None:
            stats["failed"] += 1
            continue

        opt_rgb = s2_to_rgb_uint8(s2)
        opt_png = img_dir / f"{sample_id}_optical.png"
        write_png(opt_rgb, opt_png)
        opt_sample["image_path"] = [str(opt_png)]
        opt_sample["gsd_bucket"] = native_bucket

        emitted = [opt_sample]

        if s1 is not None and s1.ndim == 3 and s1.shape[0] >= 2:
            emitted.extend(_build_sar_and_fusion_samples(
                sample_id, opt_png, opt_rgb, opt_sample["response"],
                s1, img_dir, native_bucket,
            ))

        errors = []
        for s in emitted:
            ok, errs = validate_sample(s)
            if not ok:
                errors.append((s["id"], errs))
        if errors:
            for sid, errs in errors:
                print(f"  [VALID] {sid}: {errs}", file=sys.stderr)
            stats["failed"] += 1
            continue

        for s in emitted:
            append_jsonl(s, jsonl_path)

        manifest.mark_processed(
            sample_id=sample_id,
            dataset="bigen",
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (idx + 1) % 5000 == 0:
            elapsed = time.time() - start
            rate = (idx + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {idx + 1}/{total} ({rate:.1f} patches/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess BigEarthNet")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=60000)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_tier0_bigen(
        args.input_dir,
        args.output_dir,
        max_samples=args.max_samples,
        seed=args.seed,
    )


In [ ]:
%%writefile preprocess/tier1_cdvqa.py
"""Tier 1 — CDVQA preprocessing (Change Detection VQA).

CDVQA provides bi-temporal aerial imagery with question-answer pairs about
changes between dates. Uses SECOND imagery (0.5m-2m resolution).

Spec treatment:
    - R1: RGB channels
    - R2: percentile normalization (deferred)
    - R4: NOT dual-resolution — the Tier 1 table doesn't tag CDVQA with R4
      (unlike vrsbench/rsvqa_hr/levir_cd/sn6_opt), and SECOND imagery spans
      ~0.5-2m per scene rather than one fixed native GSD anyway.
    - pair_type: bitemporal
    - dataset enum: "cdvqa"
    - gsd_bucket: VHR-native (categorical — see common/gsd.py)
    - task: change_vqa

CDVQA structure:
    <input_dir>/
        train.json / val.json
            Each entry: {"question": ..., "answer": ...,
                         "before": "path/to/before.png",
                         "after": "path/to/after.png"}
        images/
            <before/after images>

IMPORTANT: Load ONLY train/val split files. Test splits are EXCLUDED.

Usage:
    python -m preprocess.tier1_cdvqa --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path
from typing import Any

from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest, append_jsonl
from preprocess.validator import validate_sample

_DATASET = "cdvqa"
_VALID_SPLITS = {"train", "val"}


def load_split(input_dir: Path, split_name: str) -> list[dict]:
    """Load a CDVQA split file (train.json or val.json).

    Asserts the split name is valid (train/val only).
    """
    if split_name not in _VALID_SPLITS:
        raise ValueError(f"Invalid split '{split_name}'. Must be one of {_VALID_SPLITS}")

    path = input_dir / f"{split_name}.json"
    if not path.exists():
        return []

    with open(path) as f:
        data = json.load(f)

    if isinstance(data, list):
        return data
    elif isinstance(data, dict):
        return data.get("questions", data.get("data", []))
    return []


def parse_cdvqa_sample(
    entry: dict[str, Any],
    images_dir: Path | None,
    sample_id: str,
) -> dict[str, Any] | None:
    """Parse a single CDVQA annotation into our JSONL format."""
    question = entry.get("question", "")
    answer = entry.get("answer", "")
    before_path = entry.get("before", "")
    after_path = entry.get("after", "")

    if not question or not answer:
        return None
    if not before_path or not after_path:
        return None

    # Resolve image paths
    image_paths = []
    if images_dir:
        for rel_path in [before_path, after_path]:
            full_path = images_dir / rel_path
            if full_path.exists():
                image_paths.append(str(full_path))
            else:
                image_paths.append("")
    else:
        image_paths = ["", ""]

    return {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "change_vqa",
        "image_path": image_paths,
        "pair_type": "bitemporal",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": question,
        "response": str(answer),
        "bbox": None,
        "modality": "optical",
    }


def run_tier1_cdvqa(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
) -> dict[str, int]:
    """Run CDVQA preprocessing with manifest-based resumability.

    Loads ONLY train/val split files. Test splits are rejected.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "cdvqa.jsonl"
    manifest = Manifest(manifest_path)

    native_bucket = assign_gsd_bucket("cdvqa")

    # Load train + val only
    annotations = []
    for split_name in ["train", "val"]:
        split_data = load_split(input_dir, split_name)
        for entry in split_data:
            entry["_split"] = split_name
        annotations.extend(split_data)

    if not annotations:
        print(f"  [WARN] No annotations found in {input_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    images_dir = None
    for name in ["images", "Images", "SECOND"]:
        candidate = input_dir / name
        if candidate.exists():
            images_dir = candidate
            break

    total = len(annotations)
    if max_samples:
        total = min(total, max_samples)
    print(f"Loaded {len(annotations)} annotations (train+val), processing {total}")

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, entry in enumerate(annotations[:total]):
        sample_id = f"cdvqa_{i:07d}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        sample = parse_cdvqa_sample(entry, images_dir, sample_id)
        if sample is None:
            stats["failed"] += 1
            continue

        sample["gsd_bucket"] = native_bucket
        # All CDVQA samples enter as "train"; the val_internal carve-out
        # is handled downstream by split_internal_val.py.  The raw "_split"
        # value "val" is not in the schema enum ["train", "val_internal"].
        sample["split"] = "train"

        ok, errs = validate_sample(sample)
        if not ok:
            print(f"  [VALID] {sample_id}: {errs}", file=sys.stderr)
            stats["failed"] += 1
            continue

        append_jsonl(sample, jsonl_path)
        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split=sample["split"],
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 1000 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess CDVQA")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    args = parser.parse_args()

    run_tier1_cdvqa(args.input_dir, args.output_dir, max_samples=args.max_samples)


In [ ]:
%%writefile preprocess/tier1_levir_cd.py
"""Tier 1 — LEVIR-CD preprocessing (LEvir Change Detection).

LEVIR-CD provides 6,374 bi-temporal image pairs (1024×1024, 0.5m GSD)
with building change masks. Used for bi-temporal building-change grounding.

Spec treatment:
    - R1: RGB channels (already RGB)
    - R2: percentile normalization (deferred)
    - R4: dual-resolution branching
    - Masks → bounding boxes (R6 Qwen format)
    - pair_type: bitemporal
    - dataset enum: "levir_cd"
    - gsd_bucket: [GSD:0.5m] (native), CARTOSAT-proxy (proxy)
    - task: change_grounding

LEVIR-CD structure:
    <input_dir>/
        train/
            A/  <image>_1.png (before)
            B/  <image>_2.png (after)
            label/  <image>.png (binary change mask)
        val/  (same structure)
        test/ (same structure — EXCLUDED from training)

Usage:
    python -m preprocess.tier1_levir_cd --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

from preprocess.common.bbox import pixel_to_normalized
from preprocess.common.gsd import assign_gsd_bucket, create_proxy_sample
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.common.sample import quantile_edges, stratified_sample
from preprocess.validator import validate_sample

_DATASET = "levir_cd"
_NATIVE_GSD = 0.5


def find_levir_pairs(split_dir: Path) -> list[dict[str, Any]]:
    """Discover all LEVIR-CD image pairs in a split directory."""
    a_dir = split_dir / "A"
    b_dir = split_dir / "B"
    label_dir = split_dir / "label"

    if not a_dir.exists() or not b_dir.exists():
        return []

    pairs = []
    for before_path in sorted(a_dir.glob("*.png")):
        # Extract base name (remove _1 suffix)
        base_name = before_path.stem
        if base_name.endswith("_1"):
            base_name = base_name[:-2]

        after_path = b_dir / f"{base_name}_2.png"
        mask_path = label_dir / f"{base_name}.png"

        if after_path.exists():
            pairs.append({
                "base_name": base_name,
                "before": before_path,
                "after": after_path,
                "mask": mask_path if mask_path.exists() else None,
            })

    return pairs


def change_fraction(mask_path: Path | None) -> float:
    """Fraction of changed pixels in a LEVIR-CD label mask — the
    stratification key for "stratified by change magnitude decile"."""
    if mask_path is None or not mask_path.exists():
        return 0.0
    from PIL import Image
    mask = np.array(Image.open(str(mask_path)).convert("L"))
    return float((mask > 127).mean())


def mask_to_bbox(mask_path: Path, image_shape: tuple[int, int]) -> list[list[float]] | None:
    """Convert binary change mask to normalized bounding boxes."""
    try:
        from PIL import Image
        mask = np.array(Image.open(str(mask_path)).convert("L"))
    except Exception:
        return None

    # Binarize
    mask = (mask > 127).astype(np.uint8)
    if mask.sum() == 0:
        return None

    # Find bounding box of change region
    ys, xs = np.where(mask > 0)
    if len(ys) < 10:
        return None

    h, w = image_shape
    bbox_pixel = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]
    bbox_norm = pixel_to_normalized(tuple(bbox_pixel), w, h)
    return bbox_norm


def process_pair(
    pair: dict[str, Any],
    output_dir: Path,
) -> dict[str, Any] | None:
    """Process a single LEVIR-CD pair into a JSONL sample."""
    base_name = pair["base_name"]

    # Copy images to output
    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    before_out = img_dir / f"levir_{base_name}_before.png"
    after_out = img_dir / f"levir_{base_name}_after.png"

    try:
        from PIL import Image
        img = Image.open(str(pair["before"]))
        if img.mode != "RGB":
            img = img.convert("RGB")
        img.save(str(before_out), format="PNG", compress_level=0)

        img = Image.open(str(pair["after"]))
        if img.mode != "RGB":
            img = img.convert("RGB")
        img.save(str(after_out), format="PNG", compress_level=0)
    except Exception as e:
        print(f"  [FAIL] {base_name}: {e}", file=sys.stderr)
        return None

    # Extract bbox from mask
    bbox = None
    if pair["mask"] is not None:
        try:
            from PIL import Image
            img = Image.open(str(pair["before"]))
            bbox = mask_to_bbox(pair["mask"], img.size[::-1])  # (H, W)
        except Exception:
            pass

    sample_id = f"levir_{base_name}"

    return {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "change_grounding" if bbox is not None else "change_vqa",
        "image_path": [str(before_out), str(after_out)],
        "pair_type": "bitemporal",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": "What building changes occurred between these two dates?",
        "response": f"Building changes detected in {base_name}.",
        "bbox": bbox,
        "modality": "optical",
    }


def run_tier1_levir_cd(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
    sample_fraction: float = 0.10,
    seed: int = 42,
) -> dict[str, int]:
    """Run LEVIR-CD preprocessing with manifest-based resumability.

    Samples 10% stratified by change magnitude (as per spec).
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "levir_cd.jsonl"
    manifest = Manifest(manifest_path)

    native_bucket = assign_gsd_bucket("levir_cd")

    # Load train split only (per spec: 10% stratified)
    train_dir = input_dir / "train"
    if not train_dir.exists():
        print(f"  [WARN] Train directory not found: {train_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    pairs = find_levir_pairs(train_dir)
    print(f"Found {len(pairs)} LEVIR-CD pairs in train split")

    # Selection (spec): "stratified by change magnitude, not top decile" —
    # bucket pairs into change-fraction deciles and sample proportionally
    # from every bucket, so near-zero-change pairs stay represented.
    if sample_fraction < 1.0 and pairs:
        fractions = [change_fraction(p["mask"]) for p in pairs]
        edges = quantile_edges(fractions, n_buckets=10)

        from preprocess.common.sample import quantile_bucket
        deciles = [quantile_bucket(f, edges) for f in fractions]
        keyed_pairs = list(zip(pairs, deciles))

        pairs = [
            p for p, _ in stratified_sample(
                keyed_pairs, key_fn=lambda item: item[1],
                fraction=sample_fraction, seed=seed,
            )
        ]

    total = len(pairs)
    if max_samples:
        total = min(total, max_samples)

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, pair in enumerate(pairs[:total]):
        sample_id = f"levir_{pair['base_name']}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        sample = process_pair(pair, output_dir)
        if sample is None:
            stats["failed"] += 1
            continue

        sample["gsd_bucket"] = native_bucket

        # R4: dual-resolution branching (native + CARTOSAT-proxy).
        rows = [sample, create_proxy_sample(sample)]

        ok_all = True
        for row in rows:
            ok, errs = validate_sample(row)
            if not ok:
                print(f"  [VALID] {row['id']}: {errs}", file=sys.stderr)
                ok_all = False
                break
        if not ok_all:
            stats["failed"] += 1
            continue

        for row in rows:
            append_jsonl(row, jsonl_path)

        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 1000 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess LEVIR-CD")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    parser.add_argument("--sample-fraction", type=float, default=0.10)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_tier1_levir_cd(
        args.input_dir,
        args.output_dir,
        max_samples=args.max_samples,
        sample_fraction=args.sample_fraction,
        seed=args.seed,
    )


In [ ]:
%%writefile preprocess/tier1_oscd.py
"""Tier 1 — OSCD (Onera Satellite Change Detection) preprocessing.

OSCD provides 24 co-registered multispectral image pairs (13-band Sentinel-2)
with binary change masks.  Used as a bi-temporal regularizer.

Spec treatment:
    - R1: 13-band → RGB (B4/B3/B2)
    - R2: percentile normalization (deferred to stats.py global pass)
    - R4: NOT applicable (OSCD not in DUAL_RESOLUTION_DATASETS)
    - Masks → bounding boxes (R6 Qwen format)
    - pair_type: bitemporal
    - dataset enum: "oscd"
    - Split: all pairs usable (no prescribed benchmark)

Real OSCD directory layout (TorchGeo / official):
    <input_dir>/
        images/
            <location_name>/
                imgs_1/
                    <mission>_<date>_B01.tif  (individual band files)
                    <mission>_<date>_B02.tif
                    ...
                    <mission>_<date>_B12.tif
                    <mission>_<date>_B8A.tif
                imgs_2/
                    <mission>_<date>_B01.tif
                    ...
        labels/
            <location_name>/
                cm/
                    <location_name>-cm.tif  (binary change mask)

Usage:
    python -m preprocess.tier1_oscd --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Any

import numpy as np
import tifffile
from PIL import Image

from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.validator import validate_sample

# Band files we need for RGB extraction
_BAND_PRIORITY = ["B04", "B03", "B02"]  # R, G, B
# Full Sentinel-2 band order for stacking
_ALL_BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]


def _extract_band_from_filename(filename: str) -> str | None:
    """Extract band name (e.g., 'B04') from Sentinel-2 filename."""
    # Match patterns like _B04.tif, _B8A.tif, _B12.tif
    m = re.search(r"_(B\d{2}[A-Z]?)\.tif$", filename)
    if m:
        return m.group(1)
    return None


def find_oscd_pairs(input_dir: Path) -> list[dict[str, Any]]:
    """Discover all OSCD image pairs and their masks.

    Returns list of dicts with keys:
        location, before_bands_dir, after_bands_dir, mask_path
    """
    images_dir = input_dir / "images"
    labels_dir = input_dir / "labels"

    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")

    pairs = []
    for location_dir in sorted(images_dir.iterdir()):
        if not location_dir.is_dir():
            continue

        imgs_1 = location_dir / "imgs_1"
        imgs_2 = location_dir / "imgs_2"

        if not imgs_1.exists() or not imgs_2.exists():
            continue

        # Find mask
        mask_path = labels_dir / location_dir.name / "cm" / f"{location_dir.name}-cm.tif"
        if not mask_path.exists():
            mask_path = None

        pairs.append({
            "location": location_dir.name,
            "before_bands_dir": imgs_1,
            "after_bands_dir": imgs_2,
            "mask_path": mask_path,
        })

    return pairs


def load_bands_as_rgb(bands_dir: Path) -> np.ndarray:
    """Load individual band TIFs from a directory and extract RGB.

    Reads B04, B03, B02 (Sentinel-2 RGB bands) and stacks them.

    Returns (H, W, 3) uint8 array.
    """
    bands = {}
    for tif_path in bands_dir.glob("*.tif"):
        band_name = _extract_band_from_filename(tif_path.name)
        if band_name in _BAND_PRIORITY or band_name in _ALL_BANDS:
            arr = tifffile.imread(str(tif_path))
            if arr.ndim == 3:
                arr = arr[0]  # Some TIFs have extra dimension
            bands[band_name] = arr

    if not bands:
        raise ValueError(f"No valid band files found in {bands_dir}")

    # Extract RGB
    rgb_bands = []
    for band_name in _BAND_PRIORITY:
        if band_name not in bands:
            raise ValueError(f"Missing required band {band_name} in {bands_dir}")
        rgb_bands.append(bands[band_name])

    rgb = np.stack(rgb_bands, axis=-1)

    # Normalize to uint8
    if rgb.dtype == np.uint16:
        # Sentinel-2 values typically 0–10000
        rgb = np.clip(rgb, 0, 10000).astype(np.float32) / 10000.0 * 255.0
        rgb = rgb.astype(np.uint8)
    elif rgb.max() > 255:
        p98 = np.percentile(rgb, 98)
        rgb = np.clip(rgb, 0, p98).astype(np.float32) / p98 * 255.0
        rgb = rgb.astype(np.uint8)

    return rgb


def load_mask(mask_path: Path | None, target_shape: tuple[int, int] | None = None) -> np.ndarray | None:
    """Load a binary change mask.

    Returns (H, W) uint8 array with values 0 (no change) and 1 (change),
    or None if no mask is available.
    """
    if mask_path is None or not mask_path.exists():
        return None

    mask = tifffile.imread(str(mask_path))

    # Ensure 2D
    if mask.ndim == 3:
        mask = mask[:, :, 0]

    # Binarize (some masks use 255 for change)
    mask = (mask > 0).astype(np.uint8)

    # Resize if needed
    # Issue 10 fix: prefer scipy.ndimage.zoom (nearest-neighbour) over PIL for
    # consistency with the rest of the scipy-based mask processing path.
    if target_shape is not None and mask.shape != target_shape:
        try:
            from scipy import ndimage as _ndimage
            zoom_y = target_shape[0] / mask.shape[0]
            zoom_x = target_shape[1] / mask.shape[1]
            mask = _ndimage.zoom(mask, (zoom_y, zoom_x), order=0).astype(np.uint8)
        except ImportError:
            from PIL import Image as PILImage
            mask_img = PILImage.fromarray(mask, mode="L")
            mask_img = mask_img.resize(
                (target_shape[1], target_shape[0]),
                resample=PILImage.Resampling.NEAREST,
            )
            mask = np.array(mask_img)

    return mask


def mask_to_bboxes(mask: np.ndarray) -> list[list[int]]:
    """Convert binary change mask to a list of bounding boxes.

    Uses connected-component labeling to find change regions.
    Returns list of [x1, y1, x2, y2] pixel coordinates.
    """
    if mask is None or mask.sum() == 0:
        return []

    try:
        from scipy import ndimage
        labeled, num_features = ndimage.label(mask)
    except ImportError:
        ys, xs = np.where(mask > 0)
        if len(ys) == 0:
            return []
        return [[int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]]

    bboxes = []
    for i in range(1, num_features + 1):
        ys, xs = np.where(labeled == i)
        # Issue 8 fix: raise threshold from 10 → 50 pixels (spec's ≥50px² suggestion)
        if len(ys) < 50:  # Skip tiny noise regions
            continue
        bboxes.append([int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())])

    return bboxes


def make_sample_id(location: str) -> str:
    """Generate a unique, lowercase, underscore-separated sample id."""
    return f"oscd_{location}".lower().replace("-", "_")[:64]


def process_pair(
    pair: dict[str, Any],
    output_dir: Path,
) -> dict[str, Any] | None:
    """Process a single OSCD pair into JSONL sample + PNGs.

    Returns the sample dict, or None if processing failed.
    """
    location = pair["location"]

    # Load images
    try:
        before_rgb = load_bands_as_rgb(pair["before_bands_dir"])
        after_rgb = load_bands_as_rgb(pair["after_bands_dir"])
    except Exception as e:
        print(f"  ERROR loading images for {location}: {e}")
        return None

    # Issue 3 fix (Option A): OSCD change masks are spatially scattered across
    # the full image (urban growth along road networks), so even correct
    # per-component boxes for the largest component tend to span nearly the
    # entire image (~99.8% area).  Training on these teaches the model that
    # "grounding" means "the whole image" — actively harming spatial precision.
    # Demote all OSCD samples to change_vqa (bbox=None) unconditionally.
    # Mask and bboxes are still computed so mask_to_bboxes remains testable,
    # but the result is not propagated into the sample.
    _ = load_mask(pair.get("mask_path"), target_shape=before_rgb.shape[:2])

    # Write PNGs
    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    before_png = img_dir / f"{location}_before.png"
    after_png = img_dir / f"{location}_after.png"

    write_png(before_rgb, before_png)
    write_png(after_rgb, after_png)

    # Build JSONL sample
    sample_id = make_sample_id(location)
    sample = {
        "id": sample_id,
        "dataset": "oscd",
        "task": "change_vqa",  # Issue 3: always change_vqa, never change_grounding
        "image_path": [str(before_png), str(after_png)],
        "pair_type": "bitemporal",
        "gsd_bucket": assign_gsd_bucket("oscd"),
        "split": "train",
        "instruction": "What changed between the two dates?",
        "response": f"Change detected in {location}.",
        "bbox": None,  # Issue 3: always null for OSCD
        "modality": "optical",
    }

    # Validate
    ok, errs = validate_sample(sample)
    if not ok:
        print(f"  VALIDATION FAILED for {location}: {errs}")
        return None

    return sample


def run_tier1_oscd(
    input_dir: str | Path,
    output_dir: str | Path,
    manifest_path: str | Path | None = None,
) -> dict[str, Any]:
    """Run OSCD preprocessing with manifest-based resumability.

    Returns summary dict with counts.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if manifest_path is None:
        manifest_path = output_dir / "manifest.parquet"
    manifest = Manifest(manifest_path)

    jsonl_path = output_dir / "oscd.jsonl"

    # Discover pairs
    pairs = find_oscd_pairs(input_dir)
    print(f"Found {len(pairs)} OSCD pairs")

    stats = {"total": len(pairs), "processed": 0, "skipped": 0, "failed": 0}

    for pair in pairs:
        sample_id = make_sample_id(pair["location"])

        # Manifest check — skip if already processed
        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        # Process
        sample = process_pair(pair, output_dir)
        if sample is None:
            stats["failed"] += 1
            continue

        # Write JSONL
        append_jsonl(sample, jsonl_path)

        # Mark processed
        manifest.mark_processed(
            sample_id=sample_id,
            dataset="oscd",
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if stats["processed"] % 5 == 0:
            print(f"  Processed {stats['processed']}/{stats['total'] - stats['skipped']}")

    print(f"Done: {stats['processed']} processed, {stats['skipped']} skipped, {stats['failed']} failed")
    return stats


def main():
    parser = argparse.ArgumentParser(description="Preprocess OSCD dataset")
    parser.add_argument("--input-dir", required=True, help="OSCD dataset root")
    parser.add_argument("--output-dir", required=True, help="Output directory")
    parser.add_argument("--manifest", default=None, help="Manifest path")
    args = parser.parse_args()

    run_tier1_oscd(args.input_dir, args.output_dir, args.manifest)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile preprocess/tier1_rsvqa_hr.py
"""Tier 1 — RSVQA-HR preprocessing.

RSVQA-HR provides high-resolution aerial imagery (USGS, 0.15m) with
question-answer pairs for visual question answering.

Spec treatment:
    - R1: RGB channels (already RGB in source)
    - R2: percentile normalization (deferred to stats.py)
    - R4: dual-resolution branching (native + proxy)
    - gsd_bucket: [GSD:0.15m] (native), CARTOSAT-proxy (proxy)
    - dataset enum: "rsvqa_hr"
    - pair_type: single
    - task: vqa

RSVQA-HR structure:
    <input_dir>/
        images/
            <image_id>.png (or .jpg)
        questions.json / train.json / val.json / test.json
            Each entry: {"question_id": ..., "question": ..., "answer": ...,
                         "image_id": ..., "type": ...}

Usage:
    python -m preprocess.tier1_rsvqa_hr --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path
from typing import Any

from PIL import Image

from preprocess.common.gsd import assign_gsd_bucket, create_proxy_sample
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.validator import validate_sample

_NATIVE_GSD = 0.15  # USGS 15cm
_DATASET = "rsvqa_hr"


def _ensure_png(src_path: Path, png_dir: Path) -> str:
    """R7: all outputs are 8-bit 3-channel PNG. RSVQA-HR images can ship
    as .jpg/.tif — convert those; a source .png is reused as-is."""
    if src_path.suffix.lower() == ".png":
        return str(src_path)

    png_path = png_dir / f"{src_path.stem}.png"
    if not png_path.exists():
        img = Image.open(str(src_path))
        if img.mode != "RGB":
            img = img.convert("RGB")
        png_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(str(png_path), format="PNG", compress_level=0)
    return str(png_path)


def load_annotations(input_dir: Path) -> list[dict]:
    """Load RSVQA-HR annotations from JSON files.

    Tries train.json first, falls back to questions.json.
    """
    for name in ["train.json", "questions.json"]:
        path = input_dir / name
        if path.exists():
            with open(path) as f:
                data = json.load(f)
            if isinstance(data, list):
                return data
            elif isinstance(data, dict):
                # Some formats wrap in {"questions": [...]}
                return data.get("questions", data.get("data", []))
    return []


def parse_rsvqa_sample(
    entry: dict[str, Any],
    images_dir: Path | None,
    sample_id: str,
    *,
    png_dir: Path | None = None,
) -> dict[str, Any] | None:
    """Parse a single RSVQA-HR annotation into our JSONL format."""
    question = entry.get("question", "")
    answer = entry.get("answer", "")
    image_id = entry.get("image_id", "")

    if not question or not answer:
        return None

    # Handle image path — R7 requires PNG; convert non-PNG sources.
    image_path_str = ""
    if images_dir and image_id:
        for ext in [".png", ".jpg", ".jpeg", ".tif"]:
            img_path = images_dir / f"{image_id}{ext}"
            if img_path.exists():
                image_path_str = _ensure_png(img_path, png_dir or images_dir)
                break

    return {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "vqa",
        "image_path": [image_path_str] if image_path_str else [],
        "pair_type": "single",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": question,
        "response": str(answer),
        "bbox": None,
        "modality": "optical",
    }


def run_tier1_rsvqa_hr(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
) -> dict[str, int]:
    """Run RSVQA-HR preprocessing with manifest-based resumability."""
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "rsvqa_hr.jsonl"
    manifest = Manifest(manifest_path)

    native_bucket = assign_gsd_bucket("rsvqa_hr")

    annotations = load_annotations(input_dir)
    if not annotations:
        print(f"  [WARN] No annotations found in {input_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    images_dir = None
    for name in ["images", "Images", "img"]:
        candidate = input_dir / name
        if candidate.exists():
            images_dir = candidate
            break

    png_dir = output_dir / "images" / "converted"

    total = len(annotations)
    if max_samples:
        total = min(total, max_samples)
    print(f"Loaded {len(annotations)} annotations, processing {total}")

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, entry in enumerate(annotations[:total]):
        qid = entry.get("question_id", entry.get("id", i))
        sample_id = f"rsvqa_{qid}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        sample = parse_rsvqa_sample(entry, images_dir, sample_id, png_dir=png_dir)
        if sample is None:
            stats["failed"] += 1
            continue

        sample["gsd_bucket"] = native_bucket

        # R4: dual-resolution branching — emit native + CARTOSAT-proxy rows.
        rows = [sample]
        if sample["image_path"]:
            rows.append(create_proxy_sample(sample))

        ok_all = True
        for row in rows:
            ok, errs = validate_sample(row)
            if not ok:
                print(f"  [VALID] {row['id']}: {errs}", file=sys.stderr)
                ok_all = False
                break
        if not ok_all:
            stats["failed"] += 1
            continue

        for row in rows:
            append_jsonl(row, jsonl_path)

        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess RSVQA-HR")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    args = parser.parse_args()

    run_tier1_rsvqa_hr(args.input_dir, args.output_dir, max_samples=args.max_samples)


In [ ]:
%%writefile preprocess/tier1_sn6_opt.py
"""Tier 1 — SpaceNet 6 Optical preprocessing.

SpaceNet 6 provides WorldView-2 0.5m pan-sharpened RGB imagery with
building footprint polygons. Used for VHR building grounding + fusion.

Spec treatment:
    - R1: RGB channels (already RGB from pan-sharpened)
    - R2: percentile normalization (deferred)
    - R4: dual-resolution branching
    - Polygons → bounding boxes (R6 Qwen format)
    - pair_type: single (optical only, SAR handled by tier2_sn6_sar)
    - dataset enum: "sn6_opt"
    - gsd_bucket: [GSD:0.5m] (native), CARTOSAT-proxy (proxy)
    - task: grounding

SpaceNet 6 structure:
    <input_dir>/
        train/
            images/
                <tile_id>.tif (WorldView-2 pan-sharpened RGB)
            labels/
                <tile_id>.geojson (building footprints)

Usage:
    python -m preprocess.tier1_sn6_opt --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

from preprocess.common.bbox import pixel_to_normalized
from preprocess.common.gsd import assign_gsd_bucket, create_proxy_sample
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.common.sample import quantile_bucket, quantile_edges, stratified_sample
from preprocess.validator import validate_sample

_DATASET = "sn6_opt"


def load_geojson_labels(geojson_path: Path) -> list[list[list[float]]]:
    """Load building footprint polygons from GeoJSON.

    Returns list of polygons, each being a list of [x, y] coordinates.
    """
    if not geojson_path.exists():
        return []

    try:
        with open(geojson_path) as f:
            data = json.load(f)
    except Exception:
        return []

    polygons = []
    for feature in data.get("features", []):
        geom = feature.get("geometry", {})
        if geom.get("type") == "Polygon":
            coords = geom.get("coordinates", [[]])
            if coords:
                polygons.append(coords[0])
        elif geom.get("type") == "MultiPolygon":
            for poly in geom.get("coordinates", []):
                if poly:
                    polygons.append(poly[0])

    return polygons


def polygons_to_bboxes(
    polygons: list[list[list[float]]],
    img_width: int,
    img_height: int,
) -> list[list[float]]:
    """Convert polygon coordinates to normalized bounding boxes."""
    bboxes = []
    for polygon in polygons:
        if len(polygon) < 3:
            continue

        xs = [p[0] for p in polygon]
        ys = [p[1] for p in polygon]

        # Convert pixel coords to normalized [0, 1000]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)

        bbox = pixel_to_normalized(
            (int(x_min), int(y_min), int(x_max), int(y_max)),
            img_width,
            img_height,
        )
        if bbox:
            bboxes.append(bbox[0])

    return bboxes


def process_tile(
    tile_path: Path,
    geojson_path: Path,
    output_dir: Path,
    tile_id: str,
) -> dict[str, Any] | None:
    """Process a single SpaceNet 6 tile."""
    try:
        import tifffile
        img = tifffile.imread(str(tile_path))
    except Exception as e:
        print(f"  [FAIL] {tile_id}: {e}", file=sys.stderr)
        return None

    # Ensure RGB
    if img.ndim == 3 and img.shape[2] >= 3:
        img = img[:, :, :3]
    elif img.ndim == 2:
        # Grayscale → RGB
        img = np.stack([img, img, img], axis=-1)

    # Normalize to uint8
    if img.dtype == np.uint16:
        p98 = np.percentile(img, 98)
        if p98 > 0:
            img = np.clip(img, 0, p98).astype(np.float32) / p98 * 255.0
        img = img.astype(np.uint8)

    h, w = img.shape[:2]

    # Save PNG
    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    out_path = img_dir / f"sn6_{tile_id}.png"
    write_png(img, out_path)

    # Load building footprints — keep every box, not just the first, so
    # the response text (which reports the real count) and the bbox
    # field agree with each other.
    polygons = load_geojson_labels(geojson_path)
    bboxes = polygons_to_bboxes(polygons, w, h)
    n_buildings = len(bboxes)

    sample_id = f"sn6_{tile_id}"

    return {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "grounding" if bboxes else "vqa",
        "image_path": [str(out_path)],
        "pair_type": "single",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": "How many buildings are in this image?" if not bboxes
                       else "Locate the buildings in this image.",
        "response": "0" if not bboxes
                    else f"{n_buildings} building{'s' if n_buildings != 1 else ''} detected.",
        "bbox": bboxes if bboxes else None,
        "modality": "optical",
    }


def run_tier1_sn6_opt(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
    sample_fraction: float = 0.05,
    seed: int = 42,
) -> dict[str, int]:
    """Run SpaceNet 6 Optical preprocessing."""
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "sn6_opt.jsonl"
    manifest = Manifest(manifest_path)

    native_bucket = assign_gsd_bucket("sn6_opt")

    train_dir = input_dir / "train"
    images_dir = train_dir / "images"
    labels_dir = train_dir / "labels"

    if not images_dir.exists():
        print(f"  [WARN] Images directory not found: {images_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    # Find all tiles
    tile_paths = sorted(images_dir.glob("*.tif"))
    print(f"Found {len(tile_paths)} SpaceNet 6 tiles")

    # Selection (spec): "stratified by building-density quartile" — bucket
    # tiles by building-footprint count and sample proportionally from
    # every quartile, so sparse and dense tiles both stay represented.
    if sample_fraction < 1.0 and tile_paths:
        densities = [
            len(load_geojson_labels(labels_dir / f"{p.stem}.geojson"))
            for p in tile_paths
        ]
        edges = quantile_edges(densities, n_buckets=4)
        quartiles = [quantile_bucket(d, edges) for d in densities]
        keyed = list(zip(tile_paths, quartiles))
        tile_paths = [
            p for p, _ in stratified_sample(
                keyed, key_fn=lambda item: item[1],
                fraction=sample_fraction, seed=seed,
            )
        ]

    # Record the selected tile ids so tier2_sn6_sar can select the SAME
    # tiles ("paired 1:1 with optical selection" — Tier 2 table).
    selected_path = output_dir / "selected_tile_ids.json"
    selected_path.write_text(json.dumps(sorted(p.stem for p in tile_paths)))

    total = len(tile_paths)
    if max_samples:
        total = min(total, max_samples)

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, tile_path in enumerate(tile_paths[:total]):
        tile_id = tile_path.stem
        sample_id = f"sn6_{tile_id}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        geojson_path = labels_dir / f"{tile_id}.geojson"
        sample = process_tile(tile_path, geojson_path, output_dir, tile_id)
        if sample is None:
            stats["failed"] += 1
            continue

        sample["gsd_bucket"] = native_bucket

        # R4: dual-resolution branching (native + CARTOSAT-proxy).
        rows = [sample, create_proxy_sample(sample)]

        ok_all = True
        for row in rows:
            ok, errs = validate_sample(row)
            if not ok:
                print(f"  [VALID] {row['id']}: {errs}", file=sys.stderr)
                ok_all = False
                break
        if not ok_all:
            stats["failed"] += 1
            continue

        for row in rows:
            append_jsonl(row, jsonl_path)

        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 500 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess SpaceNet 6 Optical")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    parser.add_argument("--sample-fraction", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_tier1_sn6_opt(
        args.input_dir,
        args.output_dir,
        max_samples=args.max_samples,
        sample_fraction=args.sample_fraction,
        seed=args.seed,
    )


In [ ]:
%%writefile preprocess/tier1_vrsbench.py
"""VRSBench preprocessing — caption + grounding + VQA with R4 dual-resolution.

Parses VRSBench_train.json (LLaVA-format consolidated file, 142k samples).

Task prefixes in conversation:
    [caption] — image captioning (20,264 samples)
    [refer]   — visual grounding / referring (36,313 samples)
    [vqa]     — visual question answering (85,813 samples)

Referring answer format: {<x_left><y_top><x_right><y_bottom>}
    Coordinates are in GeoChat [0,100] grid (resized to 100×100).
    Converted to Qwen [0,1000] by multiplying by 10.

Each source sample produces TWO JSONL rows (R4 dual-resolution):
    Native branch  — gsd_bucket="VHR-native"
    Proxy branch   — gsd_bucket="CARTOSAT-proxy[GSD:2.0m]"

PNGs are saved when images are available; JSONL is written regardless.
"""
from __future__ import annotations

import json
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

from preprocess.common.bbox import unit01_to_qwen, validate_bbox_ordering
from preprocess.common.gsd import assign_gsd_bucket, create_proxy_sample
from preprocess.common.io import Manifest, append_jsonl, write_png

_DATASET = "vrsbench"

# Regex to extract task prefix: [caption], [refer], [vqa]
_TASK_RE = re.compile(r"\[(caption|refer|vqa)\]")

# Regex to parse referring answer: {<45><45><59><59>}
_REFERRING_RE = re.compile(r"\{<(\d+)><(\d+)><(\d+)><(\d+)>\}")


def parse_conversation(
    entry: dict[str, Any],
) -> tuple[str, str, str | None]:
    """Parse a VRSBench_train.json entry.

    Returns (task_type, instruction, response_or_coords).
    For referring, response_or_coords is the raw "{<x><y><x><y>}" string.
    """
    convs = entry["conversations"]
    if len(convs) < 2:
        return "unknown", "", None

    human_msg = convs[0]["value"]
    gpt_msg = convs[1]["value"]

    # Extract task prefix from human message
    m = _TASK_RE.search(human_msg)
    if not m:
        return "unknown", human_msg, gpt_msg

    task = m.group(1)

    # Extract instruction (everything after the prefix tag)
    instruction = human_msg[m.end():].strip()
    # Remove leading newline if present
    instruction = instruction.lstrip("\n").strip()

    return task, instruction, gpt_msg


def referring_to_qwen(answer: str) -> list[list[float]] | None:
    """Parse GeoChat-format referring answer to Qwen [0,1000] bbox.

    Input: "{<45><45><59><59>}"
    Output: [[450.0, 450.0, 590.0, 590.0]] or None if parse fails.
    """
    m = _REFERRING_RE.match(answer.strip())
    if not m:
        return None

    x1, y1, x2, y2 = (int(g) for g in m.groups())

    # Convert from GeoChat [0,100] to Qwen [0,1000]
    bbox = [[x1 * 10.0, y1 * 10.0, x2 * 10.0, y2 * 10.0]]

    if not validate_bbox_ordering(bbox):
        return None

    # Reject zero-area boxes
    bx1, by1, bx2, by2 = bbox[0]
    if bx1 >= bx2 or by1 >= by2:
        return None

    return bbox


def process_entry(
    entry: dict[str, Any],
    images_dir: Path | None,
    sample_id: str = "",
) -> list[dict[str, Any]]:
    """Process one VRSBench_train.json entry → list of sample dicts.

    Returns 0-1 samples (caller handles dual-resolution duplication).
    image_path points to the extracted source image (no copy).
    """
    task, instruction, response = parse_conversation(entry)
    image_name = entry.get("image", "")

    if not sample_id:
        sample_id = f"vrsbench_{image_name.replace('/', '_').replace('.', '_')}"

    if task == "unknown" or not instruction:
        return []

    # Reference extracted image directly — no copy
    image_path_str = ""
    if images_dir and image_name:
        img_path = images_dir / image_name
        if img_path.exists():
            image_path_str = str(img_path)

    if task == "caption":
        if not response:
            return []
        return [{
            "id": f"vrsbench_{sample_id}_caption",
            "dataset": _DATASET,
            "split": "train",
            "image_path": [image_path_str] if image_path_str else [],
            "bbox": None,
            "task": "caption",
            "pair_type": "single",
            "modality": "optical",
            "gsd_bucket": "",  # filled by caller
            "instruction": instruction,
            "response": response,
        }]

    elif task == "refer":
        if not response:
            return []
        bbox = referring_to_qwen(response)
        if bbox is None:
            return []
        return [{
            "id": f"vrsbench_{sample_id}_grounding",
            "dataset": _DATASET,
            "split": "train",
            "image_path": [image_path_str] if image_path_str else [],
            "bbox": bbox,
            "task": "grounding",
            "pair_type": "single",
            "modality": "optical",
            "gsd_bucket": "",  # filled by caller
            "instruction": instruction,
            "response": response,
        }]

    elif task == "vqa":
        if not response:
            return []
        return [{
            "id": f"vrsbench_{sample_id}_vqa",
            "dataset": _DATASET,
            "split": "train",
            "image_path": [image_path_str] if image_path_str else [],
            "bbox": None,
            "task": "vqa",
            "pair_type": "single",
            "modality": "optical",
            "gsd_bucket": "",  # filled by caller
            "instruction": instruction,
            "response": response,
        }]

    return []


def generate_proxy_image(
    src_path: Path,
    proxy_dir: Path,
    image_name: str,
    scale_factor: int = 4,
) -> str | None:
    """Generate a CARTOSAT-proxy (~2m) downsampled image via bicubic resize.

    Issue 7 fix: uses dynamic scale factor based on actual image dimensions
    rather than a hardcoded (128, 128) target, so any source size gets the
    intended 4× GSD reduction.

    Parameters
    ----------
    src_path : source image path.
    proxy_dir : directory to write the proxy PNG into.
    image_name : filename for the proxy (preserved from source).
    scale_factor : integer divisor applied to both width and height (default 4).
                   Resulting size is clamped to a minimum of 32px per side.

    Returns the proxy image path, or None on failure.
    """
    proxy_path = proxy_dir / image_name
    if proxy_path.exists():
        return str(proxy_path)

    try:
        img = Image.open(str(src_path))
        if img.mode != "RGB":
            img = img.convert("RGB")
        w, h = img.size
        # Dynamic 4× reduction — fixes Issue 7 (hardcoded 512→128 assumption)
        proxy_w = max(32, w // scale_factor)
        proxy_h = max(32, h // scale_factor)
        proxy = img.resize((proxy_w, proxy_h), Image.BICUBIC)
        proxy_path.parent.mkdir(parents=True, exist_ok=True)
        # Issue 9 fix: compress_level=6 (was 0)
        proxy.save(str(proxy_path), format="PNG", compress_level=6)
        return str(proxy_path)
    except Exception as e:
        print(f"  [WARN] Failed to generate proxy for {image_name}: {e}", file=sys.stderr)
        return None


def run_tier1_vrsbench(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
) -> dict[str, int]:
    """Run VRSBench preprocessing with manifest/resumability.

    Parses VRSBench_train.json directly. Each sample → native + proxy rows.
    Native images referenced from extracted source; proxy generated once per unique image.

    Returns stats dict: {processed, skipped, failed}.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "vrsbench.jsonl"
    manifest = Manifest(manifest_path)

    # Load consolidated JSON
    json_path = input_dir / "VRSBench_train.json"
    if not json_path.exists():
        raise FileNotFoundError(f"VRSBench_train.json not found in {input_dir}")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Check for images directory
    images_dir = None
    for name in ["Images_train", "images", "Images"]:
        candidate = input_dir / name
        if candidate.exists():
            images_dir = candidate
            break

    proxy_dir = output_dir / "images" / "proxy"
    proxy_dir.mkdir(parents=True, exist_ok=True)

    total = len(data)
    if max_samples:
        total = min(total, max_samples)
    print(f"Loaded {len(data)} entries, processing {total}")

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()
    native_bucket = assign_gsd_bucket("vrsbench", None)

    # Track unique images to generate proxy only once
    generated_proxies: set[str] = set()

    for i, entry in enumerate(data[:total]):
        sample_id = f"{i:07d}"
        manifest_id = sample_id

        if manifest.is_processed(manifest_id):
            stats["skipped"] += 1
            continue

        try:
            samples = process_entry(entry, images_dir, sample_id=sample_id)
        except Exception as e:
            print(f"  [FAIL] {sample_id}: {e}", file=sys.stderr)
            stats["failed"] += 1
            continue

        if not samples:
            stats["failed"] += 1
            continue

        # Emit native + proxy rows for each sample (R4 dual-resolution)
        for sample in samples:
            sample["gsd_bucket"] = native_bucket
            append_jsonl(sample, jsonl_path)

            # Generate proxy image once per unique source image
            proxy_path = None
            if sample["image_path"] and images_dir:
                image_name = Path(sample["image_path"][0]).name
                if image_name not in generated_proxies:
                    generate_proxy_image(
                        Path(sample["image_path"][0]),
                        proxy_dir,
                        image_name,
                    )
                    generated_proxies.add(image_name)
                proxy_path = str(proxy_dir / image_name)

            proxy_sample = create_proxy_sample(sample)
            # Point proxy sample to the downsampled image
            if proxy_path:
                proxy_sample["image_path"] = [proxy_path]
            append_jsonl(proxy_sample, jsonl_path)

        manifest.mark_processed(
            sample_id=manifest_id,
            dataset=_DATASET, split="train", output_shard="local",
        )

        stats["processed"] += 1
        if (i + 1) % 10000 == 0 or (i + 1) == total:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    print(f"Done: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Preprocess VRSBench → JSONL + PNGs (caption + grounding + VQA)",
    )
    parser.add_argument("--input-dir", type=Path, required=True,
                        help="Root VRSBench directory")
    parser.add_argument("--output-dir", type=Path, required=True,
                        help="Output directory for JSONL + images")
    parser.add_argument("--max-samples", type=int, default=None,
                        help="Process at most N entries (for testing)")
    args = parser.parse_args()

    run_tier1_vrsbench(args.input_dir, args.output_dir, max_samples=args.max_samples)


In [ ]:
%%writefile preprocess/tier2_sardet.py
"""Tier 2 — SARDet-100K preprocessing.

SARDet-100K provides 116K SAR images with object detection annotations
across 6 categories: ship, aircraft, bridge, tank, car, harbor.

Spec treatment:
    - R1/R3: SAR pseudo-RGB (VV, VH, VV-VH)
    - R5: GSD calibration to RISAT proxy band (~2-10m)
    - pair_type: single
    - dataset enum: "sardet"
    - gsd_bucket: RISAT-proxy[GSD:2m]
    - task: grounding (detection boxes)

SARDet-100K structure:
    <input_dir>/
        images/
            <image_id>.png (or .jpg)
        labels/
            <image_id>.txt (YOLO format: class cx cy w h)

Usage:
    python -m preprocess.tier2_sardet --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

from preprocess.common.bbox import pixel_to_normalized
from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.common.sample import stratified_sample
from preprocess.common.sar import sar_intensity_pseudo_gray
from preprocess.validator import validate_sample

_DATASET = "sardet"
_CATEGORIES = ["ship", "aircraft", "bridge", "tank", "car", "harbor"]


def primary_class(label_path: Path) -> str:
    """First annotated class in a YOLO label file — the stratification
    key for "stratified uniform across 6 categories"."""
    if not label_path.exists():
        return "unknown"
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                class_id = int(parts[0])
            except ValueError:
                continue
            if 0 <= class_id < len(_CATEGORIES):
                return _CATEGORIES[class_id]
    return "unknown"


def load_yolo_labels(
    label_path: Path,
    img_width: int,
    img_height: int,
) -> list[dict[str, Any]]:
    """Load YOLO format labels and convert to normalized bboxes.

    Returns list of {class_name, bbox: [[x1,y1,x2,y2]]}.
    """
    if not label_path.exists():
        return []

    annotations = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            class_id = int(parts[0])
            if class_id >= len(_CATEGORIES):
                continue

            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            # Convert from normalized center to pixel coords
            x_min = int((cx - w / 2) * img_width)
            y_min = int((cy - h / 2) * img_height)
            x_max = int((cx + w / 2) * img_width)
            y_max = int((cy + h / 2) * img_height)

            # Clamp
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            bbox_norm = pixel_to_normalized(
                (x_min, y_min, x_max, y_max),
                img_width,
                img_height,
            )
            if bbox_norm:
                annotations.append({
                    "class_name": _CATEGORIES[class_id],
                    "bbox": bbox_norm[0],
                })

    return annotations


def process_sardet_sample(
    image_path: Path,
    label_path: Path,
    output_dir: Path,
    sample_id: str,
) -> dict[str, Any] | None:
    """Process a single SARDet-100K sample."""
    try:
        from PIL import Image
        img = Image.open(str(image_path))
        if img.mode != "L":
            img = img.convert("L")  # SAR is single-channel

        # SARDet-100K ships single-channel intensity PNGs — no separate
        # VV/VH. R3's dual-pol B=(VV-VH) needs two real channels, so we
        # don't fabricate a second one; see sar_intensity_pseudo_gray().
        arr = np.array(img).astype(np.float32)
        rgb = sar_intensity_pseudo_gray(arr)
        w, h = rgb.shape[1], rgb.shape[0]
    except Exception as e:
        print(f"  [FAIL] {sample_id}: {e}", file=sys.stderr)
        return None

    # Save PNG
    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    out_path = img_dir / f"{sample_id}.png"
    write_png(rgb, out_path)

    # Load labels
    annotations = load_yolo_labels(label_path, w, h)
    if not annotations:
        return None

    # Use first bbox for primary task
    primary = annotations[0]
    all_bboxes = [a["bbox"] for a in annotations]

    class_names = list(set(a["class_name"] for a in annotations))
    response = ", ".join(class_names)

    return {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "grounding",
        "image_path": [str(out_path)],
        "pair_type": "single",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": "Locate the objects in this SAR image.",
        "response": response,
        "bbox": all_bboxes,
        "modality": "sar",
    }


def run_tier2_sardet(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
    sample_fraction: float = 0.10,
    seed: int = 42,
) -> dict[str, int]:
    """Run SARDet-100K preprocessing."""
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "sardet.jsonl"
    manifest = Manifest(manifest_path)

    sar_bucket = assign_gsd_bucket("sardet")

    images_dir = input_dir / "images"
    labels_dir = input_dir / "labels"

    if not images_dir.exists():
        print(f"  [WARN] Images directory not found: {images_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    # Find all images
    image_paths = sorted(images_dir.glob("*.png")) + sorted(images_dir.glob("*.jpg"))
    print(f"Found {len(image_paths)} SARDet-100K images")

    # Selection (spec): "stratified uniform across 6 categories".
    if sample_fraction < 1.0 and image_paths:
        keyed = [
            (p, primary_class(labels_dir / f"{p.stem}.txt"))
            for p in image_paths
        ]
        image_paths = [
            p for p, _ in stratified_sample(
                keyed, key_fn=lambda item: item[1],
                fraction=sample_fraction, seed=seed,
            )
        ]

    total = len(image_paths)
    if max_samples:
        total = min(total, max_samples)

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, image_path in enumerate(image_paths[:total]):
        sample_id = f"sardet_{image_path.stem}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        label_path = labels_dir / f"{image_path.stem}.txt"
        sample = process_sardet_sample(image_path, label_path, output_dir, sample_id)
        if sample is None:
            stats["failed"] += 1
            continue

        sample["gsd_bucket"] = sar_bucket

        ok, errs = validate_sample(sample)
        if not ok:
            print(f"  [VALID] {sample_id}: {errs}", file=sys.stderr)
            stats["failed"] += 1
            continue

        append_jsonl(sample, jsonl_path)
        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 1000 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess SARDet-100K")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    parser.add_argument("--sample-fraction", type=float, default=0.10)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_tier2_sardet(
        args.input_dir,
        args.output_dir,
        max_samples=args.max_samples,
        sample_fraction=args.sample_fraction,
        seed=args.seed,
    )


In [ ]:
%%writefile preprocess/tier2_sn6_sar.py
"""Tier 2 — SpaceNet 6 SAR preprocessing.

SpaceNet 6 provides co-registered Sentinel-1 SAR imagery alongside
optical WorldView-2. Used for cross-modal fusion + SAR artifacts.

Spec treatment:
    - R1/R3: Quad-pol SAR pseudo-RGB (R=HH, G=VV, B=VH)
    - R5: downsampled to RISAT proxy band (~2-10m)
    - pair_type: single (SAR only, optical handled by tier1_sn6_opt)
    - dataset enum: "sn6_sar"
    - gsd_bucket: RISAT-proxy[GSD:2m]
    - task: vqa (SAR interpretation)

SpaceNet 6 SAR structure:
    <input_dir>/
        train/
            sar/
                <tile_id>.tif (Sentinel-1 SAR)

Usage:
    python -m preprocess.tier2_sn6_sar --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

from preprocess.common.concat import concat_horizontal
from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.common.sar import sar_intensity_pseudo_gray, sar_pseudo_rgb
from preprocess.validator import validate_sample

_DATASET = "sn6_sar"


def load_optical_selection(sn6_opt_output_dir: Path) -> tuple[set[str], dict[str, str]]:
    """Read tier1_sn6_opt.py's output to pair SAR tile selection 1:1 with
    the optical tiles it already chose (spec: "Paired 1:1 with optical
    selection"), and to locate each tile's optical PNG for fusion samples.

    Returns (selected_tile_ids, tile_id -> optical_png_path).
    """
    selected_path = sn6_opt_output_dir / "selected_tile_ids.json"
    tile_ids: set[str] = set()
    if selected_path.exists():
        tile_ids = set(json.loads(selected_path.read_text()))

    optical_png_by_tile: dict[str, str] = {}
    jsonl_path = sn6_opt_output_dir / "sn6_opt.jsonl"
    if jsonl_path.exists():
        with open(jsonl_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                # ids look like "sn6_<tile_id>" (optionally "_proxy")
                if row.get("id", "").endswith("_proxy"):
                    continue
                tile_id = row["id"][len("sn6_"):]
                if row.get("image_path"):
                    optical_png_by_tile[tile_id] = row["image_path"][0]

    return tile_ids, optical_png_by_tile


def process_sar_tile(
    tile_path: Path,
    output_dir: Path,
    tile_id: str,
    *,
    optical_png: Path | None = None,
) -> list[dict[str, Any]]:
    """Process a single SpaceNet 6 SAR tile.

    Returns [sar_sample] normally, or [sar_sample, fusion_sample] when
    `optical_png` (the already-processed optical PNG for this same tile_id,
    from tier1_sn6_opt.py) is given — satisfying Mandate 4 for SpaceNet 6.
    Returns [] on failure.
    """
    try:
        import tifffile
        img = tifffile.imread(str(tile_path))
    except Exception as e:
        print(f"  [FAIL] {tile_id}: {e}", file=sys.stderr)
        return []

    # True quad-pol (HH, VV, VH) uses R3's real physics-corrected mapping
    # (linear->dB, per-pol clip, R=HH G=VV B=VH). A single band has no
    # real second/third polarization to build that from, so it gets the
    # honest single-channel dB render instead of a fabricated one.
    if img.ndim == 3 and img.shape[0] >= 3:
        hh = img[0].astype(np.float32)
        vv = img[1].astype(np.float32)
        vh = img[2].astype(np.float32)
        rgb = sar_pseudo_rgb(vv, vh, hh=hh)
    elif img.ndim == 3:
        rgb = sar_intensity_pseudo_gray(img[0].astype(np.float32))
    elif img.ndim == 2:
        rgb = sar_intensity_pseudo_gray(img.astype(np.float32))
    else:
        return []

    # Save PNG
    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    out_path = img_dir / f"sn6_sar_{tile_id}.png"
    write_png(rgb, out_path)

    sample_id = f"sn6_sar_{tile_id}"

    sar_sample = {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "vqa",
        "image_path": [str(out_path)],
        "pair_type": "single",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": "Describe the SAR characteristics of this scene.",
        "response": f"SAR image of {tile_id} showing surface scattering properties.",
        "bbox": None,
        "modality": "sar",
    }

    samples = [sar_sample]

    if optical_png is not None and optical_png.exists():
        opt_img = np.array(Image.open(str(optical_png)).convert("RGB"))
        concat_img = concat_horizontal(opt_img, rgb)
        concat_path = img_dir / f"sn6_fusion_{tile_id}.png"
        concat_img.save(str(concat_path), format="PNG", compress_level=0)

        samples.append({
            "id": f"sn6_fusion_{tile_id}",
            "dataset": _DATASET,
            "task": "fusion_vqa",
            "image_path": [str(optical_png), str(out_path)],
            "pair_type": "cross-modal",
            "gsd_bucket": "",  # Filled by caller
            "split": "train",
            "instruction": (
                "Using both the optical and SAR images, describe this "
                "scene's surface conditions."
            ),
            "response": (
                f"Optical and SAR imagery of {tile_id} show consistent "
                "surface conditions across both sensors."
            ),
            "bbox": None,
            "modality": "optical+sar",
        })

    return samples


def run_tier2_sn6_sar(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
    sample_fraction: float = 0.05,
    seed: int = 42,
    optical_dir: Path | None = None,
) -> dict[str, int]:
    """Run SpaceNet 6 SAR preprocessing.

    `optical_dir` should be tier1_sn6_opt.py's --output-dir. When given,
    SAR tile selection is intersected with the tiles it already chose
    ("paired 1:1 with optical selection" — Tier 2 table), and a
    cross-modal fusion_vqa sample is emitted per paired tile (Mandate 4).
    Without it, SAR tiles are sampled independently and no fusion rows
    are produced — pass optical_dir whenever tier1_sn6_opt has already run.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "sn6_sar.jsonl"
    manifest = Manifest(manifest_path)

    sar_bucket = assign_gsd_bucket("sn6_sar")

    # Find SAR tiles
    sar_dirs = [
        input_dir / "train" / "sar",
        input_dir / "sar",
        input_dir / "Sentinel-1",
    ]

    tile_paths = []
    for sar_dir in sar_dirs:
        if sar_dir.exists():
            tile_paths = sorted(sar_dir.glob("*.tif"))
            break

    if not tile_paths:
        print(f"  [WARN] No SAR tiles found in {input_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    print(f"Found {len(tile_paths)} SpaceNet 6 SAR tiles")

    optical_png_by_tile: dict[str, str] = {}
    if optical_dir is not None:
        selected_ids, optical_png_by_tile = load_optical_selection(optical_dir)
        if selected_ids:
            tile_paths = [p for p in tile_paths if p.stem in selected_ids]
            print(f"  Paired 1:1 with optical selection: {len(tile_paths)} tiles")
        else:
            print(f"  [WARN] No selected_tile_ids.json found under {optical_dir} — "
                  f"falling back to independent sampling")

    if optical_dir is None or not optical_png_by_tile:
        # ponytail: no optical pairing available — independent random
        # sample. Spec's stratification requirement here ("paired 1:1")
        # only applies when tier1_sn6_opt.py has already run; there's no
        # other natural stratification key for SAR tiles alone.
        import random
        rng = random.Random(seed)
        if sample_fraction < 1.0:
            n_sample = max(1, int(len(tile_paths) * sample_fraction))
            tile_paths = rng.sample(tile_paths, min(n_sample, len(tile_paths)))

    total = len(tile_paths)
    if max_samples:
        total = min(total, max_samples)

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, tile_path in enumerate(tile_paths[:total]):
        tile_id = tile_path.stem
        sample_id = f"sn6_sar_{tile_id}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        optical_png = optical_png_by_tile.get(tile_id)
        samples = process_sar_tile(
            tile_path, output_dir, tile_id,
            optical_png=Path(optical_png) if optical_png else None,
        )
        if not samples:
            stats["failed"] += 1
            continue

        for s in samples:
            s["gsd_bucket"] = sar_bucket

        errors = [(s["id"], e) for s in samples for ok, e in [validate_sample(s)] if not ok]
        if errors:
            for sid, errs in errors:
                print(f"  [VALID] {sid}: {errs}", file=sys.stderr)
            stats["failed"] += 1
            continue

        for s in samples:
            append_jsonl(s, jsonl_path)

        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 500 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess SpaceNet 6 SAR")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    parser.add_argument("--sample-fraction", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--optical-dir", type=Path, default=None,
                         help="tier1_sn6_opt.py's output dir, for paired 1:1 "
                              "selection and cross-modal fusion samples")
    args = parser.parse_args()

    run_tier2_sn6_sar(
        args.input_dir,
        args.output_dir,
        max_samples=args.max_samples,
        sample_fraction=args.sample_fraction,
        seed=args.seed,
        optical_dir=args.optical_dir,
    )


In [ ]:
%%writefile preprocess/tier3_sen2lulc.py
"""Tier 3 — Sen-2 LULC preprocessing.

Sen-2 LULC provides Sentinel-2 imagery over the Indian subcontinent
with 7-class land use/land cover annotations. Used for contextual
regularization with Indian geography.

Spec treatment:
    - R1: B4/B3/B2 → RGB (already in source)
    - R2: percentile normalization (deferred)
    - R4: NOT dual-resolution (10m native)
    - Masks → bounding boxes (R6 Qwen format)
    - pair_type: single
    - dataset enum: "sen2lulc"
    - gsd_bucket: [GSD:10m]
    - task: vqa (LULC classification)

Sen-2 LULC structure:
    <input_dir>/
        images/
            <image_id>.png (Sentinel-2 RGB)
        masks/
            <image_id>.png (class mask)
        metadata.csv or annotations.json
            Each entry: {"image_id": ..., "label": ...}

7-class Indian taxonomy:
    0: Built-up
    1: Vegetation
    2: Water
    3: Barren
    4: Agriculture
    5: Forest
    6: Wetland

Usage:
    python -m preprocess.tier3_sen2lulc --input-dir <path> --output-dir <path>
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

from preprocess.common.bbox import pixel_to_normalized
from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest, append_jsonl, write_png
from preprocess.common.sample import stratified_sample
from preprocess.validator import validate_sample

_DATASET = "sen2lulc"

# 7-class Indian taxonomy
_LULC_CLASSES = [
    "Built-up",
    "Vegetation",
    "Water",
    "Barren",
    "Agriculture",
    "Forest",
    "Wetland",
]


def load_metadata(input_dir: Path) -> list[dict]:
    """Load Sen-2 LULC metadata from CSV or JSON."""
    # Try CSV first
    csv_path = input_dir / "metadata.csv"
    if csv_path.exists():
        entries = []
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                entries.append(row)
        return entries

    # Try JSON
    json_path = input_dir / "annotations.json"
    if json_path.exists():
        with open(json_path) as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            return data.get("annotations", data.get("data", []))

    return []


def mask_to_bbox(mask_path: Path) -> list[list[float]] | None:
    """Convert class mask to normalized bounding boxes for each class."""
    try:
        from PIL import Image
        mask = np.array(Image.open(str(mask_path)).convert("L"))
    except Exception:
        return None

    # Find bounding boxes for each class
    bboxes = []
    for class_id in range(len(_LULC_CLASSES)):
        class_mask = (mask == class_id)
        if class_mask.sum() == 0:
            continue

        ys, xs = np.where(class_mask)
        h, w = mask.shape

        bbox_pixel = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]
        bbox_norm = pixel_to_normalized(tuple(bbox_pixel), w, h)
        if bbox_norm:
            bboxes.append(bbox_norm[0])

    return bboxes if bboxes else None


def parse_sen2lulc_sample(
    entry: dict[str, Any],
    images_dir: Path | None,
    masks_dir: Path | None,
    sample_id: str,
) -> dict[str, Any] | None:
    """Parse a single Sen-2 LULC annotation into our JSONL format."""
    image_id = entry.get("image_id", entry.get("id", ""))
    label = entry.get("label", entry.get("class", ""))

    if not image_id:
        return None

    # Handle image path
    image_path_str = ""
    if images_dir:
        for ext in [".png", ".jpg", ".tif"]:
            img_path = images_dir / f"{image_id}{ext}"
            if img_path.exists():
                image_path_str = str(img_path)
                break

    # Get bounding boxes from mask
    bbox = None
    if masks_dir:
        for ext in [".png", ".tif"]:
            mask_path = masks_dir / f"{image_id}{ext}"
            if mask_path.exists():
                bbox = mask_to_bbox(mask_path)
                break

    # Build response from label. CSV-sourced labels are always strings
    # (csv.DictReader doesn't type-coerce) — "2" must still resolve to
    # "Water", not fall through to being used as a literal class name.
    label_idx: int | None = None
    if isinstance(label, bool):
        pass
    elif isinstance(label, int):
        label_idx = label
    elif isinstance(label, str) and label.strip().lstrip("-").isdigit():
        label_idx = int(label.strip())

    if label_idx is not None and 0 <= label_idx < len(_LULC_CLASSES):
        response = _LULC_CLASSES[label_idx]
    elif isinstance(label, str) and label:
        response = label
    else:
        response = "Unknown land cover"

    return {
        "id": sample_id,
        "dataset": _DATASET,
        "task": "vqa",
        "image_path": [image_path_str] if image_path_str else [],
        "pair_type": "single",
        "gsd_bucket": "",  # Filled by caller
        "split": "train",
        "instruction": "What is the primary land cover type in this image?",
        "response": response,
        "bbox": bbox,
        "modality": "optical",
    }


def run_tier3_sen2lulc(
    input_dir: Path,
    output_dir: Path,
    *,
    max_samples: int | None = None,
    sample_fraction: float = 0.05,
    seed: int = 42,
) -> dict[str, int]:
    """Run Sen-2 LULC preprocessing."""
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.parquet"
    jsonl_path = output_dir / "sen2lulc.jsonl"
    manifest = Manifest(manifest_path)

    native_bucket = assign_gsd_bucket("sen2lulc")

    annotations = load_metadata(input_dir)
    if not annotations:
        print(f"  [WARN] No annotations found in {input_dir}")
        return {"processed": 0, "skipped": 0, "failed": 0}

    images_dir = None
    for name in ["images", "Images", "img"]:
        candidate = input_dir / name
        if candidate.exists():
            images_dir = candidate
            break

    masks_dir = None
    for name in ["masks", "Masks", "label", "labels"]:
        candidate = input_dir / name
        if candidate.exists():
            masks_dir = candidate
            break

    print(f"Loaded {len(annotations)} annotations")

    # Selection (spec): "5% (~10k of 213,761), stratified uniform across
    # 7 classes". Without this, the full 213k-entry set gets processed —
    # a 20x resource blowout against Kaggle's 20GB /kaggle/working cap.
    if sample_fraction < 1.0:
        def _label_key(entry: dict) -> object:
            return entry.get("label", entry.get("class", "unknown"))

        annotations = stratified_sample(
            annotations, key_fn=_label_key, fraction=sample_fraction, seed=seed,
        )
        print(f"Stratified sample: {len(annotations)} annotations "
              f"({sample_fraction:.0%} per class)")

    total = len(annotations)
    if max_samples:
        total = min(total, max_samples)
    print(f"Processing {total}")

    stats = {"processed": 0, "skipped": 0, "failed": 0}
    start = time.time()

    for i, entry in enumerate(annotations[:total]):
        image_id = entry.get("image_id", entry.get("id", i))
        sample_id = f"sen2lulc_{image_id}"

        if manifest.is_processed(sample_id):
            stats["skipped"] += 1
            continue

        sample = parse_sen2lulc_sample(entry, images_dir, masks_dir, sample_id)
        if sample is None:
            stats["failed"] += 1
            continue

        sample["gsd_bucket"] = native_bucket

        ok, errs = validate_sample(sample)
        if not ok:
            print(f"  [VALID] {sample_id}: {errs}", file=sys.stderr)
            stats["failed"] += 1
            continue

        append_jsonl(sample, jsonl_path)
        manifest.mark_processed(
            sample_id=sample_id,
            dataset=_DATASET,
            split="train",
            output_shard="local",
        )
        stats["processed"] += 1

        if (i + 1) % 5000 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            print(f"  Processed {i + 1}/{total} ({rate:.1f} entries/s)")

    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s: {stats['processed']} processed, "
          f"{stats['skipped']} skipped, {stats['failed']} failed")
    return stats


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Preprocess Sen-2 LULC")
    parser.add_argument("--input-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-samples", type=int, default=None)
    parser.add_argument("--sample-fraction", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_tier3_sen2lulc(
        args.input_dir,
        args.output_dir,
        max_samples=args.max_samples,
        sample_fraction=args.sample_fraction,
        seed=args.seed,
    )


In [ ]:
%%writefile preprocess/validator.py
"""SatQuery JSONL schema validator.

Validates every training/inference sample against the frozen schema (Section 4)
and enforces cross-field consistency rules.

Public API:
    validate_sample(sample) -> (is_valid, errors)
    validate_jsonl(path) -> (count, errors)
    cross_field_checks(sample) -> list[str]
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from jsonschema import Draft7Validator

# ---------------------------------------------------------------------------
# Frozen JSONL schema — Section 4 of the v4 specification
# ---------------------------------------------------------------------------

# Bbox uses oneOf: null | array-of-arrays.  The [0,1000] range constraint on
# each element is enforced by _validate_bbox_values() below because Draft7
# does not support prefixItems (2020-12 feature).
_BBOX_SCHEMA = {
    "oneOf": [
        {"type": "null"},
        {
            "type": "array",
            "items": {
                "type": "array",
                "items": {"type": "number"},
                "minItems": 4,
                "maxItems": 4,
            },
        },
    ]
}

SCHEMA: dict[str, Any] = {
    "type": "object",
    "required": [
        "id",
        "dataset",
        "task",
        "image_path",
        "pair_type",
        "gsd_bucket",
        "split",
        "instruction",
        "response",
        "bbox",
        "modality",
    ],
    "properties": {
        "id": {"type": "string", "pattern": "^[a-z0-9_]+$"},
        "dataset": {
            "enum": [
                "bigen",
                "vrsbench",
                "rsvqa_hr",
                "cdvqa",
                "levir_cd",
                "sn6_opt",
                "oscd",
                "sardet",
                "sn6_sar",
                "sen2lulc",
            ]
        },
        "task": {
            "enum": [
                "vqa",
                "caption",
                "grounding",
                "change_vqa",
                "change_grounding",
                "fusion_vqa",
                "fusion_grounding",
            ]
        },
        "image_path": {
            "type": "array",
            "minItems": 1,
            "maxItems": 2,
            "items": {"type": "string", "minLength": 1},
        },
        "pair_type": {"enum": ["single", "bitemporal", "cross-modal"]},
        "gsd_bucket": {"type": "string", "minLength": 1},
        "split": {"enum": ["train", "val_internal"]},
        "instruction": {"type": "string", "minLength": 1},
        "response": {"type": "string", "minLength": 1},
        "bbox": _BBOX_SCHEMA,
        "modality": {"enum": ["optical", "sar", "optical+sar"]},
    },
    "additionalProperties": False,
}

_validator = Draft7Validator(SCHEMA)


def _validate_bbox_values(bbox: list[list[float]]) -> list[str]:
    """Validate that each bbox coordinate is in [0, 1000]."""
    errors: list[str] = []
    for i, box in enumerate(bbox):
        if len(box) != 4:
            errors.append(f"bbox[{i}] must have exactly 4 elements, got {len(box)}")
            continue
        for j, val in enumerate(box):
            if not isinstance(val, (int, float)):
                errors.append(f"bbox[{i}][{j}] must be a number, got {type(val).__name__}")
            elif val < 0 or val > 1000:
                errors.append(
                    f"bbox[{i}][{j}]={val} out of range [0, 1000]"
                )
    return errors


# ---------------------------------------------------------------------------
# Enum sets for cross-field checks
# ---------------------------------------------------------------------------

_GROUNDING_TASKS = {"grounding", "change_grounding", "fusion_grounding"}
_VQA_LIKE_TASKS = {"vqa", "caption", "change_vqa", "fusion_vqa"}
MultiImagePairTypes = {"bitemporal", "cross-modal"}


# ---------------------------------------------------------------------------
# Cross-field consistency checks (beyond schema validation)
# ---------------------------------------------------------------------------


def cross_field_checks(sample: dict[str, Any]) -> list[str]:
    """Validate consistency rules between fields.

    Rules enforced:
        R-pair  pair_type=single  <->  len(image_path)==1
        R-pair  pair_type in {bitemporal,cross-modal} <-> len(image_path)==2
        R-bbox  grounding/change_grounding/fusion_grounding -> bbox is non-null
        R-bbox  vqa/caption/change_vqa/fusion_vqa -> bbox is null
        R-bbox  change_grounding requires pair_type=bitemporal AND len(image_path)==2
        R-bbox  fusion_grounding requires pair_type=cross-modal AND len(image_path)==2
        R-mod   pair_type=cross-modal -> modality == "optical+sar"
    """
    errors: list[str] = []
    pair_type = sample.get("pair_type")
    image_path = sample.get("image_path", [])
    task = sample.get("task")
    bbox = sample.get("bbox")
    modality = sample.get("modality")
    n_images = len(image_path)

    # R-pair: pair_type <-> image_path length
    if pair_type == "single" and n_images != 1:
        errors.append(
            f"pair_type='single' requires exactly 1 image, got {n_images}"
        )
    if pair_type in MultiImagePairTypes and n_images != 2:
        errors.append(
            f"pair_type='{pair_type}' requires exactly 2 images, got {n_images}"
        )

    # R-bbox: task <-> bbox presence
    if task in _GROUNDING_TASKS and (bbox is None or len(bbox) == 0):
        errors.append(
            f"task='{task}' requires a non-empty bbox list, got {bbox!r}"
        )
    if task in _VQA_LIKE_TASKS and bbox is not None:
        errors.append(
            f"task='{task}' requires null bbox, got non-null"
        )

    # R-bbox: change_grounding -> bitemporal + 2 images
    if task == "change_grounding":
        if pair_type != "bitemporal":
            errors.append(
                f"task='change_grounding' requires pair_type='bitemporal', "
                f"got pair_type='{pair_type}'"
            )
        if n_images != 2:
            errors.append(
                f"task='change_grounding' requires 2 images, got {n_images}"
            )

    # R-bbox: fusion_grounding -> cross-modal + 2 images
    if task == "fusion_grounding":
        if pair_type != "cross-modal":
            errors.append(
                f"task='fusion_grounding' requires pair_type='cross-modal', "
                f"got pair_type='{pair_type}'"
            )
        if n_images != 2:
            errors.append(
                f"task='fusion_grounding' requires 2 images, got {n_images}"
            )

    # R-mod: cross-modal -> optical+sar
    if pair_type == "cross-modal" and modality != "optical+sar":
        errors.append(
            f"pair_type='cross-modal' requires modality='optical+sar', "
            f"got modality='{modality}'"
        )

    return errors


# ---------------------------------------------------------------------------
# Public validation API
# ---------------------------------------------------------------------------


def validate_sample(sample: dict[str, Any]) -> tuple[bool, list[str]]:
    """Validate a single JSONL row against schema + cross-field rules.

    Returns (is_valid, list_of_error_messages).
    """
    errors: list[str] = []

    # Schema validation — include field path for clarity
    for err in _validator.iter_errors(sample):
        path = ".".join(str(p) for p in err.absolute_path) or "<root>"
        errors.append(f"schema [{path}]: {err.message}")

    # Bbox value range check (Draft7 can't express [0,1000] per-element)
    bbox = sample.get("bbox")
    if isinstance(bbox, list) and not errors:
        errors.extend(_validate_bbox_values(bbox))

    # Cross-field checks (only if schema passed — fields may be missing)
    if not errors:
        errors.extend(cross_field_checks(sample))

    return (len(errors) == 0, errors)


def validate_jsonl(path: str | Path) -> tuple[int, list[str]]:
    """Validate every line in a JSONL file.

    Returns (total_lines, list_of_all_errors_with_line_prefix).
    """
    path = Path(path)
    all_errors: list[str] = []
    count = 0

    with open(path, "r", encoding="utf-8") as f:
        for line_num, raw_line in enumerate(f, start=1):
            line = raw_line.strip()
            if not line:
                continue
            count += 1
            try:
                sample = json.loads(line)
            except json.JSONDecodeError as exc:
                all_errors.append(f"line {line_num}: JSON parse error: {exc}")
                continue

            valid, errs = validate_sample(sample)
            if not valid:
                for e in errs:
                    all_errors.append(f"line {line_num}: {e}")

    return (count, all_errors)


## `training/`

In [ ]:
%%writefile training/benchmark_dora.py
"""DoRA vs QLoRA Benchmark — 50-step comparison at Stage 1 start.

Runs 50 training steps with both QDoRA and QLoRA, comparing:
- Training loss convergence
- Memory usage
- Steps per second
- Final adapter size

Usage:
    python -m training.benchmark_dora --data data/bigen_output/bigen.jsonl
"""

from __future__ import annotations

import argparse
import json
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch


@dataclass
class BenchmarkResult:
    method: str
    steps: int
    final_loss: float
    avg_loss: float
    loss_curve: list[float]
    memory_mb: float
    steps_per_sec: float
    adapter_size_mb: float
    elapsed_sec: float
    loss_is_real: bool = False


def load_benchmark_data(jsonl_path: Path, n_samples: int = 100) -> list[dict]:
    """Load a small subset for benchmarking."""
    samples = []
    with open(jsonl_path) as f:
        for i, line in enumerate(f):
            if i >= n_samples:
                break
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


def setup_model(use_dora: bool = True):
    """Setup model with either QDoRA or QLoRA."""
    try:
        from unsloth import FastVisionModel
    except ImportError:
        print("  [ERROR] unsloth not installed")
        return None, None

    model, tokenizer = FastVisionModel.from_pretrained(
        "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )

    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_dora=use_dora,
    )

    return model, tokenizer


def run_benchmark(
    use_dora: bool,
    data: list[dict],
    n_steps: int = 50,
    batch_size: int = 2,
) -> BenchmarkResult:
    """Run benchmark for specified number of steps."""
    method = "QDoRA" if use_dora else "QLoRA"
    print(f"\nRunning {method} benchmark ({n_steps} steps)...")

    # Setup
    model, tokenizer = setup_model(use_dora)
    if model is None:
        return BenchmarkResult(
            method=method,
            steps=0,
            final_loss=float("inf"),
            avg_loss=float("inf"),
            loss_curve=[],
            memory_mb=0,
            steps_per_sec=0,
            adapter_size_mb=0,
            elapsed_sec=0,
        )

    # ponytail: no real forward/backward pass here yet (needs ChatML batch
    # formatting against the actual model — a GPU-session task, not a
    # preprocessing-repo one). Memory and steps/sec below ARE real
    # (measured from the actual loaded model), but loss is not simulated
    # here at all: run_benchmark() refuses to report a fabricated loss
    # curve rather than let compare_results() silently pick a "winner" on
    # numbers that don't depend on use_dora. Wire in real train steps
    # (see training/train.py's TrainingLoop.train_step, same stub) before
    # trusting this script's loss/convergence comparison.
    loss_curve: list[float] = []
    start_time = time.time()

    for step in range(n_steps):
        pass  # real forward/backward pass goes here

    elapsed = time.time() - start_time

    # Get memory usage
    memory_mb = 0
    if torch.cuda.is_available():
        memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024

    # Estimate adapter size
    adapter_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    adapter_size_mb = adapter_params * 4 / 1024 / 1024  # float32

    return BenchmarkResult(
        method=method,
        steps=n_steps,
        final_loss=float("nan"),
        avg_loss=float("nan"),
        loss_curve=loss_curve,
        memory_mb=memory_mb,
        steps_per_sec=n_steps / elapsed if elapsed > 0 else 0,
        adapter_size_mb=adapter_size_mb,
        elapsed_sec=elapsed,
        loss_is_real=False,
    )


def compare_results(dora: BenchmarkResult, qlora: BenchmarkResult) -> dict:
    """Compare QDoRA vs QLoRA results."""
    comparison = {
        "dora": {
            "final_loss": dora.final_loss,
            "avg_loss": dora.avg_loss,
            "memory_mb": dora.memory_mb,
            "steps_per_sec": dora.steps_per_sec,
            "adapter_size_mb": dora.adapter_size_mb,
            "elapsed_sec": dora.elapsed_sec,
        },
        "qlora": {
            "final_loss": qlora.final_loss,
            "avg_loss": qlora.avg_loss,
            "memory_mb": qlora.memory_mb,
            "steps_per_sec": qlora.steps_per_sec,
            "adapter_size_mb": qlora.adapter_size_mb,
            "elapsed_sec": qlora.elapsed_sec,
        },
        "winner": {
            "speed": "dora" if dora.steps_per_sec > qlora.steps_per_sec else "qlora",
            "memory": "dora" if dora.memory_mb < qlora.memory_mb else "qlora",
        },
    }

    if dora.loss_is_real and qlora.loss_is_real:
        comparison["winner"]["loss"] = "dora" if dora.final_loss < qlora.final_loss else "qlora"
    else:
        # No real forward/backward pass wired in yet (see run_benchmark) —
        # reporting a loss winner here would be fabricated, not measured.
        comparison["winner"]["loss"] = "unmeasured"

    # Determine overall recommendation from measured axes only.
    dora_wins = sum(1 for v in comparison["winner"].values() if v == "dora")
    qlora_wins = sum(1 for v in comparison["winner"].values() if v == "qlora")
    if comparison["winner"]["loss"] == "unmeasured":
        comparison["recommendation"] = "INCONCLUSIVE — loss not measured, see loss_is_real"
    else:
        comparison["recommendation"] = "use_dora" if dora_wins >= qlora_wins else "use_qlora"

    return comparison


def main():
    parser = argparse.ArgumentParser(description="DoRA vs QLoRA Benchmark")
    parser.add_argument("--data", type=Path, required=True, help="JSONL data file")
    parser.add_argument("--n-steps", type=int, default=50, help="Number of steps")
    parser.add_argument("--n-samples", type=int, default=100, help="Number of samples")
    parser.add_argument("--output", type=Path, default=None, help="Output JSON file")
    args = parser.parse_args()

    # Load data
    data = load_benchmark_data(args.data, args.n_samples)
    print(f"Loaded {len(data)} samples for benchmarking")

    # Run benchmarks
    dora_result = run_benchmark(use_dora=True, data=data, n_steps=args.n_steps)
    qlora_result = run_benchmark(use_dora=False, data=data, n_steps=args.n_steps)

    # Compare
    comparison = compare_results(dora_result, qlora_result)

    # Print summary
    print("\n" + "=" * 60)
    print("BENCHMARK RESULTS")
    print("=" * 60)
    print(f"\n{'Metric':<20} {'QDoRA':<15} {'QLoRA':<15} {'Winner':<10}")
    print("-" * 60)
    print(f"{'Final Loss':<20} {dora_result.final_loss:<15.4f} {qlora_result.final_loss:<15.4f} {comparison['winner']['loss']}")
    print(f"{'Avg Loss':<20} {dora_result.avg_loss:<15.4f} {qlora_result.avg_loss:<15.4f}")
    print(f"{'Memory (MB)':<20} {dora_result.memory_mb:<15.1f} {qlora_result.memory_mb:<15.1f} {comparison['winner']['memory']}")
    print(f"{'Steps/sec':<20} {dora_result.steps_per_sec:<15.2f} {qlora_result.steps_per_sec:<15.2f} {comparison['winner']['speed']}")
    print(f"{'Adapter (MB)':<20} {dora_result.adapter_size_mb:<15.2f} {qlora_result.adapter_size_mb:<15.2f}")
    print(f"\nRecommendation: {comparison['recommendation']}")

    # Save results
    if args.output:
        output = {
            "dora": dora_result.__dict__,
            "qlora": qlora_result.__dict__,
            "comparison": comparison,
        }
        with open(args.output, "w") as f:
            json.dump(output, f, indent=2)
        print(f"\nResults saved to {args.output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training/train.py
"""SatQuery Training Script — Multi-stage QDoRA/QLoRA fine-tuning.

Supports 3-stage curriculum with:
- QDoRA or QLoRA (config flag)
- Checkpoint resume
- 10% replay from previous stages
- Dual-resolution curriculum (native:proxy ratio)
- Internal validation monitoring

Usage:
    python -m training.train --config training/configs/stage1.yaml
    python -m training.train --config training/configs/stage2.yaml --resume checkpoints/stage1/last
"""

from __future__ import annotations

import argparse
import json
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import yaml


@dataclass
class ModelConfig:
    name: str = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
    use_dora: bool = True
    adapter_path: str | None = None


@dataclass
class TrainingConfig:
    stage: int = 1
    epochs: float = 1.5
    batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    lr_scheduler: str = "cosine"
    warmup_ratio: float = 0.05
    max_steps: int = 14000
    weight_decay: float = 0.01


@dataclass
class DataConfig:
    jsonl_dir: str = ""
    jsonl_dirs: list[str] = field(default_factory=list)
    val_jsonl: str | None = None
    replay_fraction: float = 0.0
    replay_jsonl: str | None = None
    max_samples: int | None = None


@dataclass
class CurriculumConfig:
    native_proxy_ratio: list[float] = field(default_factory=lambda: [1.0, 0.0])


@dataclass
class CheckpointConfig:
    save_every: int = 2000
    output_dir: str = "checkpoints"
    resume_from: str | None = None


@dataclass
class LoggingConfig:
    eval_every: int = 500
    log_every: int = 50
    wandb_project: str = "satquery"
    wandb_run_name: str = "run"


@dataclass
class TrainConfig:
    model: ModelConfig = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    data: DataConfig = field(default_factory=DataConfig)
    curriculum: CurriculumConfig = field(default_factory=CurriculumConfig)
    checkpoint: CheckpointConfig = field(default_factory=CheckpointConfig)
    logging: LoggingConfig = field(default_factory=LoggingConfig)


def load_config(config_path: Path) -> TrainConfig:
    """Load training config from YAML file."""
    with open(config_path) as f:
        raw = yaml.safe_load(f)

    config = TrainConfig()
    if "model" in raw:
        config.model = ModelConfig(**raw["model"])
    if "training" in raw:
        config.training = TrainingConfig(**raw["training"])
    if "data" in raw:
        config.data = DataConfig(**raw["data"])
    if "curriculum" in raw:
        config.curriculum = CurriculumConfig(**raw["curriculum"])
    if "checkpoint" in raw:
        config.checkpoint = CheckpointConfig(**raw["checkpoint"])
    if "logging" in raw:
        config.logging = LoggingConfig(**raw["logging"])

    return config


def load_jsonl(jsonl_path: Path) -> list[dict]:
    """Load samples from JSONL file."""
    samples = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


def get_native_proxy_split(
    samples: list[dict],
    ratio: float,
    seed: int = 42,
) -> tuple[list[dict], list[dict]]:
    """Split samples into native and proxy based on ratio.

    Args:
        samples: All samples
        ratio: Native fraction (0.0 to 1.0)
        seed: Random seed

    Returns:
        (native_samples, proxy_samples)
    """
    import random
    rng = random.Random(seed)

    native = [s for s in samples if "proxy" not in s.get("gsd_bucket", "").lower()]
    proxy = [s for s in samples if "proxy" in s.get("gsd_bucket", "").lower()]

    # If no proxy samples, return all as native
    if not proxy:
        return samples, []

    n_native = int(len(samples) * ratio)
    n_proxy = len(samples) - n_native

    # Sample proportionally
    rng.shuffle(native)
    rng.shuffle(proxy)

    selected_native = native[:n_native]
    selected_proxy = proxy[:n_proxy]

    return selected_native, selected_proxy


def get_replay_samples(
    replay_jsonl: Path,
    fraction: float,
    n_samples: int,
    seed: int = 42,
) -> list[dict]:
    """Load replay samples from previous stage."""
    import random

    if not replay_jsonl.exists() or fraction <= 0:
        return []

    all_samples = load_jsonl(replay_jsonl)
    n_replay = int(n_samples * fraction)

    rng = random.Random(seed)
    return rng.sample(all_samples, min(n_replay, len(all_samples)))


def compute_curriculum_ratio(
    step: int,
    max_steps: int,
    ratios: list[float],
) -> float:
    """Compute native:proxy ratio at current step.

    Transitions smoothly between ratios at 33% and 66% of training.
    """
    if len(ratios) == 1:
        return ratios[0]

    progress = step / max_steps

    if progress < 0.33:
        return ratios[0]
    elif progress < 0.66:
        if len(ratios) >= 2:
            return ratios[1]
        return ratios[0]
    else:
        if len(ratios) >= 3:
            return ratios[2]
        elif len(ratios) >= 2:
            return ratios[1]
        return ratios[0]


class TrainingLoop:
    """Main training loop with checkpoint resume and validation."""

    def __init__(self, config: TrainConfig):
        self.config = config
        self.step = 0
        self.epoch = 0
        self.best_val_loss = float("inf")

    def setup_model(self):
        """Initialize model with QDoRA or QLoRA adapter."""
        try:
            from unsloth import FastVisionModel
        except ImportError:
            print("  [WARN] unsloth not installed, using mock model for testing")
            return None

        model, tokenizer = FastVisionModel.from_pretrained(
            self.config.model.name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth",
        )

        # Add adapter
        if self.config.model.use_dora:
            print("  Using QDoRA adapter")
            model = FastVisionModel.get_peft_model(
                model,
                finetune_vision_layers=False,
                finetune_language_layers=True,
                finetune_attention_modules=True,
                finetune_mlp_modules=True,
                r=16,
                lora_alpha=16,
                lora_dropout=0,
                bias="none",
                random_state=3407,
                use_dora=True,
            )
        else:
            print("  Using QLoRA adapter")
            model = FastVisionModel.get_peft_model(
                model,
                finetune_vision_layers=False,
                finetune_language_layers=True,
                finetune_attention_modules=True,
                finetune_mlp_modules=True,
                r=16,
                lora_alpha=16,
                lora_dropout=0,
                bias="none",
                random_state=3407,
                use_dora=False,
            )

        # Load adapter checkpoint if resuming
        if self.config.model.adapter_path:
            adapter_path = Path(self.config.model.adapter_path)
            if adapter_path.exists():
                print(f"  Loading adapter from {adapter_path}")
                # Load adapter weights
                from safetensors import safe_open
                # Implementation depends on adapter format

        return model, tokenizer

    def load_data(self) -> list[dict]:
        """Load training data with optional replay."""
        all_samples = []

        # Load main data
        jsonl_dirs = self.config.data.jsonl_dirs or [self.config.data.jsonl_dir]
        for jsonl_dir in jsonl_dirs:
            dir_path = Path(jsonl_dir)
            if dir_path.exists():
                for jsonl_path in dir_path.glob("*.jsonl"):
                    if "_val_internal" in jsonl_path.name:
                        continue
                    samples = load_jsonl(jsonl_path)
                    all_samples.extend(samples)

        # Apply max_samples limit
        if self.config.data.max_samples:
            all_samples = all_samples[:self.config.data.max_samples]

        # Add replay samples
        if self.config.data.replay_fraction > 0 and self.config.data.replay_jsonl:
            replay_path = Path(self.config.data.replay_jsonl)
            replay = get_replay_samples(
                replay_path,
                self.config.data.replay_fraction,
                len(all_samples),
            )
            all_samples.extend(replay)
            print(f"  Added {len(replay)} replay samples")

        return all_samples

    def load_val_data(self) -> list[dict]:
        """Load validation data."""
        if not self.config.data.val_jsonl:
            return []

        val_path = Path(self.config.data.val_jsonl)
        if val_path.exists():
            return load_jsonl(val_path)
        return []

    def train_step(self, batch: list[dict], model, tokenizer) -> dict[str, float]:
        """Single training step.

        Returns loss dict.
        """
        # ponytail: no real forward/backward pass yet. Real implementation:
        # 1. Format batch into ChatML messages (per-sample pair_type ->
        #    template, per Section 11 of SPEC.md)
        # 2. Tokenize with image inputs
        # 3. Forward pass with loss computation
        # 4. Backward pass with gradient accumulation
        # 5. Optimizer step
        # Needs a live GPU session to write+verify against the real
        # Unsloth/Qwen3-VL API — do this on Kaggle, not blind.
        return {"loss": float("nan")}

    def validate(self, model, tokenizer, val_data: list[dict]) -> dict[str, float]:
        """Run validation and return metrics."""
        if not val_data:
            return {"val_loss": float("nan")}

        # ponytail: same stub as train_step — no real forward pass yet.
        return {"val_loss": float("nan")}

    def save_checkpoint(self, model, step: int):
        """Save model checkpoint."""
        output_dir = Path(self.config.checkpoint.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        ckpt_dir = output_dir / f"step_{step:06d}"
        ckpt_dir.mkdir(exist_ok=True)

        # Save adapter weights
        if hasattr(model, "save_pretrained"):
            model.save_pretrained(str(ckpt_dir))

        # Save training state
        state = {
            "step": step,
            "epoch": self.epoch,
            "best_val_loss": self.best_val_loss,
            "config": {
                "model": self.config.model.__dict__,
                "training": self.config.training.__dict__,
            },
        }
        with open(ckpt_dir / "training_state.json", "w") as f:
            json.dump(state, f, indent=2)

        print(f"  Saved checkpoint to {ckpt_dir}")

    def run(self):
        """Main training loop."""
        print("=" * 60)
        print("  WARNING: train_step()/validate() are stubs (return NaN loss).")
        print("  This loop moves data and saves checkpoints on schedule but")
        print("  does NOT actually train anything yet. See train_step().")
        print("=" * 60)
        print(f"Starting Stage {self.config.training.stage} training")
        print(f"  Model: {self.config.model.name}")
        print(f"  DoRA: {self.config.model.use_dora}")
        print(f"  Max steps: {self.config.training.max_steps}")

        # Setup
        model_tokenizer = self.setup_model()
        if model_tokenizer is None:
            print("  [ERROR] Failed to setup model")
            return

        model, tokenizer = model_tokenizer

        # Load data
        train_data = self.load_data()
        val_data = self.load_val_data()
        print(f"  Training samples: {len(train_data)}")
        print(f"  Validation samples: {len(val_data)}")

        # Resume from checkpoint if specified
        if self.config.checkpoint.resume_from:
            resume_path = Path(self.config.checkpoint.resume_from)
            if resume_path.exists():
                state_path = resume_path / "training_state.json"
                if state_path.exists():
                    with open(state_path) as f:
                        state = json.load(f)
                    self.step = state.get("step", 0)
                    self.epoch = state.get("epoch", 0)
                    print(f"  Resumed from step {self.step}")

        # Training loop
        start_time = time.time()
        while self.step < self.config.training.max_steps:
            # Get current curriculum ratio
            ratio = compute_curriculum_ratio(
                self.step,
                self.config.training.max_steps,
                self.config.curriculum.native_proxy_ratio,
            )

            # Split batch by native/proxy
            native, proxy = get_native_proxy_split(train_data, ratio)

            # Train step (placeholder)
            losses = self.train_step([], model, tokenizer)

            self.step += 1

            # Logging
            if self.step % self.config.logging.log_every == 0:
                elapsed = time.time() - start_time
                steps_per_sec = self.step / elapsed if elapsed > 0 else 0
                print(f"  Step {self.step}/{self.config.training.max_steps} "
                      f"({steps_per_sec:.1f} steps/s) "
                      f"loss={losses.get('loss', 0):.4f} "
                      f"native:proxy={ratio:.2f}:{1-ratio:.2f}")

            # Validation
            if self.step % self.config.logging.eval_every == 0:
                val_metrics = self.validate(model, tokenizer, val_data)
                print(f"  [VAL] step={self.step} loss={val_metrics.get('val_loss', 0):.4f}")

                # Save best model
                if val_metrics.get("val_loss", float("inf")) < self.best_val_loss:
                    self.best_val_loss = val_metrics["val_loss"]
                    self.save_checkpoint(model, self.step)

            # Save periodic checkpoint
            if self.step % self.config.checkpoint.save_every == 0:
                self.save_checkpoint(model, self.step)

        # Final save
        self.save_checkpoint(model, self.step)
        print(f"\nTraining complete. Final step: {self.step}")


def main():
    parser = argparse.ArgumentParser(description="SatQuery Training")
    parser.add_argument("--config", type=Path, required=True, help="Training config YAML")
    parser.add_argument("--resume", type=str, default=None, help="Resume from checkpoint")
    args = parser.parse_args()

    config = load_config(args.config)
    if args.resume:
        config.checkpoint.resume_from = args.resume

    loop = TrainingLoop(config)
    loop.run()


if __name__ == "__main__":
    main()


## `tests/`

In [ ]:
%%writefile tests/test_benchmark_dora.py
"""Tests for training.benchmark_dora — compare_results() only (pure logic,
no GPU/unsloth needed; run_benchmark/setup_model require a real GPU session).
"""

from __future__ import annotations

from training.benchmark_dora import BenchmarkResult, compare_results


def _result(method: str, loss_is_real: bool, final_loss: float = 0.3) -> BenchmarkResult:
    return BenchmarkResult(
        method=method,
        steps=50,
        final_loss=final_loss,
        avg_loss=final_loss,
        loss_curve=[final_loss] * 5,
        memory_mb=1000.0,
        steps_per_sec=2.0,
        adapter_size_mb=10.0,
        elapsed_sec=25.0,
        loss_is_real=loss_is_real,
    )


class TestCompareResults:
    def test_unmeasured_loss_gives_no_fabricated_winner(self):
        """Regression: the old fake loss curve gave the same formula to
        both methods, so a loss 'winner' was always reported despite
        being meaningless. Without a real measurement, no winner."""
        dora = _result("QDoRA", loss_is_real=False)
        qlora = _result("QLoRA", loss_is_real=False)
        comparison = compare_results(dora, qlora)

        assert comparison["winner"]["loss"] == "unmeasured"
        assert "INCONCLUSIVE" in comparison["recommendation"]

    def test_real_loss_produces_a_winner(self):
        dora = _result("QDoRA", loss_is_real=True, final_loss=0.2)
        qlora = _result("QLoRA", loss_is_real=True, final_loss=0.4)
        comparison = compare_results(dora, qlora)

        assert comparison["winner"]["loss"] == "dora"
        assert comparison["recommendation"] in ("use_dora", "use_qlora")

    def test_speed_and_memory_always_measured(self):
        """Speed/memory come from real torch/unsloth measurements even
        when loss doesn't — they should never be 'unmeasured'."""
        dora = _result("QDoRA", loss_is_real=False)
        qlora = _result("QLoRA", loss_is_real=False)
        comparison = compare_results(dora, qlora)

        assert comparison["winner"]["speed"] in ("dora", "qlora")
        assert comparison["winner"]["memory"] in ("dora", "qlora")


In [ ]:
%%writefile tests/test_common.py
"""Tests for preprocess/common/ — bbox, stats, sar, gsd modules."""

from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import pytest
from PIL import Image

from preprocess.common.bbox import (
    normalized_to_latlon,
    normalized_to_pixel,
    pixel_to_normalized,
    validate_bbox_ordering,
)
from preprocess.common.gsd import (
    CARTOSAT_PROXY,
    DUAL_RESOLUTION_DATASETS,
    RISAT_PROXY,
    assign_gsd_bucket,
    create_proxy_sample,
    curriculum_mix_ratio,
)
from preprocess.common.sar import (
    clip_vh_db,
    clip_vv_db,
    downsample_sar,
    linear_to_db,
    sar_intensity_pseudo_gray,
    sar_pseudo_rgb,
)
from preprocess.common.stats import GlobalPercentileStats, clip_percentile


# =========================================================================
# bbox.py tests
# =========================================================================


class TestPixelToNormalized:
    def test_origin(self):
        result = pixel_to_normalized((0, 0, 100, 100), 1000, 1000)
        assert result == [[0.0, 0.0, 100.0, 100.0]]

    def test_full_image(self):
        result = pixel_to_normalized((0, 0, 640, 480), 640, 480)
        assert result == [[0.0, 0.0, 1000.0, 1000.0]]

    def test_half_image(self):
        result = pixel_to_normalized((160, 120, 480, 360), 640, 480)
        expected = [[250.0, 250.0, 750.0, 750.0]]
        assert result == expected

    def test_roundtrip(self):
        original = (100, 50, 400, 300)
        norm = pixel_to_normalized(original, 800, 600)
        back = normalized_to_pixel(norm, 800, 600)
        assert back == original


class TestNormalizedToPixel:
    def test_full_extent(self):
        result = normalized_to_pixel([[0, 0, 1000, 1000]], 640, 480)
        assert result == (0, 0, 640, 480)

    def test_center_box(self):
        result = normalized_to_pixel([[250, 250, 750, 750]], 400, 400)
        assert result == (100, 100, 300, 300)


class TestValidateBboxOrdering:
    def test_valid_topleft_first(self):
        assert validate_bbox_ordering([[100, 200, 500, 600]]) is True

    def test_equal_coordinates(self):
        assert validate_bbox_ordering([[100, 100, 100, 100]]) is True

    def test_reversed_x(self):
        assert validate_bbox_ordering([[500, 100, 100, 500]]) is False

    def test_reversed_y(self):
        assert validate_bbox_ordering([[100, 500, 500, 100]]) is False


class TestNormalizedToLatlon:
    def test_identity_transform(self):
        """With identity affine, latlon should equal pixel coords."""
        from rasterio.transform import Affine

        transform = Affine(1, 0, 0, 0, -1, 100)
        bbox = [[0.0, 0.0, 1000.0, 1000.0]]
        result = normalized_to_latlon(bbox, transform, None)
        assert len(result) == 2
        assert len(result[0]) == 2  # [lon, lat]


# =========================================================================
# stats.py tests
# =========================================================================


class TestGlobalPercentileStats:
    def test_single_image(self):
        stats = GlobalPercentileStats(max_samples=1000)
        arr = np.random.rand(64, 64, 3).astype(np.float32) * 100
        stats.accumulate(arr, bands=["R", "G", "B"])

        pcts = stats.compute_percentiles(low=2, high=98)
        assert set(pcts.keys()) == {"R", "G", "B"}
        for band in ["R", "G", "B"]:
            low, high = pcts[band]
            assert low < high
            assert 0 <= low <= 100
            assert 0 <= high <= 100

    def test_multiple_images(self):
        stats = GlobalPercentileStats(max_samples=10000)
        for _ in range(10):
            arr = np.random.rand(32, 32).astype(np.float32) * 50 + 25
            stats.accumulate(arr, bands=["VV"])

        pcts = stats.compute_percentiles(low=10, high=90)
        low, high = pcts["VV"]
        # With uniform-ish data in [25, 75], 10th percentile ~29, 90th ~71
        assert 20 < low < 40
        assert 60 < high < 80

    def test_apply_clip(self):
        stats = GlobalPercentileStats(max_samples=1000)
        arr = np.arange(100, dtype=np.float64).reshape(10, 10)
        stats.accumulate(arr, bands=["B4"])
        pcts = stats.compute_percentiles(low=10, high=90)

        clipped = stats.apply_clip(arr, pcts, bands=["B4"])
        low, high = pcts["B4"]
        assert clipped.min() >= low - 1e-10
        assert clipped.max() <= high + 1e-10

    def test_save_load_roundtrip(self):
        stats = GlobalPercentileStats(max_samples=500)
        arr = np.random.rand(16, 16).astype(np.float32)
        stats.accumulate(arr, bands=["test_band"])

        with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
            tmp_path = f.name
        stats.save(tmp_path)

        loaded = GlobalPercentileStats.load(tmp_path)
        pcts_orig = stats.compute_percentiles(low=5, high=95)
        pcts_loaded = loaded.compute_percentiles(low=5, high=95)

        assert pcts_orig.keys() == pcts_loaded.keys()
        for band in pcts_orig:
            assert abs(pcts_orig[band][0] - pcts_loaded[band][0]) < 1e-6
            assert abs(pcts_orig[band][1] - pcts_loaded[band][1]) < 1e-6

        Path(tmp_path).unlink()


class TestClipPercentile:
    def test_basic_clip(self):
        arr = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
        clipped, low, high = clip_percentile(arr, 10, 90)
        assert low < high
        assert clipped.min() >= low - 1e-10
        assert clipped.max() <= high + 1e-10


# =========================================================================
# sar.py tests
# =========================================================================


class TestLinearToDb:
    def test_basic_conversion(self):
        linear = np.array([1.0, 10.0, 100.0])
        db = linear_to_db(linear)
        np.testing.assert_allclose(db, [0.0, 10.0, 20.0], atol=1e-6)

    def test_zero_handled(self):
        linear = np.array([0.0])
        db = linear_to_db(linear, epsilon=1e-10)
        assert db[0] < -90  # ~-100 dB

    def test_epsilon_prevents_log10_error(self):
        linear = np.array([0.0, 0.0, 0.0])
        db = linear_to_db(linear, epsilon=1e-10)
        assert np.all(np.isfinite(db))

    def test_negative_input_clamped_to_epsilon(self):
        """Negative linear values (from float noise) should become ~-100 dB, not NaN."""
        linear = np.array([-0.5, -1.0, -100.0])
        db = linear_to_db(linear, epsilon=1e-10)
        assert np.all(np.isfinite(db)), f"Got NaN/Inf: {db}"
        # 10*log10(1e-10) = -100 dB — this is the de facto "no signal" floor
        expected_floor = 10.0 * np.log10(1e-10)
        np.testing.assert_allclose(db, expected_floor, atol=1e-6,
            err_msg=f"Expected clamped value ~{expected_floor} dB, got {db}")


class TestSarClipping:
    def test_vv_clip_range(self):
        arr = np.array([-30.0, -25.0, -12.5, 0.0, 5.0])
        clipped = clip_vv_db(arr)
        np.testing.assert_array_equal(clipped, [-25.0, -25.0, -12.5, 0.0, 0.0])

    def test_vh_clip_range(self):
        arr = np.array([-35.0, -30.0, -17.5, -5.0, 0.0])
        clipped = clip_vh_db(arr)
        np.testing.assert_array_equal(clipped, [-30.0, -30.0, -17.5, -5.0, -5.0])


class TestSarPseudoRgb:
    def test_dual_pol_shape(self):
        vv = np.random.rand(64, 64).astype(np.float32) * 0.1
        vh = np.random.rand(64, 64).astype(np.float32) * 0.01
        rgb = sar_pseudo_rgb(vv, vh)
        assert rgb.shape == (64, 64, 3)
        assert rgb.dtype == np.uint8

    def test_dual_pol_range(self):
        vv = np.ones((16, 16), dtype=np.float32) * 0.01  # ~-20 dB
        vh = np.ones((16, 16), dtype=np.float32) * 0.001  # ~-30 dB
        rgb = sar_pseudo_rgb(vv, vh)
        assert rgb.min() >= 0
        assert rgb.max() <= 255

    def test_quad_pol_shape(self):
        hh = np.random.rand(64, 64).astype(np.float32) * 0.1
        vv = np.random.rand(64, 64).astype(np.float32) * 0.1
        vh = np.random.rand(64, 64).astype(np.float32) * 0.01
        rgb = sar_pseudo_rgb(vv, vh, hh=hh)
        assert rgb.shape == (64, 64, 3)
        assert rgb.dtype == np.uint8

    def test_identical_inputs_deterministic(self):
        vv = np.ones((8, 8), dtype=np.float32) * 0.05
        vh = np.ones((8, 8), dtype=np.float32) * 0.005
        rgb1 = sar_pseudo_rgb(vv, vh)
        rgb2 = sar_pseudo_rgb(vv, vh)
        np.testing.assert_array_equal(rgb1, rgb2)


class TestSarIntensityPseudoGray:
    def test_shape_and_dtype(self):
        intensity = np.random.rand(32, 32).astype(np.float32) * 0.1
        gray = sar_intensity_pseudo_gray(intensity)
        assert gray.shape == (32, 32, 3)
        assert gray.dtype == np.uint8

    def test_channels_identical(self):
        """Honest single-pol render: R, G, B must be identical — no
        fabricated per-channel variation."""
        intensity = np.random.rand(16, 16).astype(np.float32) * 0.05
        gray = sar_intensity_pseudo_gray(intensity)
        np.testing.assert_array_equal(gray[:, :, 0], gray[:, :, 1])
        np.testing.assert_array_equal(gray[:, :, 1], gray[:, :, 2])

    def test_deterministic(self):
        intensity = np.ones((8, 8), dtype=np.float32) * 0.02
        g1 = sar_intensity_pseudo_gray(intensity)
        g2 = sar_intensity_pseudo_gray(intensity)
        np.testing.assert_array_equal(g1, g2)


class TestDownsampleSar:
    def test_no_downsample_when_already_coarse(self):
        arr = np.random.rand(64, 64).astype(np.float32)
        result = downsample_sar(arr, target_gsd=2.0, native_gsd=5.0)
        assert result.shape == arr.shape

    def test_downsample_2x(self):
        arr = np.random.rand(128, 128).astype(np.float32)
        result = downsample_sar(arr, target_gsd=4.0, native_gsd=2.0)
        # Scale = 2/4 = 0.5, so 128*0.5 = 64
        assert result.shape[0] == 64
        assert result.shape[1] == 64

    def test_downsample_preserves_dtype(self):
        arr = np.random.rand(64, 64).astype(np.uint8) * 255
        result = downsample_sar(arr, target_gsd=4.0, native_gsd=2.0)
        assert result.dtype == np.uint8

    def test_downsample_3d(self):
        arr = np.random.rand(64, 64, 3).astype(np.float32)
        result = downsample_sar(arr, target_gsd=4.0, native_gsd=2.0)
        assert result.ndim == 3
        assert result.shape[2] == 3


# =========================================================================
# gsd.py tests
# =========================================================================


class TestAssignGsdBucket:
    def test_rsvqa_hr_literal(self):
        assert assign_gsd_bucket("rsvqa_hr") == "[GSD:0.15m]"

    def test_sn6_opt_literal(self):
        assert assign_gsd_bucket("sn6_opt") == "[GSD:0.5m]"

    def test_bigen_literal(self):
        assert assign_gsd_bucket("bigen") == "[GSD:10m]"

    def test_sardet_literal(self):
        assert assign_gsd_bucket("sardet") == "[GSD:5m]"

    def test_vrsbench_native_no_gsd(self):
        assert assign_gsd_bucket("vrsbench") == "VHR-native"

    def test_cdvqa_native_no_gsd(self):
        assert assign_gsd_bucket("cdvqa") == "VHR-native"

    def test_unknown_dataset_with_gsd(self):
        result = assign_gsd_bucket("unknown_ds", native_gsd=3.5)
        assert result == "[GSD:3.50m]"

    def test_unknown_dataset_no_gsd(self):
        assert assign_gsd_bucket("unknown_ds") == "VHR-native"

    def test_integer_gsd_format(self):
        result = assign_gsd_bucket("unknown_ds", native_gsd=10.0)
        assert result == "[GSD:10m]"


class TestCreateProxySample:
    def test_optical_proxy(self):
        sample = {
            "id": "vrsbench_001",
            "dataset": "vrsbench",
            "task": "grounding",
            "image_path": ["img.png"],
            "pair_type": "single",
            "gsd_bucket": "VHR-native",
            "split": "train",
            "instruction": "Locate.",
            "response": "Here.",
            "bbox": [[0, 0, 500, 500]],
            "modality": "optical",
        }
        proxy = create_proxy_sample(sample)
        assert proxy["id"] == "vrsbench_001_proxy"
        assert proxy["gsd_bucket"] == "CARTOSAT-proxy[GSD:2m]"
        # Original unchanged
        assert sample["id"] == "vrsbench_001"
        assert sample["gsd_bucket"] == "VHR-native"

    def test_sar_proxy(self):
        sample = {
            "id": "cdvqa_001",
            "dataset": "cdvqa",
            "task": "change_vqa",
            "image_path": ["a.png", "b.png"],
            "pair_type": "bitemporal",
            "gsd_bucket": "VHR-native",
            "split": "train",
            "instruction": "What changed?",
            "response": "New building.",
            "bbox": None,
            "modality": "sar",
        }
        proxy = create_proxy_sample(sample)
        assert "RISAT-proxy" in proxy["gsd_bucket"]

    def test_explicit_proxy_gsd(self):
        sample = {
            "id": "test_001",
            "modality": "optical",
            "gsd_bucket": "VHR-native",
        }
        proxy = create_proxy_sample(sample, proxy_gsd=1.5)
        assert "1.50m" in proxy["gsd_bucket"]


class TestCurriculumMixRatio:
    def test_stage_1_all_native(self):
        assert curriculum_mix_ratio(1) == (1.0, 0.0)

    def test_stage_2_80_20(self):
        assert curriculum_mix_ratio(2) == (0.8, 0.2)

    def test_stage_3_50_50(self):
        assert curriculum_mix_ratio(3) == (0.5, 0.5)

    def test_stage_4_30_70(self):
        assert curriculum_mix_ratio(4) == (0.3, 0.7)

    def test_stage_5_caps_at_4(self):
        assert curriculum_mix_ratio(5) == (0.3, 0.7)

    def test_stage_0_falls_back_to_1(self):
        assert curriculum_mix_ratio(0) == (1.0, 0.0)

    def test_ratios_sum_to_one(self):
        for stage in range(1, 10):
            native, proxy = curriculum_mix_ratio(stage)
            assert abs(native + proxy - 1.0) < 1e-10


# ---------------------------------------------------------------------------
# R6: unit01_to_qwen
# ---------------------------------------------------------------------------


class TestUnit01ToQwen:
    def test_zero_maps_to_zero(self):
        from preprocess.common.bbox import unit01_to_qwen
        result = unit01_to_qwen([0.0, 0.0, 1.0, 1.0])
        assert result == [[0.0, 0.0, 1000.0, 1000.0]]

    def test_half_maps_to_500(self):
        from preprocess.common.bbox import unit01_to_qwen
        result = unit01_to_qwen([0.25, 0.25, 0.75, 0.75])
        assert result == [[250.0, 250.0, 750.0, 750.0]]

    def test_preserves_ordering(self):
        from preprocess.common.bbox import unit01_to_qwen
        result = unit01_to_qwen([0.1, 0.2, 0.3, 0.4])
        assert result[0][0] < result[0][2]
        assert result[0][1] < result[0][3]


# ---------------------------------------------------------------------------
# R7: concat
# ---------------------------------------------------------------------------


class TestConcatHorizontal:
    def test_same_size_images(self):
        from preprocess.common.concat import concat_horizontal
        left = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        right = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        result = concat_horizontal(left, right)
        assert result.size == (64 + 2 + 64, 64)  # (width, height)
        assert result.mode == "RGB"

    def test_different_sizes_resizes_right(self):
        from preprocess.common.concat import concat_horizontal
        left = np.random.randint(0, 255, (100, 80, 3), dtype=np.uint8)
        right = np.random.randint(0, 255, (50, 40, 3), dtype=np.uint8)
        result = concat_horizontal(left, right)
        assert result.height == 100
        # right (50,40) resized to height 100 → width = 40 * (100/50) = 80
        assert result.width == 80 + 2 + 80

    def test_pil_input(self):
        from preprocess.common.concat import concat_horizontal
        left = Image.fromarray(np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8))
        right = Image.fromarray(np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8))
        result = concat_horizontal(left, right)
        assert result.size == (64 + 2 + 64, 64)

    def test_custom_separator(self):
        from preprocess.common.concat import concat_horizontal
        left = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        right = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        result = concat_horizontal(left, right, separator_width=10)
        assert result.width == 64 + 10 + 64

    def test_target_height(self):
        from preprocess.common.concat import concat_horizontal
        left = np.random.randint(0, 255, (100, 80, 3), dtype=np.uint8)
        right = np.random.randint(0, 255, (200, 160, 3), dtype=np.uint8)
        result = concat_horizontal(left, right, target_height=50)
        assert result.height == 50

    def test_rejects_2d_input(self):
        from preprocess.common.concat import concat_horizontal
        left = np.random.randint(0, 255, (64, 64), dtype=np.uint8)
        right = np.random.randint(0, 255, (64, 64), dtype=np.uint8)
        with pytest.raises(ValueError, match="3-channel"):
            concat_horizontal(left, right)

    def test_deterministic(self):
        from preprocess.common.concat import concat_horizontal
        arr = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        r1 = concat_horizontal(arr, arr)
        r2 = concat_horizontal(arr, arr)
        assert list(r1.getdata()) == list(r2.getdata())


class TestSaveConcat:
    def test_saves_file(self):
        from preprocess.common.concat import save_concat
        with tempfile.TemporaryDirectory() as tmpdir:
            left = Image.fromarray(np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8))
            right = Image.fromarray(np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8))
            left_path = Path(tmpdir) / "left.png"
            right_path = Path(tmpdir) / "right.png"
            out_path = Path(tmpdir) / "out" / "concat.png"
            left.save(str(left_path))
            right.save(str(right_path))
            save_concat(left_path, right_path, out_path)
            assert out_path.exists()
            result = Image.open(str(out_path))
            assert result.size == (64 + 2 + 64, 64)


# =========================================================================
# io.py tests (Issue 9 — write_png compression level)
# =========================================================================


class TestWritePng:
    """Verify write_png() uses compress_level=6 (Issue 9 fix)."""

    def test_compress_level_6_smaller_than_level_0(self, tmp_path):
        """A highly compressible solid-colour image must be smaller at level 6 than level 0."""
        from preprocess.common.io import write_png

        # Solid colour array — maximally compressible
        arr = np.full((256, 256, 3), 128, dtype=np.uint8)

        out_path = tmp_path / "test.png"
        write_png(arr, out_path)
        size_6 = out_path.stat().st_size

        # Write same content at level 0 for reference
        ref_path = tmp_path / "ref_level0.png"
        img = Image.fromarray(arr, mode="RGB")
        img.save(str(ref_path), format="PNG", compress_level=0)
        size_0 = ref_path.stat().st_size

        assert size_6 <= size_0, (
            f"write_png (level 6, {size_6} B) should be <= level 0 ({size_0} B). "
            "Is compress_level still set to 0 in io.py?"
        )

    def test_write_png_produces_valid_image(self, tmp_path):
        """write_png() output must be a valid, readable RGB PNG."""
        from preprocess.common.io import write_png

        arr = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        out_path = tmp_path / "valid.png"
        write_png(arr, out_path)

        assert out_path.exists()
        img = Image.open(str(out_path))
        assert img.mode == "RGB"
        assert img.size == (64, 64)
        np.testing.assert_array_equal(np.array(img), arr)

    def test_write_png_grayscale_expanded_to_rgb(self, tmp_path):
        """2D grayscale arrays must be replicated to 3-channel RGB."""
        from preprocess.common.io import write_png

        arr = np.random.randint(0, 255, (32, 32), dtype=np.uint8)
        out_path = tmp_path / "gray.png"
        write_png(arr, out_path)

        img = Image.open(str(out_path))
        assert img.mode == "RGB"
        loaded = np.array(img)
        # All 3 channels must be identical to the source grayscale
        np.testing.assert_array_equal(loaded[:, :, 0], arr)
        np.testing.assert_array_equal(loaded[:, :, 1], arr)
        np.testing.assert_array_equal(loaded[:, :, 2], arr)



In [ ]:
%%writefile tests/test_merge_and_package.py
"""Tests for preprocess.merge_and_package — merge & validation pipeline."""

from __future__ import annotations

import json
from pathlib import Path

import pytest

from preprocess.merge_and_package import (
    check_id_uniqueness,
    compute_stats,
    dedupe_by_id,
    load_jsonl,
    merge_and_package,
    merge_tiers,
    tar_directory,
    validate_and_filter,
    write_dataset,
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _make_sample(
    dataset: str = "vrsbench",
    task: str = "vqa",
    sample_id: str = "",
    bbox: list | None = None,
    image_path: list | None = None,
) -> dict:
    return {
        "id": sample_id or f"{dataset}_{task}_000001",
        "dataset": dataset,
        "task": task,
        "image_path": image_path or ["dummy.png"],
        "pair_type": "single",
        "gsd_bucket": "VHR-native",
        "split": "train",
        "instruction": "test instruction",
        "response": "test response",
        "bbox": bbox,
        "modality": "optical",
    }


def _write_jsonl(samples: list[dict], path: Path) -> None:
    with open(path, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")


# ---------------------------------------------------------------------------
# load_jsonl
# ---------------------------------------------------------------------------

class TestLoadJsonl:
    def test_loads_all(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(5)]
        _write_jsonl(samples, tmp_path / "test.jsonl")
        loaded = load_jsonl(tmp_path / "test.jsonl")
        assert len(loaded) == 5

    def test_empty_file(self, tmp_path):
        path = tmp_path / "empty.jsonl"
        path.touch()
        assert load_jsonl(path) == []


# ---------------------------------------------------------------------------
# validate_and_filter
# ---------------------------------------------------------------------------

class TestValidateAndFilter:
    def test_valid_sample(self):
        sample = _make_sample()
        valid, errors = validate_and_filter([sample], "test")
        assert len(valid) == 1
        assert errors == []

    def test_invalid_sample(self):
        sample = _make_sample()
        del sample["dataset"]  # Missing required field
        valid, errors = validate_and_filter([sample], "test")
        assert len(valid) == 0
        assert len(errors) == 1
        assert errors[0]["tier"] == "test"


# ---------------------------------------------------------------------------
# check_id_uniqueness
# ---------------------------------------------------------------------------

class TestCheckIdUniqueness:
    def test_no_duplicates(self):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(5)]
        dupes = check_id_uniqueness(samples)
        assert dupes == []

    def test_finds_duplicates(self):
        samples = [
            _make_sample(sample_id="dup"),
            _make_sample(sample_id="dup"),
            _make_sample(sample_id="unique"),
        ]
        dupes = check_id_uniqueness(samples)
        assert len(dupes) == 1
        assert "dup" in dupes[0]


# ---------------------------------------------------------------------------
# dedupe_by_id
# ---------------------------------------------------------------------------

class TestDedupeById:
    def test_keeps_first_drops_rest(self):
        samples = [
            _make_sample(sample_id="dup", dataset="vrsbench"),
            _make_sample(sample_id="dup", dataset="oscd"),
            _make_sample(sample_id="unique"),
        ]
        deduped, n_dropped = dedupe_by_id(samples)
        assert n_dropped == 1
        assert len(deduped) == 2
        assert [s["dataset"] for s in deduped if s["id"] == "dup"] == ["vrsbench"]

    def test_no_duplicates_drops_nothing(self):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(3)]
        deduped, n_dropped = dedupe_by_id(samples)
        assert n_dropped == 0
        assert len(deduped) == 3


class TestMergeAndPackageDedup:
    def test_duplicate_ids_dont_reach_final_dataset(self, tmp_path):
        """Regression: duplicate ids used to be reported but still written
        into dataset.jsonl — merge_and_package() must actually drop them."""
        tier = tmp_path / "tier1"
        tier.mkdir()
        _write_jsonl(
            [_make_sample(sample_id="dup"), _make_sample(sample_id="dup")],
            tier / "data.jsonl",
        )

        output_dir = tmp_path / "out"
        stats = merge_and_package([tier], output_dir)
        assert stats["duplicate_ids"] == 1
        assert stats["total_samples"] == 1

        dataset = load_jsonl(output_dir / "dataset.jsonl")
        assert len(dataset) == 1


# ---------------------------------------------------------------------------
# write_dataset
# ---------------------------------------------------------------------------

class TestWriteDataset:
    def test_writes_jsonl(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(3)]
        out = tmp_path / "out" / "dataset.jsonl"
        write_dataset(samples, out)
        assert out.exists()
        loaded = load_jsonl(out)
        assert len(loaded) == 3

    def test_creates_parent_dirs(self, tmp_path):
        samples = [_make_sample()]
        out = tmp_path / "a" / "b" / "c" / "dataset.jsonl"
        write_dataset(samples, out)
        assert out.exists()


# ---------------------------------------------------------------------------
# merge_tiers
# ---------------------------------------------------------------------------

class TestMergeTiers:
    def test_merges_multiple_tiers(self, tmp_path):
        # Create tier1 with 2 samples
        tier1 = tmp_path / "tier1"
        tier1.mkdir()
        _write_jsonl(
            [_make_sample(dataset="vrsbench", sample_id="v1"),
             _make_sample(dataset="vrsbench", sample_id="v2")],
            tier1 / "vrsbench.jsonl",
        )

        # Create tier2 with 3 samples
        tier2 = tmp_path / "tier2"
        tier2.mkdir()
        _write_jsonl(
            [_make_sample(dataset="oscd", sample_id="o1"),
             _make_sample(dataset="oscd", sample_id="o2"),
             _make_sample(dataset="oscd", sample_id="o3")],
            tier2 / "oscd.jsonl",
        )

        valid, errors = merge_tiers([tier1, tier2])
        assert len(valid) == 5
        assert errors == []

    def test_prefers_split_files_over_original(self, tmp_path):
        """When split_internal_val.py has run, its _train/_val_internal
        files carry the real split tags and must be used INSTEAD of the
        pre-split original — not skipped in favor of it."""
        tier = tmp_path / "tier1"
        tier.mkdir()
        _write_jsonl(
            [_make_sample(sample_id="main")],
            tier / "data.jsonl",
        )
        _write_jsonl(
            [_make_sample(sample_id="train")],
            tier / "data_train.jsonl",
        )
        _write_jsonl(
            [_make_sample(sample_id="val")],
            tier / "data_val_internal.jsonl",
        )

        valid, errors = merge_tiers([tier])
        ids = {s["id"] for s in valid}
        assert ids == {"train", "val"}, \
            f"Expected split files to supersede the original, got {ids}"

    def test_uses_original_when_no_split_output_exists(self, tmp_path):
        tier = tmp_path / "tier1"
        tier.mkdir()
        _write_jsonl(
            [_make_sample(sample_id="main")],
            tier / "data.jsonl",
        )

        valid, errors = merge_tiers([tier])
        assert len(valid) == 1
        assert valid[0]["id"] == "main"

    def test_val_internal_split_tag_survives_merge(self, tmp_path):
        tier = tmp_path / "tier1"
        tier.mkdir()
        _write_jsonl(
            [{**_make_sample(sample_id="train1"), "split": "train"}],
            tier / "data_train.jsonl",
        )
        _write_jsonl(
            [{**_make_sample(sample_id="val1"), "split": "val_internal"}],
            tier / "data_val_internal.jsonl",
        )

        valid, errors = merge_tiers([tier])
        splits = {s["id"]: s["split"] for s in valid}
        assert splits == {"train1": "train", "val1": "val_internal"}

    def test_empty_tier(self, tmp_path):
        tier = tmp_path / "empty_tier"
        tier.mkdir()
        valid, errors = merge_tiers([tier])
        assert valid == []


# ---------------------------------------------------------------------------
# tar_directory
# ---------------------------------------------------------------------------

class TestTarDirectory:
    def test_creates_tarball(self, tmp_path):
        src = tmp_path / "images"
        src.mkdir()
        (src / "a.png").write_bytes(b"\x89PNG")
        (src / "b.png").write_bytes(b"\x89PNG")

        tar_dir = tmp_path / "out"
        count = tar_directory(src, tar_dir, prefix="test")
        assert count == 2
        shards = list(tar_dir.glob("test_shard_*.tar.gz"))
        assert len(shards) == 1

    def test_skips_jsonl(self, tmp_path):
        src = tmp_path / "images"
        src.mkdir()
        (src / "a.png").write_bytes(b"\x89PNG")
        (src / "data.jsonl").write_text("{}")

        tar_dir = tmp_path / "out"
        count = tar_directory(src, tar_dir, prefix="test", include_jsonl=False)
        assert count == 1  # Only PNG

    def test_shards_at_file_cap(self, tmp_path):
        src = tmp_path / "images"
        src.mkdir()
        for i in range(5):
            (src / f"img{i}.png").write_bytes(b"\x89PNG")

        tar_dir = tmp_path / "out"
        count = tar_directory(src, tar_dir, prefix="test", max_files_per_shard=2)
        assert count == 5
        shards = sorted(tar_dir.glob("test_shard_*.tar.gz"))
        assert len(shards) == 3  # 2 + 2 + 1


# ---------------------------------------------------------------------------
# compute_stats
# ---------------------------------------------------------------------------

class TestComputeStats:
    def test_basic_stats(self):
        samples = [
            _make_sample(dataset="vrsbench", task="vqa", bbox=None),
            _make_sample(dataset="vrsbench", task="caption", bbox=None),
            _make_sample(dataset="oscd", task="change_grounding",
                        bbox=[[100, 200, 300, 400]]),
        ]
        stats = compute_stats(samples)
        assert stats["total_samples"] == 3
        assert stats["by_dataset"]["vrsbench"] == 2
        assert stats["by_dataset"]["oscd"] == 1
        assert stats["by_task"]["vqa"] == 1
        assert stats["by_task"]["caption"] == 1
        assert stats["by_task"]["change_grounding"] == 1

    def test_bbox_coverage(self):
        samples = [
            _make_sample(bbox=[[100, 200, 300, 400]]),
            _make_sample(bbox=None),
        ]
        stats = compute_stats(samples)
        assert "1/2" in stats["bbox_coverage"]

    def test_empty_samples(self):
        stats = compute_stats([])
        assert stats["total_samples"] == 0


In [ ]:
%%writefile tests/test_sanity_check.py
"""Tests for preprocess.sanity_check — post-download verification."""

from __future__ import annotations

import json
from pathlib import Path

import pytest

from preprocess.sanity_check import (
    SanityCheckError,
    check_cdvqa,
    check_levir_cd,
    check_oscd,
    check_rsvqa_hr,
    check_sardet,
    check_vrsbench,
    run_sanity_check,
)


class TestCheckOscd:
    def test_valid_structure(self, tmp_path):
        # Create OSCD structure
        images_dir = tmp_path / "images"
        for i in range(24):
            loc = images_dir / f"location_{i:02d}"
            (loc / "imgs_1").mkdir(parents=True)
            (loc / "imgs_2").mkdir(parents=True)

        labels_dir = tmp_path / "labels"
        for i in range(24):
            (labels_dir / f"location_{i:02d}" / "cm").mkdir(parents=True)

        results = check_oscd(tmp_path)
        assert results["status"] == "pass"

    def test_missing_images_dir(self, tmp_path):
        with pytest.raises(SanityCheckError):
            check_oscd(tmp_path)

    def test_all_pairs_incomplete_fails(self, tmp_path):
        """Locations exist but none have both imgs_1 and imgs_2 — a
        truncated/half-downloaded mirror must fail, not silently pass."""
        images_dir = tmp_path / "images"
        for i in range(5):
            (images_dir / f"location_{i:02d}" / "imgs_1").mkdir(parents=True)
            # imgs_2 deliberately missing for every location

        with pytest.raises(SanityCheckError, match="No complete OSCD pairs"):
            check_oscd(tmp_path)


class TestCheckVrsbench:
    def test_valid_structure(self, tmp_path):
        # Create VRSBench structure
        data = [
            {"conversations": [{"value": "<image>\n[caption] Describe this"}]},
            {"conversations": [{"value": "<image>\n[refer] Where is <b>?</b>"}]},
            {"conversations": [{"value": "<image>\n[vqa] What is this?"}]},
        ]
        (tmp_path / "VRSBench_train.json").write_text(json.dumps(data))
        (tmp_path / "Images_train.zip").touch()

        results = check_vrsbench(tmp_path)
        assert results["status"] == "pass"

    def test_missing_json(self, tmp_path):
        with pytest.raises(SanityCheckError):
            check_vrsbench(tmp_path)


class TestCheckCdvqa:
    def test_valid_structure(self, tmp_path):
        train_data = [{"question": "Q", "answer": "A"}] * 1600
        val_data = [{"question": "Q", "answer": "A"}] * 400

        (tmp_path / "train.json").write_text(json.dumps(train_data))
        (tmp_path / "val.json").write_text(json.dumps(val_data))

        results = check_cdvqa(tmp_path)
        assert results["status"] == "pass"

    def test_rejects_test_split(self, tmp_path):
        # A test.json alongside a real train/val should warn but not fail —
        # tier1_cdvqa.py itself refuses to load test.json regardless.
        (tmp_path / "train.json").write_text(json.dumps([]))
        (tmp_path / "test.json").write_text(json.dumps([]))

        results = check_cdvqa(tmp_path)
        assert results["status"] == "pass"

    def test_no_train_or_val_fails(self, tmp_path):
        (tmp_path / "test.json").write_text(json.dumps([]))
        with pytest.raises(SanityCheckError):
            check_cdvqa(tmp_path)


class TestCheckLevirCd:
    def test_valid_structure(self, tmp_path):
        for split in ["train", "val", "test"]:
            (tmp_path / split / "A").mkdir(parents=True)
            (tmp_path / split / "B").mkdir(parents=True)
            (tmp_path / split / "label").mkdir(parents=True)

            # Add some files
            for i in range(10):
                (tmp_path / split / "A" / f"img{i:03d}_1.png").touch()
                (tmp_path / split / "B" / f"img{i:03d}_2.png").touch()
                (tmp_path / split / "label" / f"img{i:03d}.png").touch()

        results = check_levir_cd(tmp_path)
        assert results["status"] == "pass"


class TestCheckSardet:
    def test_valid_structure(self, tmp_path):
        images_dir = tmp_path / "images"
        labels_dir = tmp_path / "labels"
        images_dir.mkdir()
        labels_dir.mkdir()

        for i in range(50):
            (images_dir / f"img{i:04d}.png").touch()
            (labels_dir / f"img{i:04d}.txt").touch()

        results = check_sardet(tmp_path)
        assert results["status"] == "pass"

    def test_empty_images_dir_fails(self, tmp_path):
        (tmp_path / "images").mkdir()
        (tmp_path / "labels").mkdir()
        with pytest.raises(SanityCheckError):
            check_sardet(tmp_path)


class TestCheckRsvqaHr:
    def test_valid_structure(self, tmp_path):
        train_data = [{"question_id": i, "question": "Q", "answer": "A"} for i in range(100)]
        (tmp_path / "train.json").write_text(json.dumps(train_data))

        images_dir = tmp_path / "images"
        images_dir.mkdir()
        for i in range(100):
            (images_dir / f"img{i:04d}.png").touch()

        results = check_rsvqa_hr(tmp_path)
        assert results["status"] == "pass"


class TestRunSanityCheck:
    def test_valid_dataset(self, tmp_path):
        # Create minimal OSCD structure
        images_dir = tmp_path / "images"
        for i in range(24):
            loc = images_dir / f"loc_{i:02d}"
            (loc / "imgs_1").mkdir(parents=True)
            (loc / "imgs_2").mkdir(parents=True)

        results = run_sanity_check("oscd", tmp_path)
        assert results["status"] == "pass"

    def test_unknown_dataset(self):
        with pytest.raises(ValueError, match="Unknown dataset"):
            run_sanity_check("nonexistent", Path("/tmp"))


In [ ]:
%%writefile tests/test_split_internal_val.py
"""Tests for preprocess.split_internal_val — R4 internal validation split."""

from __future__ import annotations

import json
import random
from pathlib import Path

import pytest

from preprocess.split_internal_val import (
    load_samples,
    split_and_write,
    split_all_tiers,
    stratified_split,
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _make_sample(
    dataset: str = "vrsbench",
    task: str = "vqa",
    sample_id: str = "",
) -> dict:
    return {
        "id": sample_id or f"{dataset}_{task}_{random.randint(0, 999999):06d}",
        "dataset": dataset,
        "task": task,
        "image_path": ["dummy.png"],
        "pair_type": "single",
        "gsd_bucket": "VHR-native",
        "split": "train",
        "instruction": "test instruction",
        "response": "test response",
        "bbox": None,
        "modality": "optical",
    }


def _write_jsonl(samples: list[dict], path: Path) -> None:
    with open(path, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")


# ---------------------------------------------------------------------------
# load_samples
# ---------------------------------------------------------------------------

class TestLoadSamples:
    def test_loads_all_samples(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(10)]
        path = tmp_path / "test.jsonl"
        _write_jsonl(samples, path)
        loaded = load_samples(path)
        assert len(loaded) == 10

    def test_empty_file(self, tmp_path):
        path = tmp_path / "empty.jsonl"
        path.touch()
        loaded = load_samples(path)
        assert loaded == []


# ---------------------------------------------------------------------------
# stratified_split
# ---------------------------------------------------------------------------

class TestStratifiedSplit:
    def test_total_preserved(self):
        samples = [_make_sample(task="vqa") for _ in range(100)]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        assert len(train) + len(val) == 100

    def test_val_fraction_approximate(self):
        samples = [_make_sample(task="vqa") for _ in range(1000)]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        ratio = len(val) / len(samples)
        assert 0.02 <= ratio <= 0.05, f"Val ratio {ratio:.3f} outside [0.02, 0.05]"

    def test_stratified_by_task(self):
        samples = (
            [_make_sample(task="vqa") for _ in range(100)] +
            [_make_sample(task="caption") for _ in range(100)] +
            [_make_sample(task="grounding") for _ in range(100)]
        )
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)

        # Check each task has some val samples
        val_tasks = [s["task"] for s in val]
        for task in ["vqa", "caption", "grounding"]:
            assert task in val_tasks, f"Task {task} missing from val split"

    def test_stratified_by_dataset(self):
        samples = (
            [_make_sample(dataset="vrsbench") for _ in range(50)] +
            [_make_sample(dataset="oscd") for _ in range(50)]
        )
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)

        val_datasets = [s["dataset"] for s in val]
        assert "vrsbench" in val_datasets
        assert "oscd" in val_datasets

    def test_deterministic_with_seed(self):
        samples = [_make_sample() for _ in range(100)]
        t1, v1 = stratified_split(samples, val_fraction=0.03, seed=42)
        t2, v2 = stratified_split(samples, val_fraction=0.03, seed=42)
        assert [s["id"] for s in v1] == [s["id"] for s in v2]

    def test_small_group_gets_minimum(self):
        # Group with only 2 samples → val gets 1, train gets 1
        samples = [_make_sample(task="rare_task", sample_id=f"rare_{i}") for i in range(2)]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        assert len(train) + len(val) == 2
        # With val_fraction=0.03, n_val = max(1, int(2*0.03))=1
        # Cap at len-1=1, so val=1, train=1
        assert len(val) == 1
        assert len(train) == 1

    def test_single_sample_group_skipped(self):
        # Group with 1 sample → train=0, val=0 (can't split)
        samples = [_make_sample(task="single_task", sample_id="only_one")]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        # n_val = min(1, 0) = 0, so everything goes to train
        assert len(train) == 1
        assert len(val) == 0


# ---------------------------------------------------------------------------
# split_and_write
# ---------------------------------------------------------------------------

class TestSplitAndWrite:
    def test_creates_two_files(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(50)]
        input_path = tmp_path / "input.jsonl"
        _write_jsonl(samples, input_path)

        stats = split_and_write(input_path, tmp_path / "output", val_fraction=0.03)

        assert stats["total"] == 50
        assert stats["train"] + stats["val"] == 50
        assert (tmp_path / "output" / "input_train.jsonl").exists()
        assert (tmp_path / "output" / "input_val_internal.jsonl").exists()

    def test_val_samples_tagged(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(50)]
        input_path = tmp_path / "input.jsonl"
        _write_jsonl(samples, input_path)

        split_and_write(input_path, tmp_path / "output", val_fraction=0.03)

        val_path = tmp_path / "output" / "input_val_internal.jsonl"
        val_samples = load_samples(val_path)
        for s in val_samples:
            assert s["split"] == "val_internal"

    def test_train_samples_keep_split(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(50)]
        input_path = tmp_path / "input.jsonl"
        _write_jsonl(samples, input_path)

        split_and_write(input_path, tmp_path / "output", val_fraction=0.03)

        train_path = tmp_path / "output" / "input_train.jsonl"
        train_samples = load_samples(train_path)
        for s in train_samples:
            assert s["split"] == "train"

    def test_empty_input(self, tmp_path):
        input_path = tmp_path / "empty.jsonl"
        input_path.touch()
        stats = split_and_write(input_path, tmp_path / "output")
        assert stats == {"total": 0, "train": 0, "val": 0}


# ---------------------------------------------------------------------------
# split_all_tiers
# ---------------------------------------------------------------------------

class TestSplitAllTiers:
    def test_processes_multiple_jsonls(self, tmp_path):
        for name in ["tier1_a.jsonl", "tier1_b.jsonl"]:
            samples = [_make_sample(sample_id=f"s{i}") for i in range(30)]
            _write_jsonl(samples, tmp_path / name)

        results = split_all_tiers(tmp_path)
        assert len(results) == 2
        for name, stats in results.items():
            assert stats["total"] == 30
            assert stats["train"] + stats["val"] == 30

    def test_skips_already_split(self, tmp_path):
        samples = [_make_sample(sample_id=f"s{i}") for i in range(30)]
        _write_jsonl(samples, tmp_path / "data.jsonl")
        _write_jsonl(samples, tmp_path / "data_train.jsonl")
        _write_jsonl(samples, tmp_path / "data_val_internal.jsonl")

        results = split_all_tiers(tmp_path)
        assert len(results) == 1  # Only data.jsonl processed

    def test_no_jsonl_files(self, tmp_path):
        results = split_all_tiers(tmp_path)
        assert results == {}


# ---------------------------------------------------------------------------
# Stratification correctness
# ---------------------------------------------------------------------------

class TestStratificationCorrectness:
    def test_no_leakage(self):
        """No sample appears in both train and val."""
        samples = [_make_sample(sample_id=f"s{i}") for i in range(100)]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        train_ids = {s["id"] for s in train}
        val_ids = {s["id"] for s in val}
        assert train_ids.isdisjoint(val_ids)

    def test_all_samples_accounted_for(self):
        """Every sample ends up in exactly one split."""
        samples = [_make_sample(sample_id=f"s{i}") for i in range(100)]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        all_ids = {s["id"] for s in samples}
        result_ids = {s["id"] for s in train} | {s["id"] for s in val}
        assert all_ids == result_ids

    def test_no_split_field_on_train(self):
        """Train samples don't have split modified."""
        samples = [_make_sample(sample_id=f"s{i}") for i in range(100)]
        train, val = stratified_split(samples, val_fraction=0.03, seed=42)
        for s in train:
            assert s["split"] == "train"


In [ ]:
%%writefile tests/test_tier0_bigen.py
"""Tests for preprocess.tier0_bigen — BigEarthNet preprocessing."""

from __future__ import annotations

import json
import tempfile
from pathlib import Path

import numpy as np
import pytest

from preprocess.tier0_bigen import (
    load_bigen_arrays,
    parse_bigen_sample,
    run_tier0_bigen,
    s2_to_rgb_uint8,
)


class TestParseBigENetSample:
    def test_valid_sample(self):
        import numpy as np
        sample = {
            "image": np.random.randint(0, 1000, (12, 64, 64), dtype=np.uint16),
            "label": np.zeros(19, dtype=np.int32),
        }
        sample["label"][0] = 1  # Continuous urban fabric
        sample["label"][11] = 1  # Non-irrigated arable land

        result = parse_bigen_sample(sample, idx=0)
        assert result is not None
        assert result["id"] == "bigen_00000000"
        assert result["dataset"] == "bigen"
        assert result["task"] == "vqa"
        assert result["modality"] == "optical"
        assert "Continuous urban fabric" in result["response"]
        assert "Non-irrigated arable land" in result["response"]

    def test_missing_image(self):
        import numpy as np
        sample = {"label": np.ones(19, dtype=np.int32)}
        result = parse_bigen_sample(sample, idx=0)
        assert result is None

    def test_missing_label(self):
        import numpy as np
        sample = {"image": np.random.randint(0, 1000, (12, 64, 64), dtype=np.uint16)}
        result = parse_bigen_sample(sample, idx=0)
        assert result is None

    def test_empty_labels_skipped(self):
        import numpy as np
        sample = {
            "image": np.random.randint(0, 1000, (12, 64, 64), dtype=np.uint16),
            "label": np.zeros(19, dtype=np.int32),
        }
        result = parse_bigen_sample(sample, idx=0)
        assert result is None

    def test_rgb_extraction(self):
        import numpy as np
        # Create image with known values
        image = np.zeros((12, 32, 32), dtype=np.uint16)
        image[3] = 100  # B04 (R)
        image[2] = 200  # B03 (G)
        image[1] = 300  # B02 (B)

        sample = {
            "image": image,
            "label": np.array([1] + [0]*18, dtype=np.int32),
        }
        result = parse_bigen_sample(sample, idx=5)
        assert result is not None
        assert result["id"] == "bigen_00000005"

    def test_sample_id_format(self):
        import numpy as np
        sample = {
            "image": np.random.randint(0, 1000, (12, 64, 64), dtype=np.uint16),
            "label": np.array([1] + [0]*18, dtype=np.int32),
        }
        result = parse_bigen_sample(sample, idx=12345)
        assert result["id"] == "bigen_00012345"


class TestBigENetSchema:
    def test_sample_has_all_fields(self):
        import numpy as np
        sample = {
            "image": np.random.randint(0, 1000, (12, 64, 64), dtype=np.uint16),
            "label": np.array([1] + [0]*18, dtype=np.int32),
        }
        result = parse_bigen_sample(sample, idx=0)
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in result, f"Missing field: {field}"

    def test_dataset_enum(self):
        import numpy as np
        sample = {
            "image": np.random.randint(0, 1000, (12, 64, 64), dtype=np.uint16),
            "label": np.array([1] + [0]*18, dtype=np.int32),
        }
        result = parse_bigen_sample(sample, idx=0)
        assert result["dataset"] == "bigen"


# ---------------------------------------------------------------------------
# load_bigen_arrays / run_tier0_bigen — the actual wired pipeline
# ---------------------------------------------------------------------------


def _make_bigen_input(root: Path, n_patches: int = 3, with_sar: bool = False) -> None:
    """Build a synthetic Kaggle-style BigEarthNet input dir."""
    import pandas as pd

    (root / "s2_npy").mkdir(parents=True, exist_ok=True)
    if with_sar:
        (root / "s1_npy").mkdir(parents=True, exist_ok=True)

    rows = []
    for i in range(n_patches):
        patch_id = f"patch_{i:04d}"
        s2 = np.random.randint(100, 8000, size=(12, 32, 32), dtype=np.uint16)
        np.save(root / "s2_npy" / f"{patch_id}.npy", s2)

        if with_sar:
            s1 = (np.random.rand(2, 32, 32).astype(np.float32) * 0.1)
            np.save(root / "s1_npy" / f"{patch_id}.npy", s1)

        rows.append({"patch_id": patch_id, "labels": [0, 11]})

    pd.DataFrame(rows).to_parquet(root / "metadata.parquet")


class TestLoadBigenArrays:
    def test_loads_s2_only(self, tmp_path):
        _make_bigen_input(tmp_path, n_patches=1, with_sar=False)
        s2, s1 = load_bigen_arrays(tmp_path, {"patch_id": "patch_0000"})
        assert s2.shape == (12, 32, 32)
        assert s1 is None

    def test_loads_s1_when_present(self, tmp_path):
        _make_bigen_input(tmp_path, n_patches=1, with_sar=True)
        s2, s1 = load_bigen_arrays(tmp_path, {"patch_id": "patch_0000"})
        assert s1 is not None
        assert s1.shape == (2, 32, 32)

    def test_missing_patch_raises(self, tmp_path):
        _make_bigen_input(tmp_path, n_patches=1, with_sar=False)
        with pytest.raises(FileNotFoundError):
            load_bigen_arrays(tmp_path, {"patch_id": "does_not_exist"})

    def test_missing_patch_id_raises(self, tmp_path):
        with pytest.raises(ValueError):
            load_bigen_arrays(tmp_path, {})


class TestS2ToRgbUint8:
    def test_shape_and_dtype(self):
        image = np.random.randint(0, 8000, size=(12, 16, 16), dtype=np.uint16)
        rgb = s2_to_rgb_uint8(image)
        assert rgb.shape == (16, 16, 3)
        assert rgb.dtype == np.uint8


class TestRunTier0Bigen:
    def test_optical_only_pipeline_produces_valid_samples(self, tmp_path):
        input_dir = tmp_path / "input"
        input_dir.mkdir()
        _make_bigen_input(input_dir, n_patches=3, with_sar=False)

        output_dir = tmp_path / "output"
        stats = run_tier0_bigen(input_dir, output_dir, max_samples=10)

        assert stats["processed"] == 3
        assert stats["failed"] == 0

        jsonl_path = output_dir / "bigen.jsonl"
        assert jsonl_path.exists()
        with open(jsonl_path) as f:
            lines = [json.loads(l) for l in f if l.strip()]

        assert len(lines) == 3  # 1 row/patch when no SAR
        for row in lines:
            assert row["modality"] == "optical"
            assert Path(row["image_path"][0]).exists()

            from preprocess.validator import validate_sample
            ok, errs = validate_sample(row)
            assert ok, f"Invalid sample {row['id']}: {errs}"

    def test_sar_and_fusion_products_emitted(self, tmp_path):
        input_dir = tmp_path / "input"
        input_dir.mkdir()
        _make_bigen_input(input_dir, n_patches=2, with_sar=True)

        output_dir = tmp_path / "output"
        stats = run_tier0_bigen(input_dir, output_dir, max_samples=10)

        assert stats["processed"] == 2

        with open(output_dir / "bigen.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]

        # 3 rows/patch when SAR is present: optical, sar, fusion
        assert len(lines) == 6

        modalities = {l["modality"] for l in lines}
        assert modalities == {"optical", "sar", "optical+sar"}

        pair_types = {l["pair_type"] for l in lines}
        assert "cross-modal" in pair_types

        fusion_rows = [l for l in lines if l["pair_type"] == "cross-modal"]
        for row in fusion_rows:
            assert row["task"] == "fusion_vqa"
            assert len(row["image_path"]) == 2
            for p in row["image_path"]:
                assert Path(p).exists()

            from preprocess.validator import validate_sample
            ok, errs = validate_sample(row)
            assert ok, f"Invalid fusion sample {row['id']}: {errs}"

    def test_second_run_skips_all(self, tmp_path):
        input_dir = tmp_path / "input"
        input_dir.mkdir()
        _make_bigen_input(input_dir, n_patches=2, with_sar=False)

        output_dir = tmp_path / "output"
        stats1 = run_tier0_bigen(input_dir, output_dir, max_samples=10)
        assert stats1["processed"] == 2

        stats2 = run_tier0_bigen(input_dir, output_dir, max_samples=10)
        assert stats2["processed"] == 0
        assert stats2["skipped"] == 2

    def test_missing_metadata_returns_zero(self, tmp_path):
        input_dir = tmp_path / "empty_input"
        input_dir.mkdir()
        output_dir = tmp_path / "output"
        stats = run_tier0_bigen(input_dir, output_dir, max_samples=10)
        assert stats == {"processed": 0, "skipped": 0, "failed": 0}

    def test_missing_npy_patch_counted_as_failed(self, tmp_path):
        import pandas as pd

        input_dir = tmp_path / "input"
        input_dir.mkdir()
        (input_dir / "s2_npy").mkdir()
        # metadata references a patch with no corresponding .npy file
        pd.DataFrame([{"patch_id": "ghost", "labels": [0]}]).to_parquet(
            input_dir / "metadata.parquet"
        )

        output_dir = tmp_path / "output"
        stats = run_tier0_bigen(input_dir, output_dir, max_samples=10)
        assert stats["failed"] == 1
        assert stats["processed"] == 0


In [ ]:
%%writefile tests/test_tier1_cdvqa.py
"""Tests for preprocess.tier1_cdvqa — CDVQA preprocessing."""

from __future__ import annotations

import json
from pathlib import Path

import pytest

from preprocess.tier1_cdvqa import load_split, parse_cdvqa_sample


class TestLoadSplit:
    def test_loads_train(self, tmp_path):
        data = [{"question": "What changed?", "answer": "building"}]
        (tmp_path / "train.json").write_text(json.dumps(data))
        result = load_split(tmp_path, "train")
        assert len(result) == 1

    def test_loads_val(self, tmp_path):
        data = [{"question": "What changed?", "answer": "road"}]
        (tmp_path / "val.json").write_text(json.dumps(data))
        result = load_split(tmp_path, "val")
        assert len(result) == 1

    def test_rejects_test_split(self, tmp_path):
        with pytest.raises(ValueError, match="Invalid split"):
            load_split(tmp_path, "test")

    def test_empty_when_missing(self, tmp_path):
        result = load_split(tmp_path, "train")
        assert result == []

    def test_wrapped_format(self, tmp_path):
        data = {"questions": [{"question": "Q", "answer": "A"}]}
        (tmp_path / "train.json").write_text(json.dumps(data))
        result = load_split(tmp_path, "train")
        assert len(result) == 1


class TestParseCdvqaSample:
    def test_valid_sample(self, tmp_path):
        entry = {
            "question": "What changed?",
            "answer": "new building",
            "before": "before.png",
            "after": "after.png",
        }
        result = parse_cdvqa_sample(entry, tmp_path, "cdvqa_0")
        assert result is not None
        assert result["id"] == "cdvqa_0"
        assert result["dataset"] == "cdvqa"
        assert result["task"] == "change_vqa"
        assert result["pair_type"] == "bitemporal"

    def test_empty_question_skipped(self):
        entry = {"question": "", "answer": "A", "before": "b.png", "after": "a.png"}
        result = parse_cdvqa_sample(entry, None, "cdvqa_0")
        assert result is None

    def test_missing_before_skipped(self):
        entry = {"question": "Q", "answer": "A", "before": "", "after": "a.png"}
        result = parse_cdvqa_sample(entry, None, "cdvqa_0")
        assert result is None

    def test_image_paths_resolved(self, tmp_path):
        (tmp_path / "before.png").write_bytes(b"\x89PNG")
        (tmp_path / "after.png").write_bytes(b"\x89PNG")
        entry = {
            "question": "Q",
            "answer": "A",
            "before": "before.png",
            "after": "after.png",
        }
        result = parse_cdvqa_sample(entry, tmp_path, "cdvqa_0")
        assert len(result["image_path"]) == 2
        assert "before.png" in result["image_path"][0]
        assert "after.png" in result["image_path"][1]

    def test_has_all_fields(self):
        entry = {"question": "Q", "answer": "A", "before": "b.png", "after": "a.png"}
        result = parse_cdvqa_sample(entry, None, "cdvqa_0")
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in result, f"Missing field: {field}"

    def test_modality_optical(self):
        entry = {"question": "Q", "answer": "A", "before": "b.png", "after": "a.png"}
        result = parse_cdvqa_sample(entry, None, "cdvqa_0")
        assert result["modality"] == "optical"


In [ ]:
%%writefile tests/test_tier1_levir_cd.py
"""Tests for preprocess.tier1_levir_cd — LEVIR-CD preprocessing."""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pytest
from PIL import Image

from preprocess.tier1_levir_cd import (
    change_fraction,
    find_levir_pairs,
    mask_to_bbox,
    process_pair,
    run_tier1_levir_cd,
)


class TestFindLevirPairs:
    def test_discovers_pairs(self, tmp_path):
        # Create train/A, train/B, train/label structure
        split_dir = tmp_path / "train"
        (split_dir / "A").mkdir(parents=True)
        (split_dir / "B").mkdir(parents=True)
        (split_dir / "label").mkdir(parents=True)

        (split_dir / "A" / "img001_1.png").write_bytes(b"\x89PNG")
        (split_dir / "B" / "img001_2.png").write_bytes(b"\x89PNG")
        (split_dir / "label" / "img001.png").write_bytes(b"\x89PNG")

        pairs = find_levir_pairs(split_dir)
        assert len(pairs) == 1
        assert pairs[0]["base_name"] == "img001"

    def test_missing_after_skipped(self, tmp_path):
        split_dir = tmp_path / "train"
        (split_dir / "A").mkdir(parents=True)
        (split_dir / "B").mkdir(parents=True)

        (split_dir / "A" / "img001_1.png").write_bytes(b"\x89PNG")
        # No corresponding _2.png in B

        pairs = find_levir_pairs(split_dir)
        assert len(pairs) == 0

    def test_empty_dir(self, tmp_path):
        split_dir = tmp_path / "train"
        split_dir.mkdir()
        pairs = find_levir_pairs(split_dir)
        assert pairs == []


class TestMaskToBbox:
    def test_finds_change_region(self, tmp_path):
        import numpy as np
        from PIL import Image

        # Create mask with change in center
        mask = np.zeros((64, 64), dtype=np.uint8)
        mask[20:40, 20:40] = 255

        mask_path = tmp_path / "mask.png"
        Image.fromarray(mask).save(str(mask_path))

        bbox = mask_to_bbox(mask_path, (64, 64))
        assert bbox is not None
        assert len(bbox) == 1
        assert len(bbox[0]) == 4

    def test_empty_mask_returns_none(self, tmp_path):
        import numpy as np
        from PIL import Image

        mask = np.zeros((64, 64), dtype=np.uint8)
        mask_path = tmp_path / "mask.png"
        Image.fromarray(mask).save(str(mask_path))

        bbox = mask_to_bbox(mask_path, (64, 64))
        assert bbox is None


class TestProcessPair:
    def test_returns_valid_sample(self, tmp_path):
        from PIL import Image
        import numpy as np

        # Create images
        img = Image.fromarray(np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8))
        img.save(str(tmp_path / "before.png"))
        img.save(str(tmp_path / "after.png"))

        # Create mask
        mask = np.zeros((64, 64), dtype=np.uint8)
        mask[20:40, 20:40] = 255
        Image.fromarray(mask).save(str(tmp_path / "mask.png"))

        pair = {
            "base_name": "test001",
            "before": tmp_path / "before.png",
            "after": tmp_path / "after.png",
            "mask": tmp_path / "mask.png",
        }
        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_pair(pair, output_dir)
        assert sample is not None
        assert sample["id"] == "levir_test001"
        assert sample["dataset"] == "levir_cd"
        assert sample["pair_type"] == "bitemporal"
        assert sample["bbox"] is not None


class TestChangeFraction:
    def test_computes_fraction(self, tmp_path):
        mask = np.zeros((64, 64), dtype=np.uint8)
        mask[0:32, :] = 255  # half changed
        mask_path = tmp_path / "mask.png"
        Image.fromarray(mask).save(str(mask_path))
        assert abs(change_fraction(mask_path) - 0.5) < 0.01

    def test_none_path_returns_zero(self):
        assert change_fraction(None) == 0.0


def _make_levir_input(root: Path, n_low: int, n_high: int) -> None:
    """n_low pairs with ~1% change, n_high pairs with ~80% change."""
    train_dir = root / "train"
    (train_dir / "A").mkdir(parents=True)
    (train_dir / "B").mkdir(parents=True)
    (train_dir / "label").mkdir(parents=True)

    idx = 0
    for _ in range(n_low):
        name = f"low{idx:03d}"
        Image.fromarray(np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)).save(
            str(train_dir / "A" / f"{name}_1.png"))
        Image.fromarray(np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)).save(
            str(train_dir / "B" / f"{name}_2.png"))
        mask = np.zeros((32, 32), dtype=np.uint8)
        mask[0:2, 0:2] = 255  # ~1.5% change
        Image.fromarray(mask).save(str(train_dir / "label" / f"{name}.png"))
        idx += 1

    for _ in range(n_high):
        name = f"high{idx:03d}"
        Image.fromarray(np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)).save(
            str(train_dir / "A" / f"{name}_1.png"))
        Image.fromarray(np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)).save(
            str(train_dir / "B" / f"{name}_2.png"))
        mask = np.full((32, 32), 255, dtype=np.uint8)  # ~100% change
        Image.fromarray(mask).save(str(train_dir / "label" / f"{name}.png"))
        idx += 1


class TestRunTier1LevirCd:
    def test_r4_dual_resolution_doubles_rows(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_levir_input(input_dir, n_low=5, n_high=5)

        output_dir = tmp_path / "output"
        stats = run_tier1_levir_cd(input_dir, output_dir, sample_fraction=1.0)
        assert stats["processed"] == 10

        with open(output_dir / "levir_cd.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        assert len(lines) == 20  # native + proxy
        assert any("CARTOSAT-proxy" in l["gsd_bucket"] for l in lines)

    def test_stratified_sampling_keeps_both_magnitudes(self, tmp_path):
        """A low sample_fraction must not wipe out the low-change stratum —
        that's exactly what plain random.sample over a high-change-majority
        pool would risk."""
        input_dir = tmp_path / "input"
        _make_levir_input(input_dir, n_low=20, n_high=2)

        output_dir = tmp_path / "output"
        run_tier1_levir_cd(input_dir, output_dir, sample_fraction=0.2, seed=1)

        with open(output_dir / "levir_cd.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]

        names = {Path(l["image_path"][0]).stem.replace("levir_", "").replace("_before", "")
                 for l in lines}
        assert any(n.startswith("high") for n in names), \
            "High-change stratum (2 items) was dropped by subsampling"

    def test_all_rows_validate(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_levir_input(input_dir, n_low=3, n_high=2)
        output_dir = tmp_path / "output"
        run_tier1_levir_cd(input_dir, output_dir, sample_fraction=1.0)

        from preprocess.validator import validate_sample
        with open(output_dir / "levir_cd.jsonl") as f:
            for line in f:
                if not line.strip():
                    continue
                ok, errs = validate_sample(json.loads(line))
                assert ok, errs


class TestLevirSchema:
    def test_has_all_fields(self, tmp_path):
        from PIL import Image
        import numpy as np

        img = Image.fromarray(np.zeros((64, 64, 3), dtype=np.uint8))
        img.save(str(tmp_path / "before.png"))
        img.save(str(tmp_path / "after.png"))

        pair = {
            "base_name": "test",
            "before": tmp_path / "before.png",
            "after": tmp_path / "after.png",
            "mask": None,
        }
        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_pair(pair, output_dir)
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in sample, f"Missing field: {field}"


In [ ]:
%%writefile tests/test_tier1_oscd.py
"""Tests for preprocess/tier1_oscd.py — manifest/resumability pattern.

Creates dummy OSCD data (individual band TIFs + masks) and verifies:
    1. First run processes all pairs and writes JSONL + PNGs
    2. Second run skips already-processed pairs (idempotent)
    3. Interrupted run can resume from manifest
    4. Output PNGs are byte-identical across runs (deterministic)
"""

from __future__ import annotations

import hashlib
import json
import tempfile
from pathlib import Path

import numpy as np
import pytest
import tifffile

from preprocess.common.io import Manifest
from preprocess.tier1_oscd import (
    find_oscd_pairs,
    load_bands_as_rgb,
    load_mask,
    make_sample_id,
    mask_to_bboxes,
    process_pair,
    run_tier1_oscd,
)

# Bands needed for RGB
_BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]


# ---------------------------------------------------------------------------
# Fixtures: create synthetic OSCD directory structure (real layout)
# ---------------------------------------------------------------------------


def _create_dummy_bands(bands_dir: Path) -> None:
    """Create individual band TIFs in a directory."""
    bands_dir.mkdir(parents=True, exist_ok=True)
    for band in _BANDS:
        arr = np.random.randint(100, 8000, size=(64, 64), dtype=np.uint16)
        tifffile.imwrite(str(bands_dir / f"dummy_{band}.tif"), arr)


def _create_dummy_mask(path: Path, change_fraction: float = 0.1) -> None:
    """Create a synthetic binary change mask."""
    path.parent.mkdir(parents=True, exist_ok=True)
    mask = np.zeros((64, 64), dtype=np.uint8)
    if change_fraction > 0:
        mask[16:48, 16:48] = 255  # Change region
    tifffile.imwrite(str(path), mask)


@pytest.fixture
def oscd_dir():
    """Create a temporary OSCD directory with 2 dummy pairs (real layout)."""
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)

        # location_a: has mask
        _create_dummy_bands(root / "images" / "location_a" / "imgs_1")
        _create_dummy_bands(root / "images" / "location_a" / "imgs_2")
        _create_dummy_mask(root / "labels" / "location_a" / "cm" / "location_a-cm.tif")

        # location_b: has mask
        _create_dummy_bands(root / "images" / "location_b" / "imgs_1")
        _create_dummy_bands(root / "images" / "location_b" / "imgs_2")
        _create_dummy_mask(root / "labels" / "location_b" / "cm" / "location_b-cm.tif")

        yield root


# ---------------------------------------------------------------------------
# Unit tests: helper functions
# ---------------------------------------------------------------------------


class TestFindOscdPairs:
    def test_discovers_all_pairs(self, oscd_dir):
        pairs = find_oscd_pairs(oscd_dir)
        assert len(pairs) == 2
        locations = {p["location"] for p in pairs}
        assert locations == {"location_a", "location_b"}

    def test_pair_has_required_keys(self, oscd_dir):
        pairs = find_oscd_pairs(oscd_dir)
        for pair in pairs:
            assert "location" in pair
            assert "before_bands_dir" in pair
            assert "after_bands_dir" in pair
            assert "mask_path" in pair
            assert pair["before_bands_dir"].exists()
            assert pair["after_bands_dir"].exists()

    def test_no_pairs_if_no_images(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            root = Path(tmpdir)
            (root / "images").mkdir()
            (root / "labels").mkdir()
            pairs = find_oscd_pairs(root)
            assert len(pairs) == 0


class TestLoadBandsAsRgb:
    def test_extracts_rgb_from_bands(self, oscd_dir):
        bands_dir = oscd_dir / "images" / "location_a" / "imgs_1"
        rgb = load_bands_as_rgb(bands_dir)
        assert rgb.ndim == 3
        assert rgb.shape[2] == 3
        assert rgb.dtype == np.uint8
        assert rgb.max() <= 255

    def test_raises_on_missing_bands(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            bands_dir = Path(tmpdir)
            # Only B01 — missing B04/B03/B02 which are in _BAND_PRIORITY
            tifffile.imwrite(str(bands_dir / "dummy_B01.tif"), np.zeros((16, 16), dtype=np.uint16))
            with pytest.raises(ValueError, match="Missing required band"):
                load_bands_as_rgb(bands_dir)


class TestLoadMask:
    def test_loads_binary_mask(self, oscd_dir):
        mask_path = oscd_dir / "labels" / "location_a" / "cm" / "location_a-cm.tif"
        mask = load_mask(mask_path)
        assert mask is not None
        assert mask.ndim == 2
        assert set(np.unique(mask)).issubset({0, 1})

    def test_returns_none_if_no_mask(self):
        result = load_mask(None)
        assert result is None

    def test_binarizes_255_to_1(self, oscd_dir):
        mask_path = oscd_dir / "labels" / "location_a" / "cm" / "location_a-cm.tif"
        mask = load_mask(mask_path)
        assert mask.max() <= 1


class TestMaskToBboxes:
    def test_finds_change_region(self, oscd_dir):
        mask_path = oscd_dir / "labels" / "location_a" / "cm" / "location_a-cm.tif"
        mask = load_mask(mask_path)
        bboxes = mask_to_bboxes(mask)
        # The dummy mask has a 32×32=1024 px change region — well above 50px threshold
        assert len(bboxes) >= 1
        x1, y1, x2, y2 = bboxes[0]
        assert x1 < x2
        assert y1 < y2

    def test_small_component_filtered_by_threshold(self):
        """Issue 8: components smaller than 50px are filtered out."""
        mask = np.zeros((64, 64), dtype=np.uint8)
        # Scatter 9 isolated change pixels (each is a 1-pixel component)
        for i in range(9):
            mask[i * 6, i * 6] = 1  # 9 single-pixel components
        bboxes = mask_to_bboxes(mask)
        # All components are 1px each — all should be filtered (< 50px threshold)
        assert bboxes == [], f"Expected no bboxes for sub-threshold components, got {bboxes}"

    def test_empty_mask_returns_empty(self):
        mask = np.zeros((64, 64), dtype=np.uint8)
        bboxes = mask_to_bboxes(mask)
        assert bboxes == []


class TestMakeSampleId:
    def test_lowercase_underscored(self):
        sid = make_sample_id("Location-A")
        assert sid == "oscd_location_a"

    def test_max_length(self):
        sid = make_sample_id("a" * 100)
        assert len(sid) <= 64


# ---------------------------------------------------------------------------
# Integration tests: manifest/resumability pattern
# ---------------------------------------------------------------------------


class TestProcessPair:
    def test_returns_valid_sample(self, oscd_dir):
        pairs = find_oscd_pairs(oscd_dir)
        sample = process_pair(pairs[0], oscd_dir / "output")
        assert sample is not None
        assert sample["dataset"] == "oscd"
        assert sample["pair_type"] == "bitemporal"
        assert sample["modality"] == "optical"
        assert len(sample["image_path"]) == 2
        # Issue 3 fix: OSCD is always demoted to change_vqa with bbox=None
        assert sample["task"] == "change_vqa", (
            f"Expected change_vqa (Issue 3 fix), got {sample['task']}"
        )
        assert sample["bbox"] is None, (
            f"Expected null bbox for all OSCD samples (Issue 3 fix), got {sample['bbox']}"
        )

    def test_empty_mask_produces_null_bbox_change_vqa(self, oscd_dir):
        """Empty mask (or any mask) → bbox=null, task=change_vqa (Issue 3 fix)."""
        # Create a location with empty mask
        _create_dummy_bands(oscd_dir / "images" / "location_empty" / "imgs_1")
        _create_dummy_bands(oscd_dir / "images" / "location_empty" / "imgs_2")
        _create_dummy_mask(oscd_dir / "labels" / "location_empty" / "cm" / "location_empty-cm.tif",
                           change_fraction=0.0)

        pairs = find_oscd_pairs(oscd_dir)
        empty_pair = [p for p in pairs if p["location"] == "location_empty"][0]
        sample = process_pair(empty_pair, oscd_dir / "output")

        assert sample is not None
        assert sample["bbox"] is None, f"Expected null bbox for empty mask, got {sample['bbox']}"
        assert sample["task"] == "change_vqa", f"Expected change_vqa task, got {sample['task']}"

    def test_writes_pngs(self, oscd_dir):
        pairs = find_oscd_pairs(oscd_dir)
        output_dir = oscd_dir / "output"
        process_pair(pairs[0], output_dir)
        assert (output_dir / "images" / f"{pairs[0]['location']}_before.png").exists()
        assert (output_dir / "images" / f"{pairs[0]['location']}_after.png").exists()


class TestManifestResumability:
    """Core test: verify the manifest pattern works as designed."""

    def test_first_run_processes_all(self, oscd_dir):
        """First run should process all pairs and write manifest entries."""
        output_dir = oscd_dir / "output"
        stats = run_tier1_oscd(oscd_dir, output_dir)

        assert stats["processed"] == 2
        assert stats["skipped"] == 0
        assert stats["failed"] == 0

        jsonl_path = output_dir / "oscd.jsonl"
        assert jsonl_path.exists()
        with open(jsonl_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        assert len(lines) == 2

    def test_second_run_skips_all(self, oscd_dir):
        """Second run should skip all already-processed pairs."""
        output_dir = oscd_dir / "output"

        stats1 = run_tier1_oscd(oscd_dir, output_dir)
        assert stats1["processed"] == 2

        stats2 = run_tier1_oscd(oscd_dir, output_dir)
        assert stats2["processed"] == 0
        assert stats2["skipped"] == 2

    def test_idempotent_output(self, oscd_dir):
        """Re-running produces byte-identical PNGs."""
        output_dir = oscd_dir / "output"

        run_tier1_oscd(oscd_dir, output_dir)

        png1 = output_dir / "images" / "location_a_before.png"
        hash1 = hashlib.md5(png1.read_bytes()).hexdigest()

        run_tier1_oscd(oscd_dir, output_dir)
        hash2 = hashlib.md5(png1.read_bytes()).hexdigest()

        assert hash1 == hash2, "PNG changed across runs — not idempotent"

    def test_manifest_counts_accurate(self, oscd_dir):
        """Manifest processed count matches actual output."""
        output_dir = oscd_dir / "output"
        manifest_path = output_dir / "manifest.parquet"

        run_tier1_oscd(oscd_dir, output_dir)

        manifest = Manifest(manifest_path)
        assert manifest.count_processed() == 2

    def test_partial_run_resumes(self, oscd_dir):
        """Simulate interrupted run: process 1 pair, then resume."""
        output_dir = oscd_dir / "output"
        manifest_path = output_dir / "manifest.parquet"
        jsonl_path = output_dir / "oscd.jsonl"

        # Manually process only the first pair
        pairs = find_oscd_pairs(oscd_dir)
        manifest = Manifest(manifest_path)
        sample = process_pair(pairs[0], output_dir)
        assert sample is not None

        from preprocess.common.io import append_jsonl
        append_jsonl(sample, jsonl_path)
        manifest.mark_processed(
            sample_id=make_sample_id(pairs[0]["location"]),
            dataset="oscd", split="train", output_shard="local",
        )

        # Now run the full pipeline — should process only the second pair
        stats = run_tier1_oscd(oscd_dir, output_dir)
        assert stats["processed"] == 1
        assert stats["skipped"] == 1

        with open(jsonl_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        assert len(lines) == 2


class TestValidation:
    """Verify produced samples pass the frozen schema."""

    def test_all_samples_validate(self, oscd_dir):
        pairs = find_oscd_pairs(oscd_dir)
        output_dir = oscd_dir / "output"
        for pair in pairs:
            sample = process_pair(pair, output_dir)
            assert sample is not None
            from preprocess.validator import validate_sample
            ok, errs = validate_sample(sample)
            assert ok, f"Validation failed for {pair['location']}: {errs}"


In [ ]:
%%writefile tests/test_tier1_rsvqa_hr.py
"""Tests for preprocess.tier1_rsvqa_hr — RSVQA-HR preprocessing."""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pytest
from PIL import Image

from preprocess.tier1_rsvqa_hr import load_annotations, parse_rsvqa_sample, run_tier1_rsvqa_hr


class TestLoadAnnotations:
    def test_loads_train_json(self, tmp_path):
        data = [{"question_id": 1, "question": "What is this?", "answer": "building"}]
        (tmp_path / "train.json").write_text(json.dumps(data))
        result = load_annotations(tmp_path)
        assert len(result) == 1

    def test_loads_questions_json(self, tmp_path):
        data = [{"question_id": 2, "question": "How many?", "answer": "5"}]
        (tmp_path / "questions.json").write_text(json.dumps(data))
        result = load_annotations(tmp_path)
        assert len(result) == 1

    def test_train_takes_priority(self, tmp_path):
        train = [{"question_id": 1}]
        questions = [{"question_id": 2}]
        (tmp_path / "train.json").write_text(json.dumps(train))
        (tmp_path / "questions.json").write_text(json.dumps(questions))
        result = load_annotations(tmp_path)
        assert result[0]["question_id"] == 1

    def test_wrapped_format(self, tmp_path):
        data = {"questions": [{"question_id": 1}]}
        (tmp_path / "train.json").write_text(json.dumps(data))
        result = load_annotations(tmp_path)
        assert len(result) == 1

    def test_empty_dir(self, tmp_path):
        result = load_annotations(tmp_path)
        assert result == []


class TestParseRsvqaSample:
    def test_valid_sample(self, tmp_path):
        entry = {"question_id": 1, "question": "What is this?", "answer": "building"}
        result = parse_rsvqa_sample(entry, None, "rsvqa_1")
        assert result is not None
        assert result["id"] == "rsvqa_1"
        assert result["dataset"] == "rsvqa_hr"
        assert result["task"] == "vqa"
        assert result["instruction"] == "What is this?"
        assert result["response"] == "building"

    def test_empty_question_skipped(self):
        entry = {"question_id": 1, "question": "", "answer": "building"}
        result = parse_rsvqa_sample(entry, None, "rsvqa_1")
        assert result is None

    def test_empty_answer_skipped(self):
        entry = {"question_id": 1, "question": "What?", "answer": ""}
        result = parse_rsvqa_sample(entry, None, "rsvqa_1")
        assert result is None

    def test_image_path_set_when_available(self, tmp_path):
        images_dir = tmp_path / "images"
        images_dir.mkdir()
        (images_dir / "img001.png").write_bytes(b"\x89PNG")

        entry = {"question_id": 1, "question": "What?", "answer": "road", "image_id": "img001"}
        result = parse_rsvqa_sample(entry, images_dir, "rsvqa_1")
        assert result is not None
        assert len(result["image_path"]) == 1
        assert "img001.png" in result["image_path"][0]

    def test_image_path_empty_when_missing(self):
        entry = {"question_id": 1, "question": "What?", "answer": "road", "image_id": "nonexistent"}
        result = parse_rsvqa_sample(entry, Path("/nonexistent"), "rsvqa_1")
        assert result is not None
        assert result["image_path"] == []

    def test_jpg_source_converted_to_png(self, tmp_path):
        """R7: all outputs must be PNG — a .jpg source must be converted,
        not referenced directly."""
        images_dir = tmp_path / "images"
        images_dir.mkdir()
        png_dir = tmp_path / "converted"
        img = Image.fromarray(np.zeros((32, 32, 3), dtype=np.uint8))
        img.save(str(images_dir / "img002.jpg"), format="JPEG")

        entry = {"question_id": 2, "question": "What?", "answer": "field", "image_id": "img002"}
        result = parse_rsvqa_sample(entry, images_dir, "rsvqa_2", png_dir=png_dir)
        assert result is not None
        assert result["image_path"][0].endswith(".png")
        assert Path(result["image_path"][0]).exists()


class TestRunTier1RsvqaHr:
    def _make_input(self, root: Path, n: int = 4) -> None:
        images_dir = root / "images"
        images_dir.mkdir(parents=True)
        entries = []
        for i in range(n):
            Image.fromarray(np.zeros((16, 16, 3), dtype=np.uint8)).save(
                str(images_dir / f"img{i:03d}.png")
            )
            entries.append({
                "question_id": i,
                "question": "What is this?",
                "answer": "building",
                "image_id": f"img{i:03d}",
            })
        (root / "train.json").write_text(json.dumps(entries))

    def test_r4_dual_resolution_doubles_rows(self, tmp_path):
        """R4: RSVQA-HR is in DUAL_RESOLUTION_DATASETS — every sample
        with an image must produce a native + CARTOSAT-proxy row."""
        input_dir = tmp_path / "input"
        self._make_input(input_dir)

        output_dir = tmp_path / "output"
        stats = run_tier1_rsvqa_hr(input_dir, output_dir)
        assert stats["processed"] == 4

        with open(output_dir / "rsvqa_hr.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]

        assert len(lines) == 8  # 4 samples x 2 (native + proxy)
        buckets = [l["gsd_bucket"] for l in lines]
        assert "[GSD:0.15m]" in buckets
        assert any("CARTOSAT-proxy" in b for b in buckets)

    def test_all_rows_validate(self, tmp_path):
        input_dir = tmp_path / "input"
        self._make_input(input_dir)
        output_dir = tmp_path / "output"
        run_tier1_rsvqa_hr(input_dir, output_dir)

        from preprocess.validator import validate_sample
        with open(output_dir / "rsvqa_hr.jsonl") as f:
            for line in f:
                if not line.strip():
                    continue
                ok, errs = validate_sample(json.loads(line))
                assert ok, errs


class TestRsvqaHrSchema:
    def test_has_all_fields(self):
        entry = {"question_id": 1, "question": "Q", "answer": "A"}
        result = parse_rsvqa_sample(entry, None, "rsvqa_1")
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in result, f"Missing field: {field}"

    def test_dataset_enum(self):
        entry = {"question_id": 1, "question": "Q", "answer": "A"}
        result = parse_rsvqa_sample(entry, None, "rsvqa_1")
        assert result["dataset"] == "rsvqa_hr"
        assert result["pair_type"] == "single"
        assert result["modality"] == "optical"


In [ ]:
%%writefile tests/test_tier1_sn6_opt.py
"""Tests for preprocess.tier1_sn6_opt — SpaceNet 6 Optical preprocessing."""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pytest
import tifffile

from preprocess.tier1_sn6_opt import (
    load_geojson_labels,
    polygons_to_bboxes,
    process_tile,
    run_tier1_sn6_opt,
)


class TestLoadGeojsonLabels:
    def test_loads_polygons(self, tmp_path):
        geojson = {
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[[0, 0], [100, 0], [100, 100], [0, 100], [0, 0]]],
                },
            }],
        }
        path = tmp_path / "labels.geojson"
        path.write_text(json.dumps(geojson))
        polygons = load_geojson_labels(path)
        assert len(polygons) == 1
        assert len(polygons[0]) == 5

    def test_loads_multipolygon(self, tmp_path):
        geojson = {
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "geometry": {
                    "type": "MultiPolygon",
                    "coordinates": [
                        [[[0, 0], [50, 0], [50, 50], [0, 50], [0, 0]]],
                        [[[60, 60], [100, 60], [100, 100], [60, 100], [60, 60]]],
                    ],
                },
            }],
        }
        path = tmp_path / "labels.geojson"
        path.write_text(json.dumps(geojson))
        polygons = load_geojson_labels(path)
        assert len(polygons) == 2

    def test_empty_file(self, tmp_path):
        path = tmp_path / "empty.geojson"
        path.write_text("{}")
        polygons = load_geojson_labels(path)
        assert polygons == []

    def test_missing_file(self):
        polygons = load_geojson_labels(Path("/nonexistent.geojson"))
        assert polygons == []


class TestPolygonsToBboxes:
    def test_converts_polygon(self):
        polygon = [[0, 0], [100, 0], [100, 100], [0, 100], [0, 0]]
        bboxes = polygons_to_bboxes([polygon], 200, 200)
        assert len(bboxes) == 1
        assert len(bboxes[0]) == 4
        # Should be normalized to [0, 1000]
        assert bboxes[0][0] >= 0
        assert bboxes[0][2] <= 1000

    def test_small_polygon_included(self):
        polygon = [[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]
        bboxes = polygons_to_bboxes([polygon], 1000, 1000)
        assert len(bboxes) == 1


class TestProcessTile:
    def test_returns_valid_sample(self, tmp_path):
        import numpy as np
        import tifffile

        # Create a test TIF
        img = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        tif_path = tmp_path / "tile001.tif"
        tifffile.imwrite(str(tif_path), img)

        # Create empty geojson
        geojson = {"type": "FeatureCollection", "features": []}
        geojson_path = tmp_path / "tile001.geojson"
        geojson_path.write_text(json.dumps(geojson))

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_tile(tif_path, geojson_path, output_dir, "tile001")
        assert sample is not None
        assert sample["id"] == "sn6_tile001"
        assert sample["dataset"] == "sn6_opt"


    def test_bbox_count_matches_response_count(self, tmp_path):
        """Response text must describe exactly the boxes returned, not a
        total that only the first box represents."""
        img = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        tif_path = tmp_path / "tile002.tif"
        tifffile.imwrite(str(tif_path), img)

        def _poly(x0, y0, x1, y1):
            return [[x0, y0], [x1, y0], [x1, y1], [x0, y1], [x0, y0]]

        geojson = {
            "type": "FeatureCollection",
            "features": [
                {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [_poly(0, 0, 10, 10)]}},
                {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [_poly(20, 20, 30, 30)]}},
                {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [_poly(40, 40, 50, 50)]}},
            ],
        }
        geojson_path = tmp_path / "tile002.geojson"
        geojson_path.write_text(json.dumps(geojson))

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_tile(tif_path, geojson_path, output_dir, "tile002")
        assert sample is not None
        assert sample["bbox"] is not None
        assert len(sample["bbox"]) == 3
        assert "3 buildings detected" in sample["response"]


def _make_sn6_input(root: Path, tiles: list[int]) -> None:
    """tiles: list of building counts, one tile per entry."""
    images_dir = root / "train" / "images"
    labels_dir = root / "train" / "labels"
    images_dir.mkdir(parents=True)
    labels_dir.mkdir(parents=True)

    for i, n_buildings in enumerate(tiles):
        tile_id = f"tile{i:04d}"
        img = np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)
        tifffile.imwrite(str(images_dir / f"{tile_id}.tif"), img)

        features = []
        for b in range(n_buildings):
            x0 = (b * 2) % 30
            y0 = (b * 3) % 30
            features.append({
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[[x0, y0], [x0 + 1, y0], [x0 + 1, y0 + 1], [x0, y0 + 1], [x0, y0]]],
                },
            })
        geojson = {"type": "FeatureCollection", "features": features}
        (labels_dir / f"{tile_id}.geojson").write_text(json.dumps(geojson))


class TestRunTier1Sn6Opt:
    def test_r4_dual_resolution_doubles_rows(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_sn6_input(input_dir, tiles=[2, 5, 0])

        output_dir = tmp_path / "output"
        stats = run_tier1_sn6_opt(input_dir, output_dir, sample_fraction=1.0)
        assert stats["processed"] == 3

        with open(output_dir / "sn6_opt.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        assert len(lines) == 6
        assert any("CARTOSAT-proxy" in l["gsd_bucket"] for l in lines)

    def test_selected_tile_ids_written(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_sn6_input(input_dir, tiles=[1, 2, 3])
        output_dir = tmp_path / "output"
        run_tier1_sn6_opt(input_dir, output_dir, sample_fraction=1.0)

        selected_path = output_dir / "selected_tile_ids.json"
        assert selected_path.exists()
        ids = json.loads(selected_path.read_text())
        assert set(ids) == {"tile0000", "tile0001", "tile0002"}

    def test_stratified_sampling_keeps_sparse_and_dense(self, tmp_path):
        input_dir = tmp_path / "input"
        # Mostly dense tiles, a couple of empty (sparse) ones.
        _make_sn6_input(input_dir, tiles=[10] * 10 + [0, 0])

        output_dir = tmp_path / "output"
        run_tier1_sn6_opt(input_dir, output_dir, sample_fraction=0.3, seed=1)

        selected = json.loads((output_dir / "selected_tile_ids.json").read_text())
        counts = {"tile0010": 0, "tile0011": 0}  # the 2 sparse (0-building) tiles
        assert any(t in selected for t in counts), \
            "Sparse-density stratum was dropped by subsampling"


class TestSn6OptSchema:
    def test_has_all_fields(self, tmp_path):
        import numpy as np
        import tifffile

        img = np.zeros((32, 32, 3), dtype=np.uint8)
        tif_path = tmp_path / "tile.tif"
        tifffile.imwrite(str(tif_path), img)

        geojson = {"type": "FeatureCollection", "features": []}
        geojson_path = tmp_path / "tile.geojson"
        geojson_path.write_text(json.dumps(geojson))

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_tile(tif_path, geojson_path, output_dir, "tile")
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in sample, f"Missing field: {field}"


In [ ]:
%%writefile tests/test_tier1_vrsbench.py
"""Tests for preprocess/tier1_vrsbench.py — caption + grounding + VQA + dual-resolution.

Parses VRSBench_train.json (LLaVA format) and verifies:
    1. Conversation parsing: [caption]/[refer]/[vqa] prefix extraction
    2. Referring coordinate conversion: {<x><y><x><y>} → [0,1000]
    3. R4 dual-resolution: each sample → native + proxy rows
    4. Manifest/resumability: first run, second-run-skips, idempotent, partial resume
    5. Task-bbox consistency: grounding → non-null bbox, caption/vqa → null bbox
"""

from __future__ import annotations

import hashlib
import json
import tempfile
from pathlib import Path

import pytest

from preprocess.common.gsd import assign_gsd_bucket
from preprocess.common.io import Manifest
from preprocess.tier1_vrsbench import (
    parse_conversation,
    referring_to_qwen,
    process_entry,
    run_tier1_vrsbench,
)


# ---------------------------------------------------------------------------
# Fixtures: create synthetic VRSBench_train.json
# ---------------------------------------------------------------------------


def _make_entry(
    image: str = "P0000.png",
    task: str = "caption",
    instruction: str = "Describe this image in detail.",
    response: str = "A detailed aerial image.",
    entry_id: str = "test_001",
) -> dict:
    """Create a synthetic VRSBench_train.json entry."""
    prefix = f"[{task}]"
    return {
        "id": entry_id,
        "image": image,
        "conversations": [
            {"from": "human", "value": f"<image>\n{prefix} {instruction}"},
            {"from": "gpt", "value": response},
        ],
    }


@pytest.fixture
def vrsbench_dir():
    """Create a temporary VRSBench directory with synthetic train JSON."""
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)

        entries = [
            _make_entry("P0000.png", "caption", "Describe this image.",
                        "An urban scene with buildings."),
            _make_entry("P0000.png", "refer", "Where is the aircraft?",
                        "{<45><45><59><59>}", entry_id="test_002"),
            _make_entry("P0000.png", "vqa", "What is this?",
                        "airport", entry_id="test_003"),
            _make_entry("P0001.png", "caption", "Describe this image.",
                        "A coastal area with water.", entry_id="test_004"),
            _make_entry("P0001.png", "vqa", "How many buildings?",
                        "5", entry_id="test_005"),
        ]

        with open(root / "VRSBench_train.json", "w") as f:
            json.dump(entries, f)

        yield root


# ---------------------------------------------------------------------------
# Unit tests: parse_conversation
# ---------------------------------------------------------------------------


class TestParseConversation:
    def test_caption(self):
        entry = _make_entry(task="caption", instruction="Describe this image.",
                            response="An urban scene with buildings.")
        task, inst, resp = parse_conversation(entry)
        assert task == "caption"
        assert inst == "Describe this image."
        assert resp == "An urban scene with buildings."

    def test_refer(self):
        entry = _make_entry(task="refer", instruction="Where is the aircraft?",
                            response="{<45><45><59><59>}")
        task, inst, resp = parse_conversation(entry)
        assert task == "refer"
        assert inst == "Where is the aircraft?"
        assert resp == "{<45><45><59><59>}"

    def test_vqa(self):
        entry = _make_entry(task="vqa", instruction="What is this?",
                            response="airport")
        task, inst, resp = parse_conversation(entry)
        assert task == "vqa"
        assert inst == "What is this?"
        assert resp == "airport"

    def test_unknown_prefix(self):
        entry = {
            "id": "x", "image": "x.png",
            "conversations": [
                {"from": "human", "value": "<image>\nSomething else"},
                {"from": "gpt", "value": "answer"},
            ]
        }
        task, inst, resp = parse_conversation(entry)
        assert task == "unknown"

    def test_strips_newline(self):
        entry = {
            "id": "x", "image": "x.png",
            "conversations": [
                {"from": "human", "value": "<image>\n\n[caption] Describe it."},
                {"from": "gpt", "value": "A scene."},
            ]
        }
        task, inst, resp = parse_conversation(entry)
        assert task == "caption"
        assert inst == "Describe it."


# ---------------------------------------------------------------------------
# Unit tests: referring_to_qwen
# ---------------------------------------------------------------------------


class TestReferringToQwen:
    def test_basic_conversion(self):
        bbox = referring_to_qwen("{<45><45><59><59>}")
        assert bbox == [[450.0, 450.0, 590.0, 590.0]]

    def test_zero_coordinates(self):
        bbox = referring_to_qwen("{<0><0><10><10>}")
        assert bbox == [[0.0, 0.0, 100.0, 100.0]]

    def test_max_coordinates(self):
        bbox = referring_to_qwen("{<99><99><100><100>}")
        assert bbox is not None
        assert bbox[0][2] == 1000.0

    def test_reversed_coordinates_returns_none(self):
        bbox = referring_to_qwen("{<59><59><45><45>}")
        assert bbox is None

    def test_zero_area_returns_none(self):
        bbox = referring_to_qwen("{<50><50><50><60>}")
        assert bbox is None

    def test_invalid_format_returns_none(self):
        bbox = referring_to_qwen("not a bbox")
        assert bbox is None

    def test_empty_string_returns_none(self):
        bbox = referring_to_qwen("")
        assert bbox is None


# ---------------------------------------------------------------------------
# Unit tests: process_entry
# ---------------------------------------------------------------------------


class TestProcessEntry:
    def test_caption_produces_sample(self):
        entry = _make_entry(task="caption")
        samples = process_entry(entry, None)
        assert len(samples) == 1
        assert samples[0]["task"] == "caption"
        assert samples[0]["bbox"] is None

    def test_refer_produces_sample_with_bbox(self):
        entry = _make_entry(task="refer", response="{<10><20><30><40>}")
        samples = process_entry(entry, None)
        assert len(samples) == 1
        assert samples[0]["task"] == "grounding"
        assert samples[0]["bbox"] == [[100.0, 200.0, 300.0, 400.0]]

    def test_vqa_produces_sample(self):
        entry = _make_entry(task="vqa", response="airport")
        samples = process_entry(entry, None)
        assert len(samples) == 1
        assert samples[0]["task"] == "vqa"
        assert samples[0]["bbox"] is None

    def test_invalid_referring_skipped(self):
        entry = _make_entry(task="refer", response="invalid")
        samples = process_entry(entry, None)
        assert len(samples) == 0

    def test_empty_response_skipped(self):
        entry = _make_entry(task="caption", response="")
        samples = process_entry(entry, None)
        assert len(samples) == 0

    def test_image_path_set_when_available(self, vrsbench_dir):
        images_dir = vrsbench_dir / "images"
        images_dir.mkdir(exist_ok=True)
        # Create a dummy image
        from PIL import Image
        import numpy as np
        img = Image.fromarray(np.zeros((64, 64, 3), dtype=np.uint8))
        img.save(str(images_dir / "P0000.png"))

        entry = _make_entry("P0000.png", "caption")
        samples = process_entry(entry, images_dir)
        assert len(samples) == 1
        assert len(samples[0]["image_path"]) == 1
        assert Path(samples[0]["image_path"][0]).exists()


# ---------------------------------------------------------------------------
# R4 dual-resolution: native + proxy rows
# ---------------------------------------------------------------------------


class TestDualResolution:
    def test_each_sample_produces_two_rows(self, vrsbench_dir):
        output_dir = vrsbench_dir / "output"
        stats = run_tier1_vrsbench(vrsbench_dir, output_dir)
        assert stats["processed"] == 5

        with open(output_dir / "vrsbench.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]

        # 5 entries × 2 rows (native + proxy) = 10 rows
        assert len(lines) == 10

    def test_native_and_proxy_differ(self, vrsbench_dir):
        output_dir = vrsbench_dir / "output"
        run_tier1_vrsbench(vrsbench_dir, output_dir)

        with open(output_dir / "vrsbench.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]

        # Check that each pair has different gsd_bucket
        buckets = [l["gsd_bucket"] for l in lines]
        assert "VHR-native" in buckets
        assert any("CARTOSAT-proxy" in b for b in buckets)


# ---------------------------------------------------------------------------
# Manifest/resumability (same 5-test pattern as OSCD)
# ---------------------------------------------------------------------------


class TestManifestResumability:
    def test_first_run_processes_all(self, vrsbench_dir):
        output_dir = vrsbench_dir / "output"
        stats = run_tier1_vrsbench(vrsbench_dir, output_dir)
        assert stats["processed"] == 5
        assert stats["skipped"] == 0

    def test_second_run_skips_all(self, vrsbench_dir):
        output_dir = vrsbench_dir / "output"
        stats1 = run_tier1_vrsbench(vrsbench_dir, output_dir)
        assert stats1["processed"] == 5

        stats2 = run_tier1_vrsbench(vrsbench_dir, output_dir)
        assert stats2["processed"] == 0
        assert stats2["skipped"] == 5

    def test_idempotent_output(self, vrsbench_dir):
        output_dir = vrsbench_dir / "output"
        run_tier1_vrsbench(vrsbench_dir, output_dir)
        jsonl_path = output_dir / "vrsbench.jsonl"
        hash1 = hashlib.md5(jsonl_path.read_bytes()).hexdigest()

        run_tier1_vrsbench(vrsbench_dir, output_dir)
        hash2 = hashlib.md5(jsonl_path.read_bytes()).hexdigest()
        assert hash1 == hash2

    def test_partial_run_resumes(self, vrsbench_dir):
        output_dir = vrsbench_dir / "output"
        manifest_path = output_dir / "manifest.parquet"
        jsonl_path = output_dir / "vrsbench.jsonl"

        # Manually process first 2 entries
        with open(vrsbench_dir / "VRSBench_train.json") as f:
            data = json.load(f)

        manifest = Manifest(manifest_path)
        native_bucket = assign_gsd_bucket("vrsbench", None)
        from preprocess.common.gsd import create_proxy_sample
        from preprocess.common.io import append_jsonl

        for i, entry in enumerate(data[:2]):
            samples = process_entry(entry, None, sample_id=f"{i:07d}")
            for s in samples:
                s["gsd_bucket"] = native_bucket
                append_jsonl(s, jsonl_path)
                proxy = create_proxy_sample(s)
                append_jsonl(proxy, jsonl_path)

            manifest.mark_processed(
                sample_id=f"{i:07d}",
                dataset="vrsbench", split="train", output_shard="local",
            )

        # Full run should process remaining 3
        stats = run_tier1_vrsbench(vrsbench_dir, output_dir)
        assert stats["processed"] == 3
        assert stats["skipped"] == 2


# ---------------------------------------------------------------------------
# Validation: produced samples pass frozen schema
# ---------------------------------------------------------------------------


class TestValidation:
    def test_all_samples_validate(self, vrsbench_dir):
        with open(vrsbench_dir / "VRSBench_train.json") as f:
            data = json.load(f)

        output_dir = vrsbench_dir / "output"
        native_bucket = assign_gsd_bucket("vrsbench", None)

        for entry in data:
            samples = process_entry(entry, None)
            for sample in samples:
                from preprocess.validator import validate_sample
                sample["gsd_bucket"] = native_bucket
                # Skip validation if no image_path (images not available locally)
                if not sample["image_path"]:
                    continue
                ok, errs = validate_sample(sample)
                assert ok, f"Validation failed for {sample['id']}: {errs}"


# ---------------------------------------------------------------------------
# Task-bbox consistency
# ---------------------------------------------------------------------------


class TestTaskBboxConsistency:
    def test_grounding_has_nonnull_bbox(self, vrsbench_dir):
        with open(vrsbench_dir / "VRSBench_train.json") as f:
            data = json.load(f)
        for entry in data:
            if "[refer]" in entry["conversations"][0]["value"]:
                samples = process_entry(entry, None)
                for s in samples:
                    assert s["bbox"] is not None, f"Grounding sample {s['id']} has null bbox"

    def test_caption_vqa_have_null_bbox(self, vrsbench_dir):
        with open(vrsbench_dir / "VRSBench_train.json") as f:
            data = json.load(f)
        for entry in data:
            conv_text = entry["conversations"][0]["value"]
            if "[caption]" in conv_text or "[vqa]" in conv_text:
                samples = process_entry(entry, None)
                for s in samples:
                    assert s["bbox"] is None, f"{s['task']} sample {s['id']} has non-null bbox"


# ---------------------------------------------------------------------------
# Issue 7: dynamic proxy scale factor
# Issue 9: proxy PNG compression
# ---------------------------------------------------------------------------


class TestGenerateProxyImage:
    """Tests for the generate_proxy_image() function (Issues 7 + 9)."""

    def test_dynamic_scale_factor_non_square(self, tmp_path):
        """Issue 7: a 256×128 source image should yield a 64×32 proxy (4× reduction)."""
        import numpy as np
        from PIL import Image
        from preprocess.tier1_vrsbench import generate_proxy_image

        src = tmp_path / "src.png"
        img = Image.fromarray(np.zeros((128, 256, 3), dtype=np.uint8))  # H=128, W=256
        img.save(str(src))

        proxy_dir = tmp_path / "proxy"
        result = generate_proxy_image(src, proxy_dir, "src.png", scale_factor=4)
        assert result is not None
        proxy = Image.open(result)
        w, h = proxy.size  # PIL: (width, height)
        assert w == 64, f"Expected proxy width 64, got {w}"
        assert h == 32, f"Expected proxy height 32, got {h}"

    def test_small_image_clamped_to_min_32(self, tmp_path):
        """Issue 7: a 16×16 image divided by 4 = 4, must be clamped to 32."""
        import numpy as np
        from PIL import Image
        from preprocess.tier1_vrsbench import generate_proxy_image

        src = tmp_path / "tiny.png"
        img = Image.fromarray(np.zeros((16, 16, 3), dtype=np.uint8))
        img.save(str(src))

        proxy_dir = tmp_path / "proxy"
        result = generate_proxy_image(src, proxy_dir, "tiny.png", scale_factor=4)
        assert result is not None
        proxy = Image.open(result)
        w, h = proxy.size
        assert w >= 32, f"Expected width >= 32 (clamped), got {w}"
        assert h >= 32, f"Expected height >= 32 (clamped), got {h}"

    def test_proxy_smaller_than_compress_level_0(self, tmp_path):
        """Issue 9: compress_level=6 proxy file must be smaller than level=0 equivalent."""
        import numpy as np
        from PIL import Image
        from preprocess.tier1_vrsbench import generate_proxy_image

        # Use a solid-color image — highly compressible
        src = tmp_path / "big.png"
        arr = np.full((512, 512, 3), 128, dtype=np.uint8)
        img = Image.fromarray(arr)
        img.save(str(src))

        proxy_dir_6 = tmp_path / "proxy_6"
        result = generate_proxy_image(src, proxy_dir_6, "big.png")
        assert result is not None
        size_6 = Path(result).stat().st_size

        # Write same content at level 0 for comparison
        proxy_l0 = tmp_path / "proxy_l0.png"
        proxy_img = Image.open(result)
        proxy_img.save(str(proxy_l0), format="PNG", compress_level=0)
        size_0 = proxy_l0.stat().st_size

        assert size_6 <= size_0, (
            f"compress_level=6 ({size_6} B) should be <= level=0 ({size_0} B)"
        )

    def test_idempotent_existing_proxy_not_overwritten(self, tmp_path):
        """generate_proxy_image() returns existing path without regenerating."""
        import numpy as np
        from PIL import Image
        from preprocess.tier1_vrsbench import generate_proxy_image

        src = tmp_path / "src.png"
        Image.fromarray(np.zeros((64, 64, 3), dtype=np.uint8)).save(str(src))

        proxy_dir = tmp_path / "proxy"
        result1 = generate_proxy_image(src, proxy_dir, "src.png")
        mtime1 = Path(result1).stat().st_mtime

        result2 = generate_proxy_image(src, proxy_dir, "src.png")
        mtime2 = Path(result2).stat().st_mtime

        assert result1 == result2
        assert mtime1 == mtime2, "Proxy was regenerated when it should have been reused"



In [ ]:
%%writefile tests/test_tier2_sardet.py
"""Tests for preprocess.tier2_sardet — SARDet-100K preprocessing."""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pytest
from PIL import Image

from preprocess.tier2_sardet import (
    load_yolo_labels,
    primary_class,
    process_sardet_sample,
    run_tier2_sardet,
)


class TestLoadYoloLabels:
    def test_loads_labels(self, tmp_path):
        # YOLO format: class_id cx cy w h (normalized)
        label_path = tmp_path / "img001.txt"
        label_path.write_text("0 0.5 0.5 0.1 0.2\n1 0.3 0.3 0.05 0.1\n")

        annotations = load_yolo_labels(label_path, 100, 100)
        assert len(annotations) == 2
        assert annotations[0]["class_name"] == "ship"
        assert annotations[1]["class_name"] == "aircraft"

    def test_empty_file(self, tmp_path):
        label_path = tmp_path / "empty.txt"
        label_path.write_text("")
        annotations = load_yolo_labels(label_path, 100, 100)
        assert annotations == []

    def test_missing_file(self):
        annotations = load_yolo_labels(Path("/nonexistent.txt"), 100, 100)
        assert annotations == []

    def test_invalid_class_id_skipped(self, tmp_path):
        label_path = tmp_path / "img.txt"
        label_path.write_text("99 0.5 0.5 0.1 0.1\n")
        annotations = load_yolo_labels(label_path, 100, 100)
        assert annotations == []


class TestProcessSardetSample:
    def test_returns_valid_sample(self, tmp_path):
        from PIL import Image
        import numpy as np

        # Create SAR-like image
        img = Image.fromarray(np.random.randint(0, 255, (64, 64), dtype=np.uint8))
        img.save(str(tmp_path / "img001.png"))

        # Create labels
        (tmp_path / "img001.txt").write_text("0 0.5 0.5 0.2 0.2\n")

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_sardet_sample(
            tmp_path / "img001.png",
            tmp_path / "img001.txt",
            output_dir,
            "sardet_001",
        )
        assert sample is not None
        assert sample["id"] == "sardet_001"
        assert sample["dataset"] == "sardet"
        assert sample["modality"] == "sar"

    def test_no_labels_returns_none(self, tmp_path):
        from PIL import Image
        import numpy as np

        img = Image.fromarray(np.zeros((64, 64), dtype=np.uint8))
        img.save(str(tmp_path / "img001.png"))

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_sardet_sample(
            tmp_path / "img001.png",
            tmp_path / "nonexistent.txt",
            output_dir,
            "sardet_001",
        )
        assert sample is None


class TestPrimaryClass:
    def test_reads_first_class(self, tmp_path):
        label_path = tmp_path / "img.txt"
        label_path.write_text("2 0.5 0.5 0.1 0.1\n0 0.2 0.2 0.1 0.1\n")
        assert primary_class(label_path) == "bridge"

    def test_missing_file_returns_unknown(self, tmp_path):
        assert primary_class(tmp_path / "nope.txt") == "unknown"


def _make_sardet_input(root: Path, class_counts: dict[str, int]) -> None:
    """class_counts: {category_name: n_images_with_that_primary_class}."""
    images_dir = root / "images"
    labels_dir = root / "labels"
    images_dir.mkdir(parents=True)
    labels_dir.mkdir(parents=True)

    categories = ["ship", "aircraft", "bridge", "tank", "car", "harbor"]
    idx = 0
    for cat, n in class_counts.items():
        class_id = categories.index(cat)
        for _ in range(n):
            name = f"img{idx:04d}"
            Image.fromarray(np.random.randint(0, 255, (32, 32), dtype=np.uint8)).save(
                str(images_dir / f"{name}.png"))
            (labels_dir / f"{name}.txt").write_text(f"{class_id} 0.5 0.5 0.2 0.2\n")
            idx += 1


class TestRunTier2Sardet:
    def test_stratified_sampling_keeps_rare_category(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_sardet_input(input_dir, {"ship": 20, "harbor": 2})

        output_dir = tmp_path / "output"
        run_tier2_sardet(input_dir, output_dir, sample_fraction=0.2, seed=1)

        with open(output_dir / "sardet.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        responses = [l["response"] for l in lines]
        assert any("harbor" in r for r in responses), \
            "Rare category (harbor, n=2) was dropped by subsampling"

    def test_all_rows_validate(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_sardet_input(input_dir, {"ship": 3, "car": 2})
        output_dir = tmp_path / "output"
        run_tier2_sardet(input_dir, output_dir, sample_fraction=1.0)

        from preprocess.validator import validate_sample
        with open(output_dir / "sardet.jsonl") as f:
            for line in f:
                if not line.strip():
                    continue
                ok, errs = validate_sample(json.loads(line))
                assert ok, errs


class TestSardetSchema:
    def test_has_all_fields(self, tmp_path):
        from PIL import Image
        import numpy as np

        img = Image.fromarray(np.zeros((64, 64), dtype=np.uint8))
        img.save(str(tmp_path / "img.png"))
        (tmp_path / "img.txt").write_text("0 0.5 0.5 0.2 0.2\n")

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        sample = process_sardet_sample(
            tmp_path / "img.png",
            tmp_path / "img.txt",
            output_dir,
            "sardet_001",
        )
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in sample, f"Missing field: {field}"


In [ ]:
%%writefile tests/test_tier2_sn6_sar.py
"""Tests for preprocess.tier2_sn6_sar — SpaceNet 6 SAR preprocessing."""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pytest
import tifffile
from PIL import Image

from preprocess.tier2_sn6_sar import (
    load_optical_selection,
    process_sar_tile,
    run_tier2_sn6_sar,
)


class TestProcessSarTile:
    def test_returns_valid_sample(self, tmp_path):
        # Create a 2D SAR tile
        img = np.random.randint(0, 1000, (64, 64), dtype=np.uint32)
        tile_path = tmp_path / "tile001.tif"
        tifffile.imwrite(str(tile_path), img)

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        samples = process_sar_tile(tile_path, output_dir, "tile001")
        assert len(samples) == 1
        sample = samples[0]
        assert sample["id"] == "sn6_sar_tile001"
        assert sample["dataset"] == "sn6_sar"
        assert sample["modality"] == "sar"

    def test_3d_sar_tile(self, tmp_path):
        # Create a 3D SAR tile (multi-pol)
        img = np.random.randint(0, 1000, (3, 64, 64), dtype=np.uint32)
        tile_path = tmp_path / "tile002.tif"
        tifffile.imwrite(str(tile_path), img)

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        samples = process_sar_tile(tile_path, output_dir, "tile002")
        assert len(samples) == 1
        assert samples[0]["modality"] == "sar"

    def test_fusion_sample_emitted_when_optical_given(self, tmp_path):
        img = np.random.randint(0, 1000, (64, 64), dtype=np.uint32)
        tile_path = tmp_path / "tile003.tif"
        tifffile.imwrite(str(tile_path), img)

        optical_png = tmp_path / "optical.png"
        Image.fromarray(np.zeros((64, 64, 3), dtype=np.uint8)).save(str(optical_png))

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        samples = process_sar_tile(tile_path, output_dir, "tile003", optical_png=optical_png)
        assert len(samples) == 2
        fusion = [s for s in samples if s["pair_type"] == "cross-modal"][0]
        assert fusion["task"] == "fusion_vqa"
        assert fusion["modality"] == "optical+sar"
        assert len(fusion["image_path"]) == 2


class TestLoadOpticalSelection:
    def test_reads_selection_and_paths(self, tmp_path):
        opt_dir = tmp_path / "opt_output"
        opt_dir.mkdir()
        (opt_dir / "selected_tile_ids.json").write_text(json.dumps(["tile001", "tile002"]))

        rows = [
            {"id": "sn6_tile001", "image_path": ["/x/tile001.png"]},
            {"id": "sn6_tile001_proxy", "image_path": ["/x/tile001_proxy.png"]},
            {"id": "sn6_tile002", "image_path": ["/x/tile002.png"]},
        ]
        with open(opt_dir / "sn6_opt.jsonl", "w") as f:
            for r in rows:
                f.write(json.dumps(r) + "\n")

        ids, by_tile = load_optical_selection(opt_dir)
        assert ids == {"tile001", "tile002"}
        assert by_tile == {"tile001": "/x/tile001.png", "tile002": "/x/tile002.png"}

    def test_missing_files_returns_empty(self, tmp_path):
        ids, by_tile = load_optical_selection(tmp_path / "nonexistent")
        assert ids == set()
        assert by_tile == {}


class TestRunTier2Sn6Sar:
    def _make_sar_input(self, root: Path, tile_ids: list[str]) -> None:
        sar_dir = root / "train" / "sar"
        sar_dir.mkdir(parents=True)
        for tid in tile_ids:
            img = np.random.randint(0, 1000, (32, 32), dtype=np.uint32)
            tifffile.imwrite(str(sar_dir / f"{tid}.tif"), img)

    def test_paired_selection_restricts_to_optical_tiles(self, tmp_path):
        sar_input = tmp_path / "sar_input"
        self._make_sar_input(sar_input, ["t1", "t2", "t3", "t4"])

        opt_dir = tmp_path / "opt_output"
        opt_dir.mkdir()
        (opt_dir / "selected_tile_ids.json").write_text(json.dumps(["t1", "t3"]))
        Image.fromarray(np.zeros((16, 16, 3), dtype=np.uint8)).save(str(opt_dir / "t1.png"))
        rows = [{"id": "sn6_t1", "image_path": [str(opt_dir / "t1.png")]}]
        with open(opt_dir / "sn6_opt.jsonl", "w") as f:
            for r in rows:
                f.write(json.dumps(r) + "\n")

        output_dir = tmp_path / "output"
        stats = run_tier2_sn6_sar(sar_input, output_dir, optical_dir=opt_dir)
        assert stats["processed"] == 2  # only t1, t3 — not t2/t4

    def test_fusion_row_emitted_for_paired_tile(self, tmp_path):
        sar_input = tmp_path / "sar_input"
        self._make_sar_input(sar_input, ["t1"])

        opt_dir = tmp_path / "opt_output"
        opt_dir.mkdir()
        (opt_dir / "selected_tile_ids.json").write_text(json.dumps(["t1"]))
        Image.fromarray(np.zeros((16, 16, 3), dtype=np.uint8)).save(str(opt_dir / "t1.png"))
        with open(opt_dir / "sn6_opt.jsonl", "w") as f:
            f.write(json.dumps({"id": "sn6_t1", "image_path": [str(opt_dir / "t1.png")]}) + "\n")

        output_dir = tmp_path / "output"
        run_tier2_sn6_sar(sar_input, output_dir, optical_dir=opt_dir)

        with open(output_dir / "sn6_sar.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        assert any(l["pair_type"] == "cross-modal" for l in lines)

    def test_all_rows_validate(self, tmp_path):
        sar_input = tmp_path / "sar_input"
        self._make_sar_input(sar_input, ["t1", "t2"])
        output_dir = tmp_path / "output"
        run_tier2_sn6_sar(sar_input, output_dir, sample_fraction=1.0)

        from preprocess.validator import validate_sample
        with open(output_dir / "sn6_sar.jsonl") as f:
            for line in f:
                if not line.strip():
                    continue
                ok, errs = validate_sample(json.loads(line))
                assert ok, errs


class TestSn6SarSchema:
    def test_has_all_fields(self, tmp_path):
        img = np.zeros((64, 64), dtype=np.uint32)
        tile_path = tmp_path / "tile.tif"
        tifffile.imwrite(str(tile_path), img)

        output_dir = tmp_path / "output"
        output_dir.mkdir()

        samples = process_sar_tile(tile_path, output_dir, "tile")
        sample = samples[0]
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in sample, f"Missing field: {field}"


In [ ]:
%%writefile tests/test_tier3_sen2lulc.py
"""Tests for preprocess.tier3_sen2lulc — Sen-2 LULC preprocessing."""

from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np
import pytest
from PIL import Image

from preprocess.tier3_sen2lulc import (
    load_metadata,
    mask_to_bbox,
    parse_sen2lulc_sample,
    run_tier3_sen2lulc,
)


class TestLoadMetadata:
    def test_loads_csv(self, tmp_path):
        csv_path = tmp_path / "metadata.csv"
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["image_id", "label"])
            writer.writeheader()
            writer.writerow({"image_id": "img001", "label": "0"})
            writer.writerow({"image_id": "img002", "label": "3"})

        entries = load_metadata(tmp_path)
        assert len(entries) == 2

    def test_loads_json(self, tmp_path):
        data = [{"image_id": "img001", "label": 1}]
        (tmp_path / "annotations.json").write_text(json.dumps(data))
        entries = load_metadata(tmp_path)
        assert len(entries) == 1

    def test_empty_dir(self, tmp_path):
        entries = load_metadata(tmp_path)
        assert entries == []


class TestMaskToBbox:
    def test_finds_class_regions(self, tmp_path):
        import numpy as np
        from PIL import Image

        # Create mask with class 0 (Built-up) in top-left, class 2 (Water) in bottom-right
        mask = np.zeros((64, 64), dtype=np.uint8)
        mask[0:32, 0:32] = 0  # Built-up
        mask[32:64, 32:64] = 2  # Water

        mask_path = tmp_path / "mask.png"
        Image.fromarray(mask).save(str(mask_path))

        bboxes = mask_to_bbox(mask_path)
        assert bboxes is not None
        assert len(bboxes) == 2

    def test_empty_mask_returns_none(self, tmp_path):
        import numpy as np
        from PIL import Image

        mask = np.full((64, 64), 255, dtype=np.uint8)  # Class 255 doesn't exist
        mask_path = tmp_path / "mask.png"
        Image.fromarray(mask).save(str(mask_path))

        bboxes = mask_to_bbox(mask_path)
        assert bboxes is None


class TestParseSen2LulcSample:
    def test_valid_sample(self):
        entry = {"image_id": "img001", "label": 0}
        result = parse_sen2lulc_sample(entry, None, None, "sen2lulc_img001")
        assert result is not None
        assert result["id"] == "sen2lulc_img001"
        assert result["dataset"] == "sen2lulc"
        assert result["response"] == "Built-up"

    def test_string_label(self):
        entry = {"image_id": "img001", "label": "Water"}
        result = parse_sen2lulc_sample(entry, None, None, "sen2lulc_img001")
        assert result["response"] == "Water"

    def test_missing_image_id(self):
        entry = {"label": 0}
        result = parse_sen2lulc_sample(entry, None, None, "sen2lulc_001")
        assert result is None

    def test_has_all_fields(self):
        entry = {"image_id": "img001", "label": 1}
        result = parse_sen2lulc_sample(entry, None, None, "sen2lulc_img001")
        required = ["id", "dataset", "task", "image_path", "pair_type",
                     "gsd_bucket", "split", "instruction", "response",
                     "bbox", "modality"]
        for field in required:
            assert field in result, f"Missing field: {field}"


def _make_sen2lulc_input(root: Path, class_counts: dict[int, int]) -> None:
    images_dir = root / "images"
    images_dir.mkdir(parents=True)
    rows = []
    idx = 0
    for label, n in class_counts.items():
        for _ in range(n):
            image_id = f"img{idx:04d}"
            Image.fromarray(np.zeros((16, 16, 3), dtype=np.uint8)).save(
                str(images_dir / f"{image_id}.png"))
            rows.append({"image_id": image_id, "label": label})
            idx += 1

    import csv as csv_mod
    with open(root / "metadata.csv", "w", newline="") as f:
        writer = csv_mod.DictWriter(f, fieldnames=["image_id", "label"])
        writer.writeheader()
        for r in rows:
            writer.writerow(r)


class TestRunTier3Sen2lulc:
    def test_sample_fraction_actually_subsamples(self, tmp_path):
        """Regression: sample_fraction used to be a dead parameter — the
        full annotation set got processed regardless, risking a resource
        blowout against Kaggle's 20GB /kaggle/working cap."""
        input_dir = tmp_path / "input"
        _make_sen2lulc_input(input_dir, {0: 100})

        output_dir = tmp_path / "output"
        stats = run_tier3_sen2lulc(input_dir, output_dir, sample_fraction=0.1, seed=1)
        assert 5 <= stats["processed"] <= 20  # ~10% of 100, not all 100

    def test_stratified_sampling_keeps_rare_class(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_sen2lulc_input(input_dir, {0: 50, 2: 2})  # class 2 = Water, rare

        output_dir = tmp_path / "output"
        run_tier3_sen2lulc(input_dir, output_dir, sample_fraction=0.2, seed=1)

        with open(output_dir / "sen2lulc.jsonl") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        assert any(l["response"] == "Water" for l in lines), \
            "Rare class (Water, n=2) was dropped by subsampling"

    def test_all_rows_validate(self, tmp_path):
        input_dir = tmp_path / "input"
        _make_sen2lulc_input(input_dir, {0: 5, 1: 5})
        output_dir = tmp_path / "output"
        run_tier3_sen2lulc(input_dir, output_dir, sample_fraction=1.0)

        from preprocess.validator import validate_sample
        with open(output_dir / "sen2lulc.jsonl") as f:
            for line in f:
                if not line.strip():
                    continue
                ok, errs = validate_sample(json.loads(line))
                assert ok, errs


In [ ]:
%%writefile tests/test_validator.py
"""Unit tests for preprocess/validator.py.

Tests all 7 hand-crafted fixtures from tests/fixtures/samples.json:
    #1 — Valid single-image VQA (should pass)
    #2 — Valid single-image grounding (should pass)
    #3 — Valid bitemporal change_vqa (should pass)
    #4 — Valid cross-modal fusion_grounding (should pass)
    #5 — INVALID: missing modality field (schema rejection)
    #6 — INVALID: bbox value 1500 exceeds [0,1000] (schema rejection)
    #7 — INVALID: pair_type='single' but image_path has 2 entries (cross-field)
"""

from __future__ import annotations

import json
import tempfile
from pathlib import Path

import pytest

from preprocess.validator import cross_field_checks, validate_jsonl, validate_sample

FIXTURES_DIR = Path(__file__).parent / "fixtures"
FIXTURES_JSON = FIXTURES_DIR / "samples.json"


def _load_fixtures() -> list[dict]:
    with open(FIXTURES_JSON, "r", encoding="utf-8") as f:
        return json.load(f)


@pytest.fixture(scope="module")
def samples() -> list[dict]:
    return _load_fixtures()


# ---- Schema-level validation ----


class TestValidSamples:
    """Samples #1–#4 must pass schema + cross-field validation."""

    @pytest.fixture(autouse=True)
    def _load(self, samples):
        self.valid_vqa = samples[0]           # #1
        self.valid_grounding = samples[1]     # #2
        self.valid_change_vqa = samples[2]    # #3
        self.valid_fusion = samples[3]        # #4

    def test_valid_single_vqa(self):
        ok, errs = validate_sample(self.valid_vqa)
        assert ok, f"Expected valid, got errors: {errs}"

    def test_valid_single_grounding(self):
        ok, errs = validate_sample(self.valid_grounding)
        assert ok, f"Expected valid, got errors: {errs}"

    def test_valid_bitemporal_change_vqa(self):
        ok, errs = validate_sample(self.valid_change_vqa)
        assert ok, f"Expected valid, got errors: {errs}"

    def test_valid_cross_modal_fusion_grounding(self):
        ok, errs = validate_sample(self.valid_fusion)
        assert ok, f"Expected valid, got errors: {errs}"

    def test_all_four_valid_no_errors(self):
        for i, sample in enumerate([self.valid_vqa, self.valid_grounding,
                                     self.valid_change_vqa, self.valid_fusion]):
            ok, errs = validate_sample(sample)
            assert ok, f"Sample #{i+1} unexpected errors: {errs}"


class TestInvalidSamples:
    """Samples #5–#7 must fail with specific errors."""

    @pytest.fixture(autouse=True)
    def _load(self, samples):
        self.missing_modality = samples[4]   # #5
        self.bbox_overflow = samples[5]      # #6
        self.pair_image_mismatch = samples[6]  # #7

    def test_missing_modality(self):
        ok, errs = validate_sample(self.missing_modality)
        assert not ok, "Expected invalid (missing modality)"
        assert any("'modality'" in e or "modality" in e.lower() for e in errs), \
            f"Expected modality-related error, got: {errs}"

    def test_bbox_overflow(self):
        ok, errs = validate_sample(self.bbox_overflow)
        assert not ok, "Expected invalid (bbox value 1500)"
        assert any("1500" in e or "1000" in e for e in errs), \
            f"Expected bbox range error, got: {errs}"

    def test_pair_image_mismatch(self):
        ok, errs = validate_sample(self.pair_image_mismatch)
        assert not ok, "Expected invalid (pair_type != len(image_path))"
        assert any("image" in e.lower() and ("pair_type" in e or "single" in e)
                    for e in errs), \
            f"Expected pair/image mismatch error, got: {errs}"


# ---- Cross-field checks (isolated) ----


class TestCrossFieldChecks:
    """Direct tests of cross_field_checks beyond schema validation."""

    def test_change_grounding_valid_bitemporal_two_images(self):
        """Positive: change_grounding + bitemporal + 2 images should pass."""
        sample = {
            "id": "x", "dataset": "cdvqa", "task": "change_grounding",
            "image_path": ["a.png", "b.png"], "pair_type": "bitemporal",
            "gsd_bucket": "[GSD:2m]", "split": "train",
            "instruction": "Where?", "response": "Here.",
            "bbox": [[0, 0, 500, 500]], "modality": "optical",
        }
        errs = cross_field_checks(sample)
        assert errs == [], f"Expected no errors, got: {errs}"

    def test_change_grounding_requires_bitemporal(self):
        sample = {
            "id": "x", "dataset": "cdvqa", "task": "change_grounding",
            "image_path": ["a.png", "b.png"], "pair_type": "cross-modal",
            "gsd_bucket": "[GSD:2m]", "split": "train",
            "instruction": "Where?", "response": "Here.",
            "bbox": [[0, 0, 500, 500]], "modality": "optical+sar",
        }
        errs = cross_field_checks(sample)
        assert any("bitemporal" in e for e in errs), \
            f"Expected bitemporal requirement error, got: {errs}"

    def test_change_grounding_requires_two_images(self):
        sample = {
            "id": "x", "dataset": "cdvqa", "task": "change_grounding",
            "image_path": ["a.png"], "pair_type": "bitemporal",
            "gsd_bucket": "[GSD:2m]", "split": "train",
            "instruction": "Where?", "response": "Here.",
            "bbox": [[0, 0, 500, 500]], "modality": "optical",
        }
        errs = cross_field_checks(sample)
        assert any("2 images" in e for e in errs), \
            f"Expected 2-images requirement error, got: {errs}"

    def test_fusion_grounding_requires_cross_modal(self):
        sample = {
            "id": "x", "dataset": "sn6_sar", "task": "fusion_grounding",
            "image_path": ["a.tif", "b.tif"], "pair_type": "bitemporal",
            "gsd_bucket": "[GSD:0.5m]", "split": "train",
            "instruction": "Where?", "response": "Here.",
            "bbox": [[0, 0, 500, 500]], "modality": "optical+sar",
        }
        errs = cross_field_checks(sample)
        assert any("cross-modal" in e for e in errs), \
            f"Expected cross-modal requirement error, got: {errs}"

    def test_cross_modal_requires_optical_sar(self):
        sample = {
            "id": "x", "dataset": "sn6_sar", "task": "fusion_vqa",
            "image_path": ["a.tif", "b.tif"], "pair_type": "cross-modal",
            "gsd_bucket": "[GSD:0.5m]", "split": "train",
            "instruction": "What?", "response": "This.",
            "bbox": None, "modality": "optical",
        }
        errs = cross_field_checks(sample)
        assert any("optical+sar" in e for e in errs), \
            f"Expected optical+sar requirement error, got: {errs}"

    def test_grounding_with_empty_bbox_list_rejected(self):
        """Regression: bbox=[] (empty, not null) used to slip past the
        None-only check on a grounding task."""
        sample = {
            "id": "x", "dataset": "vrsbench", "task": "grounding",
            "image_path": ["a.png"], "pair_type": "single",
            "gsd_bucket": "VHR-native", "split": "train",
            "instruction": "Where?", "response": "Here.",
            "bbox": [], "modality": "optical",
        }
        errs = cross_field_checks(sample)
        assert any("non-empty bbox" in e for e in errs), \
            f"Expected empty-bbox rejection, got: {errs}"

    def test_single_with_two_images(self):
        sample = {
            "id": "x", "dataset": "cdvqa", "task": "vqa",
            "image_path": ["a.png", "b.png"], "pair_type": "single",
            "gsd_bucket": "[GSD:2m]", "split": "train",
            "instruction": "What?", "response": "This.",
            "bbox": None, "modality": "optical",
        }
        errs = cross_field_checks(sample)
        assert any("single" in e and "1 image" in e for e in errs), \
            f"Expected single/1-image error, got: {errs}"


# ---- validate_jsonl (file-level) ----


class TestValidateJsonl:
    def test_valid_jsonl(self):
        """Write 4 valid samples to a temp file and validate."""
        samples = _load_fixtures()[:4]
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".jsonl", delete=False, encoding="utf-8"
        ) as f:
            for s in samples:
                f.write(json.dumps(s) + "\n")
            tmp_path = f.name

        count, errs = validate_jsonl(tmp_path)
        assert count == 4
        assert errs == [], f"Unexpected errors: {errs}"

        Path(tmp_path).unlink()

    def test_mixed_valid_invalid(self):
        """Write 2 valid + 1 invalid, expect errors only for invalid."""
        samples = _load_fixtures()
        rows = [samples[0], samples[4], samples[1]]  # valid, invalid, valid
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".jsonl", delete=False, encoding="utf-8"
        ) as f:
            for s in rows:
                f.write(json.dumps(s) + "\n")
            tmp_path = f.name

        count, errs = validate_jsonl(tmp_path)
        assert count == 3
        assert len(errs) > 0, "Expected errors for invalid sample"
        assert any("line 2" in e for e in errs), \
            f"Expected error on line 2, got: {errs}"

        Path(tmp_path).unlink()

    def test_empty_file(self):
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".jsonl", delete=False, encoding="utf-8"
        ) as f:
            tmp_path = f.name

        count, errs = validate_jsonl(tmp_path)
        assert count == 0
        assert errs == []

        Path(tmp_path).unlink()


# ---- Edge cases ----


class TestEdgeCases:
    def test_empty_string_instruction_rejected(self):
        sample = {
            "id": "x", "dataset": "bigen", "task": "vqa",
            "image_path": ["a.png"], "pair_type": "single",
            "gsd_bucket": "[GSD:10m]", "split": "train",
            "instruction": "", "response": "Answer.",
            "bbox": None, "modality": "optical",
        }
        ok, errs = validate_sample(sample)
        assert not ok
        assert any("instruction" in e for e in errs)

    def test_bbox_at_boundaries_valid(self):
        sample = {
            "id": "x", "dataset": "vrsbench", "task": "grounding",
            "image_path": ["a.png"], "pair_type": "single",
            "gsd_bucket": "VHR-native", "split": "train",
            "instruction": "Locate.", "response": "Here.",
            "bbox": [[0, 0, 1000, 1000]], "modality": "optical",
        }
        ok, errs = validate_sample(sample)
        assert ok, f"Boundary bbox [0,0,1000,1000] should be valid, got: {errs}"

    def test_negative_bbox_rejected(self):
        sample = {
            "id": "x", "dataset": "vrsbench", "task": "grounding",
            "image_path": ["a.png"], "pair_type": "single",
            "gsd_bucket": "VHR-native", "split": "train",
            "instruction": "Locate.", "response": "Here.",
            "bbox": [[-1, 0, 500, 500]], "modality": "optical",
        }
        ok, errs = validate_sample(sample)
        assert not ok
        assert any("-1" in e or "minimum" in e.lower() for e in errs)

    def test_unknown_dataset_rejected(self):
        sample = {
            "id": "x", "dataset": "unknown_ds", "task": "vqa",
            "image_path": ["a.png"], "pair_type": "single",
            "gsd_bucket": "[GSD:10m]", "split": "train",
            "instruction": "Q?", "response": "A.",
            "bbox": None, "modality": "optical",
        }
        ok, errs = validate_sample(sample)
        assert not ok
        assert any("unknown_ds" in e or "dataset" in e for e in errs)

    def test_additional_properties_rejected(self):
        sample = {
            "id": "x", "dataset": "bigen", "task": "vqa",
            "image_path": ["a.png"], "pair_type": "single",
            "gsd_bucket": "[GSD:10m]", "split": "train",
            "instruction": "Q?", "response": "A.",
            "bbox": None, "modality": "optical",
            "extra_field": "should not be here",
        }
        ok, errs = validate_sample(sample)
        assert not ok
        assert any("extra_field" in e or "Additional" in e or "additional" in e
                    for e in errs)


## Sanity check — imports + unit tests
Confirms every module written above imports cleanly before touching real data.

In [ ]:
!python -m pytest tests/ -q

## Run the pipeline

Point `--input-dir` at a mounted Kaggle dataset (e.g. `/kaggle/input/oscd`). Uncomment the tiers you have data for. Each tier writes JSONL + images to its own `--output-dir`; `merge_and_package.py` combines tier outputs into the final shard set.

In [ ]:
import os

os.makedirs("/kaggle/working/out", exist_ok=True)

# --- Tier 0 ---
# !python -m preprocess.tier0_bigen \
#     --input-dir /kaggle/input/bigearthnet --output-dir /kaggle/working/out/tier0_bigen

# --- Tier 1 ---
# !python -m preprocess.tier1_oscd \
#     --input-dir /kaggle/input/oscd --output-dir /kaggle/working/out/tier1_oscd
# !python -m preprocess.tier1_vrsbench \
#     --input-dir /kaggle/input/vrsbench --output-dir /kaggle/working/out/tier1_vrsbench
# !python -m preprocess.tier1_rsvqa_hr \
#     --input-dir /kaggle/input/rsvqa-hr --output-dir /kaggle/working/out/tier1_rsvqa_hr
# !python -m preprocess.tier1_levir_cd \
#     --input-dir /kaggle/input/levir-cd --output-dir /kaggle/working/out/tier1_levir_cd
# !python -m preprocess.tier1_sn6_opt \
#     --input-dir /kaggle/input/spacenet6 --output-dir /kaggle/working/out/tier1_sn6_opt
# !python -m preprocess.tier1_cdvqa \
#     --input-dir /kaggle/input/cdvqa --output-dir /kaggle/working/out/tier1_cdvqa

# --- Tier 2 ---
# !python -m preprocess.tier2_sardet \
#     --input-dir /kaggle/input/sardet-100k --output-dir /kaggle/working/out/tier2_sardet
# !python -m preprocess.tier2_sn6_sar \
#     --input-dir /kaggle/input/spacenet6-sar --output-dir /kaggle/working/out/tier2_sn6_sar \
#     --optical-dir /kaggle/working/out/tier1_sn6_opt

# --- Tier 3 ---
# !python -m preprocess.tier3_sen2lulc \
#     --input-dir /kaggle/input/sen2-lulc --output-dir /kaggle/working/out/tier3_sen2lulc

# --- Merge + val split (after the tiers you need have run) ---
# !python -m preprocess.merge_and_package \
#     --tier-dirs /kaggle/working/out/tier0_bigen /kaggle/working/out/tier1_oscd \
#     --output-dir /kaggle/working/out/merged --tar-shards
# !python -m preprocess.split_internal_val --jsonl-dir /kaggle/working/out/merged


## Sanity-check a raw download
Run before preprocessing a freshly downloaded/mounted dataset.

In [ ]:
# !python -m preprocess.sanity_check --dataset oscd --input-dir /kaggle/input/oscd
# !python -m preprocess.sanity_check --dataset all --input-dir /kaggle/input


## Training (separate GPU session)
`training/` is written above for completeness, but Unsloth/torch/bitsandbytes are **not** installed by this notebook — install them per Unsloth's Kaggle GPU install docs in a GPU-enabled session before running `train.py`.

In [ ]:
# !python -m training.train --config path/to/stage_a.yaml
